# Cairo classification of gingival recession: training and evaluation

This notebook trains and evaluates every model of the study and writes all tables and figures. The partition is
`data/folds.csv`: five development folds and a test set, with the photographs of each clinical case kept in one partition.

- **Models:** single branches (DINOv2-small, -base and -large, MobileNetV2, EfficientNetB0, MaxViT-T, ConvNeXt-XXLarge CLIP),
  hybrids of DINOv2-small with each CNN branch, and the proposed hybrid without cross-attention and without anatomical supervision.
- **Protocol:** 3 seeds × 5 folds, inner-split early stopping; the annotated tissue masks are an auxiliary training signal on
  the DINOv2 patch tokens.
- **Evaluation after each run:** test probabilities on the original and degraded photographs and Grad-CAM maps.

## How to run on Kaggle (GPU T4, Internet on)
Attach a dataset that contains `folds.csv`, the photographs in `images/<class>/<image_id>.png` and the masks in `masks/`.

1. **Part A:** keep `PART = "A"`, then *Save & Run All*, and download `gingival_partA.zip`.
2. **Part B:** attach `gingival_partA.zip` as a dataset, set `PART = "B"`, run, and download `gingival_partB.zip`.
3. **Part C:** attach both earlier outputs, set `PART = "C"`, and run. It trains the components of the proposed hybrid,
   measures the cost of every model and writes the results (`gingival_partC.zip`, folder `results/`).

A part that reaches the time limit stops cleanly; run it again with its own output attached, and finished runs are skipped.

In [ ]:
PART = "A"                     # "A", "B" or "C"

import os, sys, glob, zipfile
W = os.environ.get("NB_WORK", "/kaggle/working"); INP = os.environ.get("NB_INPUT", "/kaggle/input")
folds = glob.glob(f"{INP}/**/folds.csv", recursive=True)
assert folds, "attach the dataset containing folds.csv, images/ and masks/"
ROOT = os.path.dirname(folds[0])
os.environ.update(GR_IMG_ROOT=os.path.join(ROOT, "images"), GR_MASK_ROOT=ROOT, GR_ROOT=W, GR_PKG=f"{W}/pkg",
                  GR_WORK=f"{W}/work", GR_OUT=f"{W}/output", HF_HUB_DOWNLOAD_TIMEOUT="120")
os.makedirs(f"{W}/src", exist_ok=True); sys.path.insert(0, f"{W}/src")
print("data:", ROOT, "| part", PART)

## Partition

In [ ]:
%%writefile {W}/src/folds.csv
image_id,cls,split,fold,excluded_reason,case_group,similarity_group,image,mask
HEALTHY_0000,HEALTHY,dev,1,,C0000,G0000,HEALTHY/HEALTHY_0000.png,masks/HEALTHY_0000_mask.png
HEALTHY_0001,HEALTHY,final_test,test,,C0001,G0001,HEALTHY/HEALTHY_0001.png,masks/HEALTHY_0001_mask.png
HEALTHY_0002,HEALTHY,dev,-1,near-duplicate of a final_test image,C0002,G0002,HEALTHY/HEALTHY_0002.png,masks/HEALTHY_0002_mask.png
HEALTHY_0003,HEALTHY,final_test,test,,C0002,G0002,HEALTHY/HEALTHY_0003.png,masks/HEALTHY_0003_mask.png
HEALTHY_0004,HEALTHY,dev,2,,C0003,G0003,HEALTHY/HEALTHY_0004.png,masks/HEALTHY_0004_mask.png
HEALTHY_0005,HEALTHY,dev,3,,C0004,G0004,HEALTHY/HEALTHY_0005.png,masks/HEALTHY_0005_mask.png
HEALTHY_0006,HEALTHY,final_test,test,,C0005,G0005,HEALTHY/HEALTHY_0006.png,masks/HEALTHY_0006_mask.png
HEALTHY_0007,HEALTHY,dev,3,,C0006,G0006,HEALTHY/HEALTHY_0007.png,masks/HEALTHY_0007_mask.png
HEALTHY_0008,HEALTHY,dev,3,,C0007,G0007,HEALTHY/HEALTHY_0008.png,masks/HEALTHY_0008_mask.png
HEALTHY_0009,HEALTHY,dev,1,,C0008,G0008,HEALTHY/HEALTHY_0009.png,masks/HEALTHY_0009_mask.png
HEALTHY_0010,HEALTHY,dev,3,,C0009,G0009,HEALTHY/HEALTHY_0010.png,masks/HEALTHY_0010_mask.png
HEALTHY_0011,HEALTHY,dev,0,,C0010,G0010,HEALTHY/HEALTHY_0011.png,masks/HEALTHY_0011_mask.png
HEALTHY_0012,HEALTHY,dev,2,,C0011,G0011,HEALTHY/HEALTHY_0012.png,masks/HEALTHY_0012_mask.png
HEALTHY_0013,HEALTHY,dev,1,,C0012,G0012,HEALTHY/HEALTHY_0013.png,masks/HEALTHY_0013_mask.png
HEALTHY_0014,HEALTHY,final_test,test,,C0013,G0013,HEALTHY/HEALTHY_0014.png,masks/HEALTHY_0014_mask.png
HEALTHY_0015,HEALTHY,dev,2,,C0014,G0014,HEALTHY/HEALTHY_0015.png,masks/HEALTHY_0015_mask.png
HEALTHY_0016,HEALTHY,dev,4,,C0015,G0015,HEALTHY/HEALTHY_0016.png,masks/HEALTHY_0016_mask.png
HEALTHY_0017,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0016,G0016,HEALTHY/HEALTHY_0017.png,masks/HEALTHY_0017_mask.png
HEALTHY_0018,HEALTHY,dev,3,,C0017,G0017,HEALTHY/HEALTHY_0018.png,masks/HEALTHY_0018_mask.png
HEALTHY_0019,HEALTHY,dev,0,,C0018,G0018,HEALTHY/HEALTHY_0019.png,masks/HEALTHY_0019_mask.png
HEALTHY_0020,HEALTHY,dev,1,,C0019,G0019,HEALTHY/HEALTHY_0020.png,masks/HEALTHY_0020_mask.png
HEALTHY_0021,HEALTHY,dev,2,,C0020,G0020,HEALTHY/HEALTHY_0021.png,masks/HEALTHY_0021_mask.png
HEALTHY_0022,HEALTHY,final_test,test,,C0021,G0021,HEALTHY/HEALTHY_0022.png,masks/HEALTHY_0022_mask.png
HEALTHY_0023,HEALTHY,dev,3,,C0022,G0022,HEALTHY/HEALTHY_0023.png,masks/HEALTHY_0023_mask.png
HEALTHY_0024,HEALTHY,dev,-1,near-duplicate of a final_test image,C0021,G0021,HEALTHY/HEALTHY_0024.png,masks/HEALTHY_0024_mask.png
HEALTHY_0025,HEALTHY,dev,0,,C0023,G0023,HEALTHY/HEALTHY_0025.png,masks/HEALTHY_0025_mask.png
HEALTHY_0026,HEALTHY,dev,1,,C0024,G0024,HEALTHY/HEALTHY_0026.png,masks/HEALTHY_0026_mask.png
HEALTHY_0027,HEALTHY,final_test,test,,C0025,G0025,HEALTHY/HEALTHY_0027.png,masks/HEALTHY_0027_mask.png
HEALTHY_0028,HEALTHY,dev,4,,C0026,G0026,HEALTHY/HEALTHY_0028.png,masks/HEALTHY_0028_mask.png
HEALTHY_0029,HEALTHY,dev,0,,C0027,G0027,HEALTHY/HEALTHY_0029.png,masks/HEALTHY_0029_mask.png
HEALTHY_0030,HEALTHY,dev,0,,C0028,G0028,HEALTHY/HEALTHY_0030.png,masks/HEALTHY_0030_mask.png
HEALTHY_0031,HEALTHY,dev,3,,C0029,G0029,HEALTHY/HEALTHY_0031.png,masks/HEALTHY_0031_mask.png
HEALTHY_0032,HEALTHY,final_test,test,,C0030,G0030,HEALTHY/HEALTHY_0032.png,masks/HEALTHY_0032_mask.png
HEALTHY_0033,HEALTHY,dev,2,,C0031,G0031,HEALTHY/HEALTHY_0033.png,masks/HEALTHY_0033_mask.png
HEALTHY_0034,HEALTHY,final_test,test,,C0032,G0032,HEALTHY/HEALTHY_0034.png,masks/HEALTHY_0034_mask.png
HEALTHY_0035,HEALTHY,dev,2,,C0033,G0033,HEALTHY/HEALTHY_0035.png,masks/HEALTHY_0035_mask.png
HEALTHY_0036,HEALTHY,dev,3,,C0034,G0034,HEALTHY/HEALTHY_0036.png,masks/HEALTHY_0036_mask.png
HEALTHY_0037,HEALTHY,dev,4,,C0035,G0035,HEALTHY/HEALTHY_0037.png,masks/HEALTHY_0037_mask.png
HEALTHY_0038,HEALTHY,dev,1,,C0036,G0036,HEALTHY/HEALTHY_0038.png,masks/HEALTHY_0038_mask.png
HEALTHY_0039,HEALTHY,dev,0,,C0037,G0037,HEALTHY/HEALTHY_0039.png,masks/HEALTHY_0039_mask.png
HEALTHY_0040,HEALTHY,final_test,test,,C0038,G0038,HEALTHY/HEALTHY_0040.png,masks/HEALTHY_0040_mask.png
HEALTHY_0041,HEALTHY,final_test,test,,C0039,G0039,HEALTHY/HEALTHY_0041.png,masks/HEALTHY_0041_mask.png
HEALTHY_0042,HEALTHY,final_test,test,,C0040,G0040,HEALTHY/HEALTHY_0042.png,masks/HEALTHY_0042_mask.png
HEALTHY_0043,HEALTHY,dev,1,,C0041,G0041,HEALTHY/HEALTHY_0043.png,masks/HEALTHY_0043_mask.png
HEALTHY_0044,HEALTHY,dev,2,,C0042,G0042,HEALTHY/HEALTHY_0044.png,masks/HEALTHY_0044_mask.png
HEALTHY_0045,HEALTHY,dev,3,,C0043,G0043,HEALTHY/HEALTHY_0045.png,masks/HEALTHY_0045_mask.png
HEALTHY_0046,HEALTHY,dev,1,,C0044,G0044,HEALTHY/HEALTHY_0046.png,masks/HEALTHY_0046_mask.png
HEALTHY_0047,HEALTHY,dev,1,,C0045,G0045,HEALTHY/HEALTHY_0047.png,masks/HEALTHY_0047_mask.png
HEALTHY_0048,HEALTHY,dev,2,,C0046,G0046,HEALTHY/HEALTHY_0048.png,masks/HEALTHY_0048_mask.png
HEALTHY_0049,HEALTHY,dev,4,,C0047,G0047,HEALTHY/HEALTHY_0049.png,masks/HEALTHY_0049_mask.png
HEALTHY_0050,HEALTHY,dev,3,,C0048,G0048,HEALTHY/HEALTHY_0050.png,masks/HEALTHY_0050_mask.png
HEALTHY_0051,HEALTHY,dev,3,,C0049,G0049,HEALTHY/HEALTHY_0051.png,masks/HEALTHY_0051_mask.png
HEALTHY_0052,HEALTHY,final_test,test,,C0050,G0050,HEALTHY/HEALTHY_0052.png,masks/HEALTHY_0052_mask.png
HEALTHY_0053,HEALTHY,dev,1,,C0051,G0051,HEALTHY/HEALTHY_0053.png,masks/HEALTHY_0053_mask.png
HEALTHY_0054,HEALTHY,dev,0,,C0052,G0052,HEALTHY/HEALTHY_0054.png,masks/HEALTHY_0054_mask.png
HEALTHY_0055,HEALTHY,dev,2,,C0053,G0053,HEALTHY/HEALTHY_0055.png,masks/HEALTHY_0055_mask.png
HEALTHY_0056,HEALTHY,dev,-1,near-duplicate of a final_test image,C0054,G0054,HEALTHY/HEALTHY_0056.png,masks/HEALTHY_0056_mask.png
HEALTHY_0057,HEALTHY,final_test,test,,C0055,G0055,HEALTHY/HEALTHY_0057.png,masks/HEALTHY_0057_mask.png
HEALTHY_0058,HEALTHY,final_test,test,,C0056,G0056,HEALTHY/HEALTHY_0058.png,masks/HEALTHY_0058_mask.png
HEALTHY_0059,HEALTHY,dev,3,,C0057,G0057,HEALTHY/HEALTHY_0059.png,masks/HEALTHY_0059_mask.png
HEALTHY_0060,HEALTHY,dev,1,,C0058,G0058,HEALTHY/HEALTHY_0060.png,masks/HEALTHY_0060_mask.png
HEALTHY_0061,HEALTHY,dev,4,,C0059,G0059,HEALTHY/HEALTHY_0061.png,masks/HEALTHY_0061_mask.png
HEALTHY_0062,HEALTHY,final_test,test,,C0060,G0060,HEALTHY/HEALTHY_0062.png,masks/HEALTHY_0062_mask.png
HEALTHY_0063,HEALTHY,dev,4,,C0061,G0061,HEALTHY/HEALTHY_0063.png,masks/HEALTHY_0063_mask.png
HEALTHY_0064,HEALTHY,dev,2,,C0062,G0062,HEALTHY/HEALTHY_0064.png,masks/HEALTHY_0064_mask.png
HEALTHY_0065,HEALTHY,dev,1,,C0063,G0063,HEALTHY/HEALTHY_0065.png,masks/HEALTHY_0065_mask.png
HEALTHY_0066,HEALTHY,final_test,test,,C0064,G0064,HEALTHY/HEALTHY_0066.png,masks/HEALTHY_0066_mask.png
HEALTHY_0067,HEALTHY,dev,0,,C0065,G0065,HEALTHY/HEALTHY_0067.png,masks/HEALTHY_0067_mask.png
HEALTHY_0068,HEALTHY,dev,2,,C0066,G0066,HEALTHY/HEALTHY_0068.png,masks/HEALTHY_0068_mask.png
HEALTHY_0069,HEALTHY,dev,3,,C0067,G0067,HEALTHY/HEALTHY_0069.png,masks/HEALTHY_0069_mask.png
HEALTHY_0070,HEALTHY,final_test,test,,C0068,G0068,HEALTHY/HEALTHY_0070.png,masks/HEALTHY_0070_mask.png
HEALTHY_0071,HEALTHY,final_test,test,,C0069,G0069,HEALTHY/HEALTHY_0071.png,masks/HEALTHY_0071_mask.png
HEALTHY_0072,HEALTHY,dev,0,,C0070,G0070,HEALTHY/HEALTHY_0072.png,masks/HEALTHY_0072_mask.png
HEALTHY_0073,HEALTHY,dev,0,,C0070,G0070,HEALTHY/HEALTHY_0073.png,masks/HEALTHY_0073_mask.png
HEALTHY_0074,HEALTHY,final_test,test,,C0071,G0071,HEALTHY/HEALTHY_0074.png,masks/HEALTHY_0074_mask.png
HEALTHY_0075,HEALTHY,final_test,test,,C0072,G0072,HEALTHY/HEALTHY_0075.png,masks/HEALTHY_0075_mask.png
HEALTHY_0076,HEALTHY,final_test,test,,C0073,G0073,HEALTHY/HEALTHY_0076.png,masks/HEALTHY_0076_mask.png
HEALTHY_0077,HEALTHY,dev,1,,C0074,G0074,HEALTHY/HEALTHY_0077.png,masks/HEALTHY_0077_mask.png
HEALTHY_0078,HEALTHY,final_test,test,,C0075,G0075,HEALTHY/HEALTHY_0078.png,masks/HEALTHY_0078_mask.png
HEALTHY_0079,HEALTHY,final_test,test,,C0054,G0054,HEALTHY/HEALTHY_0079.png,masks/HEALTHY_0079_mask.png
HEALTHY_0080,HEALTHY,final_test,test,,C0076,G0076,HEALTHY/HEALTHY_0080.png,masks/HEALTHY_0080_mask.png
HEALTHY_0081,HEALTHY,dev,-1,near-duplicate of a final_test image,C0076,G0076,HEALTHY/HEALTHY_0081.png,masks/HEALTHY_0081_mask.png
HEALTHY_0082,HEALTHY,dev,1,,C0077,G0077,HEALTHY/HEALTHY_0082.png,masks/HEALTHY_0082_mask.png
HEALTHY_0083,HEALTHY,final_test,test,,C0078,G0078,HEALTHY/HEALTHY_0083.png,masks/HEALTHY_0083_mask.png
HEALTHY_0084,HEALTHY,dev,3,,C0079,G0079,HEALTHY/HEALTHY_0084.png,masks/HEALTHY_0084_mask.png
HEALTHY_0085,HEALTHY,dev,4,,C0080,G0080,HEALTHY/HEALTHY_0085.png,masks/HEALTHY_0085_mask.png
HEALTHY_0086,HEALTHY,dev,0,,C0081,G0081,HEALTHY/HEALTHY_0086.png,masks/HEALTHY_0086_mask.png
HEALTHY_0087,HEALTHY,dev,1,,C0082,G0082,HEALTHY/HEALTHY_0087.png,masks/HEALTHY_0087_mask.png
HEALTHY_0088,HEALTHY,dev,4,,C0083,G0083,HEALTHY/HEALTHY_0088.png,masks/HEALTHY_0088_mask.png
HEALTHY_0089,HEALTHY,dev,2,,C0084,G0084,HEALTHY/HEALTHY_0089.png,masks/HEALTHY_0089_mask.png
HEALTHY_0090,HEALTHY,dev,3,,C0085,G0085,HEALTHY/HEALTHY_0090.png,masks/HEALTHY_0090_mask.png
HEALTHY_0091,HEALTHY,final_test,test,,C0086,G0086,HEALTHY/HEALTHY_0091.png,masks/HEALTHY_0091_mask.png
HEALTHY_0092,HEALTHY,dev,4,,C0087,G0087,HEALTHY/HEALTHY_0092.png,masks/HEALTHY_0092_mask.png
HEALTHY_0093,HEALTHY,dev,4,,C0087,G0087,HEALTHY/HEALTHY_0093.png,masks/HEALTHY_0093_mask.png
HEALTHY_0094,HEALTHY,final_test,test,,C0088,G0088,HEALTHY/HEALTHY_0094.png,masks/HEALTHY_0094_mask.png
HEALTHY_0095,HEALTHY,final_test,test,,C0089,G0089,HEALTHY/HEALTHY_0095.png,masks/HEALTHY_0095_mask.png
HEALTHY_0096,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0090,G0090,HEALTHY/HEALTHY_0096.png,masks/HEALTHY_0096_mask.png
HEALTHY_0097,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0090,G0091,HEALTHY/HEALTHY_0097.png,masks/HEALTHY_0097_mask.png
HEALTHY_0098,HEALTHY,dev,1,,C0091,G0092,HEALTHY/HEALTHY_0098.png,masks/HEALTHY_0098_mask.png
HEALTHY_0099,HEALTHY,dev,2,,C0092,G0093,HEALTHY/HEALTHY_0099.png,masks/HEALTHY_0099_mask.png
HEALTHY_0100,HEALTHY,dev,4,,C0093,G0094,HEALTHY/HEALTHY_0100.png,masks/HEALTHY_0100_mask.png
HEALTHY_0101,HEALTHY,dev,3,,C0094,G0095,HEALTHY/HEALTHY_0101.png,masks/HEALTHY_0101_mask.png
HEALTHY_0102,HEALTHY,final_test,test,,C0095,G0096,HEALTHY/HEALTHY_0102.png,masks/HEALTHY_0102_mask.png
HEALTHY_0103,HEALTHY,final_test,test,,C0096,G0097,HEALTHY/HEALTHY_0103.png,masks/HEALTHY_0103_mask.png
HEALTHY_0104,HEALTHY,dev,0,,C0097,G0098,HEALTHY/HEALTHY_0104.png,masks/HEALTHY_0104_mask.png
HEALTHY_0105,HEALTHY,dev,1,,C0098,G0099,HEALTHY/HEALTHY_0105.png,masks/HEALTHY_0105_mask.png
HEALTHY_0106,HEALTHY,dev,2,,C0099,G0100,HEALTHY/HEALTHY_0106.png,masks/HEALTHY_0106_mask.png
HEALTHY_0107,HEALTHY,final_test,test,,C0100,G0101,HEALTHY/HEALTHY_0107.png,masks/HEALTHY_0107_mask.png
HEALTHY_0108,HEALTHY,final_test,test,,C0101,G0102,HEALTHY/HEALTHY_0108.png,masks/HEALTHY_0108_mask.png
HEALTHY_0109,HEALTHY,dev,0,,C0102,G0103,HEALTHY/HEALTHY_0109.png,masks/HEALTHY_0109_mask.png
HEALTHY_0110,HEALTHY,dev,-1,near-duplicate of a final_test image,C0103,G0104,HEALTHY/HEALTHY_0110.png,masks/HEALTHY_0110_mask.png
HEALTHY_0111,HEALTHY,dev,0,,C0104,G0105,HEALTHY/HEALTHY_0111.png,masks/HEALTHY_0111_mask.png
HEALTHY_0112,HEALTHY,dev,2,,C0105,G0106,HEALTHY/HEALTHY_0112.png,masks/HEALTHY_0112_mask.png
HEALTHY_0113,HEALTHY,dev,4,,C0106,G0107,HEALTHY/HEALTHY_0113.png,masks/HEALTHY_0113_mask.png
HEALTHY_0114,HEALTHY,dev,2,,C0107,G0108,HEALTHY/HEALTHY_0114.png,masks/HEALTHY_0114_mask.png
HEALTHY_0115,HEALTHY,final_test,test,,C0108,G0109,HEALTHY/HEALTHY_0115.png,masks/HEALTHY_0115_mask.png
HEALTHY_0116,HEALTHY,dev,4,,C0109,G0110,HEALTHY/HEALTHY_0116.png,masks/HEALTHY_0116_mask.png
HEALTHY_0117,HEALTHY,dev,1,,C0110,G0111,HEALTHY/HEALTHY_0117.png,masks/HEALTHY_0117_mask.png
HEALTHY_0118,HEALTHY,dev,3,,C0111,G0112,HEALTHY/HEALTHY_0118.png,masks/HEALTHY_0118_mask.png
HEALTHY_0119,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0112,G0113,HEALTHY/HEALTHY_0119.png,masks/HEALTHY_0119_mask.png
HEALTHY_0120,HEALTHY,dev,3,,C0113,G0114,HEALTHY/HEALTHY_0120.png,masks/HEALTHY_0120_mask.png
HEALTHY_0121,HEALTHY,final_test,test,,C0114,G0115,HEALTHY/HEALTHY_0121.png,masks/HEALTHY_0121_mask.png
HEALTHY_0122,HEALTHY,dev,2,,C0115,G0116,HEALTHY/HEALTHY_0122.png,masks/HEALTHY_0122_mask.png
HEALTHY_0123,HEALTHY,dev,3,,C0116,G0117,HEALTHY/HEALTHY_0123.png,masks/HEALTHY_0123_mask.png
HEALTHY_0124,HEALTHY,dev,0,,C0117,G0118,HEALTHY/HEALTHY_0124.png,masks/HEALTHY_0124_mask.png
HEALTHY_0125,HEALTHY,dev,0,,C0118,G0119,HEALTHY/HEALTHY_0125.png,masks/HEALTHY_0125_mask.png
HEALTHY_0126,HEALTHY,dev,2,,C0119,G0120,HEALTHY/HEALTHY_0126.png,masks/HEALTHY_0126_mask.png
HEALTHY_0127,HEALTHY,dev,4,,C0120,G0121,HEALTHY/HEALTHY_0127.png,masks/HEALTHY_0127_mask.png
HEALTHY_0128,HEALTHY,dev,1,,C0121,G0122,HEALTHY/HEALTHY_0128.png,masks/HEALTHY_0128_mask.png
HEALTHY_0129,HEALTHY,final_test,test,,C0122,G0123,HEALTHY/HEALTHY_0129.png,masks/HEALTHY_0129_mask.png
HEALTHY_0130,HEALTHY,dev,0,,C0123,G0124,HEALTHY/HEALTHY_0130.png,masks/HEALTHY_0130_mask.png
HEALTHY_0131,HEALTHY,final_test,test,,C0124,G0125,HEALTHY/HEALTHY_0131.png,masks/HEALTHY_0131_mask.png
HEALTHY_0132,HEALTHY,final_test,test,,C0125,G0126,HEALTHY/HEALTHY_0132.png,masks/HEALTHY_0132_mask.png
HEALTHY_0133,HEALTHY,dev,1,,C0126,G0127,HEALTHY/HEALTHY_0133.png,masks/HEALTHY_0133_mask.png
HEALTHY_0134,HEALTHY,dev,2,,C0127,G0128,HEALTHY/HEALTHY_0134.png,masks/HEALTHY_0134_mask.png
HEALTHY_0135,HEALTHY,dev,4,,C0128,G0129,HEALTHY/HEALTHY_0135.png,masks/HEALTHY_0135_mask.png
HEALTHY_0136,HEALTHY,dev,4,,C0129,G0130,HEALTHY/HEALTHY_0136.png,masks/HEALTHY_0136_mask.png
HEALTHY_0137,HEALTHY,dev,3,,C0130,G0131,HEALTHY/HEALTHY_0137.png,masks/HEALTHY_0137_mask.png
HEALTHY_0138,HEALTHY,dev,3,,C0130,G0131,HEALTHY/HEALTHY_0138.png,masks/HEALTHY_0138_mask.png
HEALTHY_0139,HEALTHY,dev,1,,C0131,G0132,HEALTHY/HEALTHY_0139.png,masks/HEALTHY_0139_mask.png
HEALTHY_0140,HEALTHY,dev,0,,C0132,G0133,HEALTHY/HEALTHY_0140.png,masks/HEALTHY_0140_mask.png
HEALTHY_0141,HEALTHY,final_test,test,,C0133,G0134,HEALTHY/HEALTHY_0141.png,masks/HEALTHY_0141_mask.png
HEALTHY_0142,HEALTHY,dev,-1,near-duplicate of a final_test image,C0054,G0054,HEALTHY/HEALTHY_0142.png,masks/HEALTHY_0142_mask.png
HEALTHY_0143,HEALTHY,dev,-1,near-duplicate of a final_test image,C0054,G0054,HEALTHY/HEALTHY_0143.png,masks/HEALTHY_0143_mask.png
HEALTHY_0144,HEALTHY,dev,2,,C0134,G0135,HEALTHY/HEALTHY_0144.png,masks/HEALTHY_0144_mask.png
HEALTHY_0145,HEALTHY,dev,2,,C0135,G0136,HEALTHY/HEALTHY_0145.png,masks/HEALTHY_0145_mask.png
HEALTHY_0146,HEALTHY,dev,3,,C0136,G0137,HEALTHY/HEALTHY_0146.png,masks/HEALTHY_0146_mask.png
HEALTHY_0147,HEALTHY,dev,4,,C0137,G0138,HEALTHY/HEALTHY_0147.png,masks/HEALTHY_0147_mask.png
HEALTHY_0148,HEALTHY,dev,4,,C0138,G0139,HEALTHY/HEALTHY_0148.png,masks/HEALTHY_0148_mask.png
HEALTHY_0149,HEALTHY,dev,0,,C0139,G0140,HEALTHY/HEALTHY_0149.png,masks/HEALTHY_0149_mask.png
HEALTHY_0150,HEALTHY,dev,1,,C0140,G0141,HEALTHY/HEALTHY_0150.png,masks/HEALTHY_0150_mask.png
HEALTHY_0151,HEALTHY,final_test,test,,C0141,G0142,HEALTHY/HEALTHY_0151.png,masks/HEALTHY_0151_mask.png
HEALTHY_0152,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0142,G0143,HEALTHY/HEALTHY_0152.png,masks/HEALTHY_0152_mask.png
HEALTHY_0153,HEALTHY,dev,0,,C0143,G0144,HEALTHY/HEALTHY_0153.png,masks/HEALTHY_0153_mask.png
HEALTHY_0154,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0144,G0145,HEALTHY/HEALTHY_0154.png,masks/HEALTHY_0154_mask.png
HEALTHY_0155,HEALTHY,dev,1,,C0145,G0146,HEALTHY/HEALTHY_0155.png,masks/HEALTHY_0155_mask.png
HEALTHY_0156,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0146,G0147,HEALTHY/HEALTHY_0156.png,masks/HEALTHY_0156_mask.png
HEALTHY_0157,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0147,G0148,HEALTHY/HEALTHY_0157.png,masks/HEALTHY_0157_mask.png
HEALTHY_0158,HEALTHY,dev,3,,C0148,G0149,HEALTHY/HEALTHY_0158.png,masks/HEALTHY_0158_mask.png
HEALTHY_0159,HEALTHY,dev,4,,C0149,G0150,HEALTHY/HEALTHY_0159.png,masks/HEALTHY_0159_mask.png
HEALTHY_0160,HEALTHY,dev,0,,C0150,G0151,HEALTHY/HEALTHY_0160.png,masks/HEALTHY_0160_mask.png
HEALTHY_0161,HEALTHY,dev,0,,C0151,G0152,HEALTHY/HEALTHY_0161.png,masks/HEALTHY_0161_mask.png
HEALTHY_0162,HEALTHY,dev,3,,C0152,G0153,HEALTHY/HEALTHY_0162.png,masks/HEALTHY_0162_mask.png
HEALTHY_0163,HEALTHY,dev,4,,C0153,G0154,HEALTHY/HEALTHY_0163.png,masks/HEALTHY_0163_mask.png
HEALTHY_0164,HEALTHY,dev,1,,C0154,G0155,HEALTHY/HEALTHY_0164.png,masks/HEALTHY_0164_mask.png
HEALTHY_0165,HEALTHY,dev,2,,C0155,G0156,HEALTHY/HEALTHY_0165.png,masks/HEALTHY_0165_mask.png
HEALTHY_0166,HEALTHY,dev,4,,C0156,G0157,HEALTHY/HEALTHY_0166.png,masks/HEALTHY_0166_mask.png
HEALTHY_0167,HEALTHY,dev,3,,C0157,G0158,HEALTHY/HEALTHY_0167.png,masks/HEALTHY_0167_mask.png
HEALTHY_0168,HEALTHY,dev,1,,C0158,G0159,HEALTHY/HEALTHY_0168.png,masks/HEALTHY_0168_mask.png
HEALTHY_0169,HEALTHY,dev,4,,C0159,G0160,HEALTHY/HEALTHY_0169.png,masks/HEALTHY_0169_mask.png
HEALTHY_0170,HEALTHY,dev,2,,C0160,G0161,HEALTHY/HEALTHY_0170.png,masks/HEALTHY_0170_mask.png
HEALTHY_0171,HEALTHY,dev,0,,C0161,G0162,HEALTHY/HEALTHY_0171.png,masks/HEALTHY_0171_mask.png
HEALTHY_0172,HEALTHY,dev,4,,C0162,G0163,HEALTHY/HEALTHY_0172.png,masks/HEALTHY_0172_mask.png
HEALTHY_0173,HEALTHY,dev,0,,C0163,G0164,HEALTHY/HEALTHY_0173.png,masks/HEALTHY_0173_mask.png
HEALTHY_0174,HEALTHY,dev,1,,C0164,G0165,HEALTHY/HEALTHY_0174.png,masks/HEALTHY_0174_mask.png
HEALTHY_0175,HEALTHY,dev,1,,C0165,G0166,HEALTHY/HEALTHY_0175.png,masks/HEALTHY_0175_mask.png
HEALTHY_0176,HEALTHY,final_test,test,,C0166,G0167,HEALTHY/HEALTHY_0176.png,masks/HEALTHY_0176_mask.png
HEALTHY_0177,HEALTHY,dev,2,,C0167,G0168,HEALTHY/HEALTHY_0177.png,masks/HEALTHY_0177_mask.png
HEALTHY_0178,HEALTHY,dev,1,,C0168,G0169,HEALTHY/HEALTHY_0178.png,masks/HEALTHY_0178_mask.png
HEALTHY_0179,HEALTHY,dev,3,,C0169,G0170,HEALTHY/HEALTHY_0179.png,masks/HEALTHY_0179_mask.png
HEALTHY_0180,HEALTHY,dev,3,,C0170,G0171,HEALTHY/HEALTHY_0180.png,masks/HEALTHY_0180_mask.png
HEALTHY_0181,HEALTHY,dev,0,,C0171,G0172,HEALTHY/HEALTHY_0181.png,masks/HEALTHY_0181_mask.png
HEALTHY_0182,HEALTHY,dev,2,,C0172,G0173,HEALTHY/HEALTHY_0182.png,masks/HEALTHY_0182_mask.png
HEALTHY_0183,HEALTHY,dev,4,,C0173,G0174,HEALTHY/HEALTHY_0183.png,masks/HEALTHY_0183_mask.png
HEALTHY_0184,HEALTHY,dev,1,,C0174,G0175,HEALTHY/HEALTHY_0184.png,masks/HEALTHY_0184_mask.png
HEALTHY_0185,HEALTHY,dev,4,,C0175,G0176,HEALTHY/HEALTHY_0185.png,masks/HEALTHY_0185_mask.png
HEALTHY_0186,HEALTHY,dev,0,,C0176,G0177,HEALTHY/HEALTHY_0186.png,masks/HEALTHY_0186_mask.png
HEALTHY_0187,HEALTHY,dev,3,,C0177,G0178,HEALTHY/HEALTHY_0187.png,masks/HEALTHY_0187_mask.png
HEALTHY_0188,HEALTHY,dev,2,,C0178,G0179,HEALTHY/HEALTHY_0188.png,masks/HEALTHY_0188_mask.png
HEALTHY_0189,HEALTHY,dev,-1,near-duplicate of a final_test image,C0179,G0180,HEALTHY/HEALTHY_0189.png,masks/HEALTHY_0189_mask.png
HEALTHY_0190,HEALTHY,dev,-1,near-duplicate of a final_test image,C0180,G0181,HEALTHY/HEALTHY_0190.png,masks/HEALTHY_0190_mask.png
HEALTHY_0191,HEALTHY,final_test,test,,C0179,G0180,HEALTHY/HEALTHY_0191.png,masks/HEALTHY_0191_mask.png
HEALTHY_0192,HEALTHY,final_test,test,,C0180,G0181,HEALTHY/HEALTHY_0192.png,masks/HEALTHY_0192_mask.png
HEALTHY_0193,HEALTHY,final_test,test,,C0181,G0182,HEALTHY/HEALTHY_0193.png,masks/HEALTHY_0193_mask.png
HEALTHY_0194,HEALTHY,final_test,test,,C0182,G0183,HEALTHY/HEALTHY_0194.png,masks/HEALTHY_0194_mask.png
HEALTHY_0195,HEALTHY,final_test,test,,C0183,G0184,HEALTHY/HEALTHY_0195.png,masks/HEALTHY_0195_mask.png
HEALTHY_0196,HEALTHY,dev,0,,C0184,G0185,HEALTHY/HEALTHY_0196.png,masks/HEALTHY_0196_mask.png
HEALTHY_0197,HEALTHY,dev,1,,C0185,G0186,HEALTHY/HEALTHY_0197.png,masks/HEALTHY_0197_mask.png
HEALTHY_0198,HEALTHY,dev,3,,C0186,G0187,HEALTHY/HEALTHY_0198.png,masks/HEALTHY_0198_mask.png
HEALTHY_0199,HEALTHY,dev,4,,C0187,G0188,HEALTHY/HEALTHY_0199.png,masks/HEALTHY_0199_mask.png
HEALTHY_0200,HEALTHY,dev,2,,C0188,G0189,HEALTHY/HEALTHY_0200.png,masks/HEALTHY_0200_mask.png
HEALTHY_0201,HEALTHY,dev,2,,C0188,G0189,HEALTHY/HEALTHY_0201.png,masks/HEALTHY_0201_mask.png
HEALTHY_0202,HEALTHY,dev,2,,C0189,G0190,HEALTHY/HEALTHY_0202.png,masks/HEALTHY_0202_mask.png
HEALTHY_0203,HEALTHY,dev,0,,C0190,G0191,HEALTHY/HEALTHY_0203.png,masks/HEALTHY_0203_mask.png
HEALTHY_0204,HEALTHY,dev,1,,C0191,G0192,HEALTHY/HEALTHY_0204.png,masks/HEALTHY_0204_mask.png
HEALTHY_0205,HEALTHY,dev,3,,C0192,G0193,HEALTHY/HEALTHY_0205.png,masks/HEALTHY_0205_mask.png
HEALTHY_0206,HEALTHY,dev,3,,C0192,G0193,HEALTHY/HEALTHY_0206.png,masks/HEALTHY_0206_mask.png
HEALTHY_0207,HEALTHY,dev,4,,C0193,G0194,HEALTHY/HEALTHY_0207.png,masks/HEALTHY_0207_mask.png
HEALTHY_0208,HEALTHY,final_test,test,,C0194,G0195,HEALTHY/HEALTHY_0208.png,masks/HEALTHY_0208_mask.png
HEALTHY_0209,HEALTHY,dev,-1,near-duplicate of a final_test image,C0141,G0196,HEALTHY/HEALTHY_0209.png,masks/HEALTHY_0209_mask.png
HEALTHY_0210,HEALTHY,dev,2,,C0195,G0197,HEALTHY/HEALTHY_0210.png,masks/HEALTHY_0210_mask.png
HEALTHY_0211,HEALTHY,dev,0,,C0196,G0198,HEALTHY/HEALTHY_0211.png,masks/HEALTHY_0211_mask.png
HEALTHY_0212,HEALTHY,dev,3,,C0197,G0199,HEALTHY/HEALTHY_0212.png,masks/HEALTHY_0212_mask.png
HEALTHY_0213,HEALTHY,dev,1,,C0198,G0200,HEALTHY/HEALTHY_0213.png,masks/HEALTHY_0213_mask.png
HEALTHY_0214,HEALTHY,dev,3,,C0199,G0201,HEALTHY/HEALTHY_0214.png,masks/HEALTHY_0214_mask.png
HEALTHY_0215,HEALTHY,dev,4,,C0200,G0202,HEALTHY/HEALTHY_0215.png,masks/HEALTHY_0215_mask.png
HEALTHY_0216,HEALTHY,dev,4,,C0201,G0203,HEALTHY/HEALTHY_0216.png,masks/HEALTHY_0216_mask.png
HEALTHY_0217,HEALTHY,final_test,test,,C0202,G0204,HEALTHY/HEALTHY_0217.png,masks/HEALTHY_0217_mask.png
HEALTHY_0218,HEALTHY,final_test,test,,C0203,G0205,HEALTHY/HEALTHY_0218.png,masks/HEALTHY_0218_mask.png
HEALTHY_0219,HEALTHY,dev,0,,C0204,G0206,HEALTHY/HEALTHY_0219.png,masks/HEALTHY_0219_mask.png
HEALTHY_0220,HEALTHY,dev,3,,C0205,G0207,HEALTHY/HEALTHY_0220.png,masks/HEALTHY_0220_mask.png
HEALTHY_0221,HEALTHY,dev,2,,C0206,G0208,HEALTHY/HEALTHY_0221.png,masks/HEALTHY_0221_mask.png
HEALTHY_0222,HEALTHY,dev,0,,C0207,G0209,HEALTHY/HEALTHY_0222.png,masks/HEALTHY_0222_mask.png
HEALTHY_0223,HEALTHY,dev,1,,C0208,G0210,HEALTHY/HEALTHY_0223.png,masks/HEALTHY_0223_mask.png
HEALTHY_0224,HEALTHY,dev,4,,C0209,G0211,HEALTHY/HEALTHY_0224.png,masks/HEALTHY_0224_mask.png
HEALTHY_0225,HEALTHY,dev,2,,C0210,G0212,HEALTHY/HEALTHY_0225.png,masks/HEALTHY_0225_mask.png
HEALTHY_0226,HEALTHY,dev,3,,C0211,G0213,HEALTHY/HEALTHY_0226.png,masks/HEALTHY_0226_mask.png
HEALTHY_0227,HEALTHY,dev,3,,C0211,G0213,HEALTHY/HEALTHY_0227.png,masks/HEALTHY_0227_mask.png
HEALTHY_0228,HEALTHY,dev,1,,C0212,G0214,HEALTHY/HEALTHY_0228.png,masks/HEALTHY_0228_mask.png
HEALTHY_0229,HEALTHY,dev,4,,C0213,G0215,HEALTHY/HEALTHY_0229.png,masks/HEALTHY_0229_mask.png
HEALTHY_0230,HEALTHY,dev,0,,C0214,G0216,HEALTHY/HEALTHY_0230.png,masks/HEALTHY_0230_mask.png
HEALTHY_0231,HEALTHY,dev,2,,C0215,G0217,HEALTHY/HEALTHY_0231.png,masks/HEALTHY_0231_mask.png
HEALTHY_0232,HEALTHY,dev,3,,C0216,G0218,HEALTHY/HEALTHY_0232.png,masks/HEALTHY_0232_mask.png
HEALTHY_0233,HEALTHY,dev,4,,C0217,G0219,HEALTHY/HEALTHY_0233.png,masks/HEALTHY_0233_mask.png
HEALTHY_0234,HEALTHY,dev,0,,C0218,G0220,HEALTHY/HEALTHY_0234.png,masks/HEALTHY_0234_mask.png
HEALTHY_0235,HEALTHY,dev,1,,C0219,G0221,HEALTHY/HEALTHY_0235.png,masks/HEALTHY_0235_mask.png
HEALTHY_0236,HEALTHY,dev,4,,C0220,G0222,HEALTHY/HEALTHY_0236.png,masks/HEALTHY_0236_mask.png
HEALTHY_0237,HEALTHY,dev,2,,C0221,G0223,HEALTHY/HEALTHY_0237.png,masks/HEALTHY_0237_mask.png
HEALTHY_0238,HEALTHY,dev,0,,C0222,G0224,HEALTHY/HEALTHY_0238.png,masks/HEALTHY_0238_mask.png
HEALTHY_0239,HEALTHY,dev,3,,C0223,G0225,HEALTHY/HEALTHY_0239.png,masks/HEALTHY_0239_mask.png
HEALTHY_0240,HEALTHY,final_test,test,,C0224,G0226,HEALTHY/HEALTHY_0240.png,masks/HEALTHY_0240_mask.png
HEALTHY_0241,HEALTHY,dev,0,,C0225,G0227,HEALTHY/HEALTHY_0241.png,masks/HEALTHY_0241_mask.png
HEALTHY_0242,HEALTHY,dev,2,,C0226,G0228,HEALTHY/HEALTHY_0242.png,masks/HEALTHY_0242_mask.png
HEALTHY_0243,HEALTHY,dev,1,,C0227,G0229,HEALTHY/HEALTHY_0243.png,masks/HEALTHY_0243_mask.png
HEALTHY_0244,HEALTHY,dev,4,,C0228,G0230,HEALTHY/HEALTHY_0244.png,masks/HEALTHY_0244_mask.png
HEALTHY_0245,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0141,G0231,HEALTHY/HEALTHY_0245.png,masks/HEALTHY_0245_mask.png
HEALTHY_0246,HEALTHY,dev,4,,C0229,G0232,HEALTHY/HEALTHY_0246.png,masks/HEALTHY_0246_mask.png
HEALTHY_0247,HEALTHY,dev,1,,C0230,G0233,HEALTHY/HEALTHY_0247.png,masks/HEALTHY_0247_mask.png
HEALTHY_0248,HEALTHY,dev,-1,near-duplicate of a final_test image,C0231,G0234,HEALTHY/HEALTHY_0248.png,masks/HEALTHY_0248_mask.png
HEALTHY_0249,HEALTHY,final_test,test,,C0232,G0235,HEALTHY/HEALTHY_0249.png,masks/HEALTHY_0249_mask.png
HEALTHY_0250,HEALTHY,dev,0,,C0233,G0236,HEALTHY/HEALTHY_0250.png,masks/HEALTHY_0250_mask.png
HEALTHY_0251,HEALTHY,final_test,test,,C0234,G0237,HEALTHY/HEALTHY_0251.png,masks/HEALTHY_0251_mask.png
HEALTHY_0252,HEALTHY,dev,4,,C0235,G0238,HEALTHY/HEALTHY_0252.png,masks/HEALTHY_0252_mask.png
HEALTHY_0253,HEALTHY,dev,3,,C0236,G0239,HEALTHY/HEALTHY_0253.png,masks/HEALTHY_0253_mask.png
HEALTHY_0254,HEALTHY,dev,4,,C0138,G0139,HEALTHY/HEALTHY_0254.png,masks/HEALTHY_0254_mask.png
HEALTHY_0255,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0237,G0240,HEALTHY/HEALTHY_0255.png,masks/HEALTHY_0255_mask.png
HEALTHY_0256,HEALTHY,dev,0,,C0238,G0241,HEALTHY/HEALTHY_0256.png,masks/HEALTHY_0256_mask.png
HEALTHY_0257,HEALTHY,dev,1,,C0239,G0242,HEALTHY/HEALTHY_0257.png,masks/HEALTHY_0257_mask.png
HEALTHY_0258,HEALTHY,dev,2,,C0240,G0243,HEALTHY/HEALTHY_0258.png,masks/HEALTHY_0258_mask.png
HEALTHY_0259,HEALTHY,dev,4,,C0241,G0244,HEALTHY/HEALTHY_0259.png,masks/HEALTHY_0259_mask.png
HEALTHY_0260,HEALTHY,dev,4,,C0242,G0245,HEALTHY/HEALTHY_0260.png,masks/HEALTHY_0260_mask.png
HEALTHY_0261,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0013,G0246,HEALTHY/HEALTHY_0261.png,masks/HEALTHY_0261_mask.png
HEALTHY_0262,HEALTHY,dev,3,,C0243,G0247,HEALTHY/HEALTHY_0262.png,masks/HEALTHY_0262_mask.png
HEALTHY_0263,HEALTHY,dev,2,,C0244,G0248,HEALTHY/HEALTHY_0263.png,masks/HEALTHY_0263_mask.png
HEALTHY_0264,HEALTHY,final_test,test,,C0245,G0249,HEALTHY/HEALTHY_0264.png,masks/HEALTHY_0264_mask.png
HEALTHY_0265,HEALTHY,dev,-1,near-duplicate of a final_test image,C0245,G0249,HEALTHY/HEALTHY_0265.png,masks/HEALTHY_0265_mask.png
HEALTHY_0266,HEALTHY,dev,1,,C0246,G0250,HEALTHY/HEALTHY_0266.png,masks/HEALTHY_0266_mask.png
HEALTHY_0267,HEALTHY,dev,1,,C0247,G0251,HEALTHY/HEALTHY_0267.png,masks/HEALTHY_0267_mask.png
HEALTHY_0268,HEALTHY,dev,1,,C0248,G0252,HEALTHY/HEALTHY_0268.png,masks/HEALTHY_0268_mask.png
HEALTHY_0269,HEALTHY,dev,0,,C0249,G0253,HEALTHY/HEALTHY_0269.png,masks/HEALTHY_0269_mask.png
HEALTHY_0270,HEALTHY,dev,2,,C0250,G0254,HEALTHY/HEALTHY_0270.png,masks/HEALTHY_0270_mask.png
HEALTHY_0271,HEALTHY,dev,3,,C0251,G0255,HEALTHY/HEALTHY_0271.png,masks/HEALTHY_0271_mask.png
HEALTHY_0272,HEALTHY,dev,4,,C0252,G0256,HEALTHY/HEALTHY_0272.png,masks/HEALTHY_0272_mask.png
HEALTHY_0273,HEALTHY,dev,2,,C0253,G0257,HEALTHY/HEALTHY_0273.png,masks/HEALTHY_0273_mask.png
HEALTHY_0274,HEALTHY,dev,1,,C0254,G0258,HEALTHY/HEALTHY_0274.png,masks/HEALTHY_0274_mask.png
HEALTHY_0275,HEALTHY,dev,0,,C0255,G0259,HEALTHY/HEALTHY_0275.png,masks/HEALTHY_0275_mask.png
HEALTHY_0276,HEALTHY,final_test,test,,C0256,G0260,HEALTHY/HEALTHY_0276.png,masks/HEALTHY_0276_mask.png
HEALTHY_0277,HEALTHY,dev,3,,C0257,G0261,HEALTHY/HEALTHY_0277.png,masks/HEALTHY_0277_mask.png
HEALTHY_0278,HEALTHY,dev,4,,C0258,G0262,HEALTHY/HEALTHY_0278.png,masks/HEALTHY_0278_mask.png
HEALTHY_0279,HEALTHY,final_test,test,,C0259,G0263,HEALTHY/HEALTHY_0279.png,masks/HEALTHY_0279_mask.png
HEALTHY_0280,HEALTHY,dev,0,,C0260,G0264,HEALTHY/HEALTHY_0280.png,masks/HEALTHY_0280_mask.png
HEALTHY_0281,HEALTHY,final_test,test,,C0261,G0265,HEALTHY/HEALTHY_0281.png,masks/HEALTHY_0281_mask.png
HEALTHY_0282,HEALTHY,final_test,test,,C0262,G0266,HEALTHY/HEALTHY_0282.png,masks/HEALTHY_0282_mask.png
HEALTHY_0283,HEALTHY,final_test,test,,C0263,G0267,HEALTHY/HEALTHY_0283.png,masks/HEALTHY_0283_mask.png
HEALTHY_0284,HEALTHY,dev,1,,C0264,G0268,HEALTHY/HEALTHY_0284.png,masks/HEALTHY_0284_mask.png
HEALTHY_0285,HEALTHY,dev,3,,C0265,G0269,HEALTHY/HEALTHY_0285.png,masks/HEALTHY_0285_mask.png
HEALTHY_0286,HEALTHY,dev,2,,C0266,G0270,HEALTHY/HEALTHY_0286.png,masks/HEALTHY_0286_mask.png
HEALTHY_0287,HEALTHY,final_test,test,,C0267,G0271,HEALTHY/HEALTHY_0287.png,masks/HEALTHY_0287_mask.png
HEALTHY_0288,HEALTHY,dev,-1,near-duplicate of a final_test image,C0267,G0271,HEALTHY/HEALTHY_0288.png,masks/HEALTHY_0288_mask.png
HEALTHY_0289,HEALTHY,dev,4,,C0268,G0272,HEALTHY/HEALTHY_0289.png,masks/HEALTHY_0289_mask.png
HEALTHY_0290,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0013,G0273,HEALTHY/HEALTHY_0290.png,masks/HEALTHY_0290_mask.png
HEALTHY_0291,HEALTHY,dev,0,,C0269,G0274,HEALTHY/HEALTHY_0291.png,masks/HEALTHY_0291_mask.png
HEALTHY_0292,HEALTHY,dev,2,,C0270,G0275,HEALTHY/HEALTHY_0292.png,masks/HEALTHY_0292_mask.png
HEALTHY_0293,HEALTHY,dev,3,,C0271,G0276,HEALTHY/HEALTHY_0293.png,masks/HEALTHY_0293_mask.png
HEALTHY_0294,HEALTHY,dev,-1,"linked to a test image (same case, source records)",C0272,G0277,HEALTHY/HEALTHY_0294.png,masks/HEALTHY_0294_mask.png
HEALTHY_0295,HEALTHY,final_test,test,,C0273,G0278,HEALTHY/HEALTHY_0295.png,masks/HEALTHY_0295_mask.png
HEALTHY_0296,HEALTHY,final_test,test,,C0231,G0234,HEALTHY/HEALTHY_0296.png,masks/HEALTHY_0296_mask.png
HEALTHY_0297,HEALTHY,dev,1,,C0274,G0279,HEALTHY/HEALTHY_0297.png,masks/HEALTHY_0297_mask.png
HEALTHY_0298,HEALTHY,dev,0,,C0275,G0280,HEALTHY/HEALTHY_0298.png,masks/HEALTHY_0298_mask.png
HEALTHY_0299,HEALTHY,dev,1,,C0276,G0281,HEALTHY/HEALTHY_0299.png,masks/HEALTHY_0299_mask.png
HEALTHY_0300,HEALTHY,dev,3,,C0277,G0282,HEALTHY/HEALTHY_0300.png,masks/HEALTHY_0300_mask.png
HEALTHY_0301,HEALTHY,dev,4,,C0278,G0283,HEALTHY/HEALTHY_0301.png,masks/HEALTHY_0301_mask.png
HEALTHY_0302,HEALTHY,dev,0,,C0279,G0284,HEALTHY/HEALTHY_0302.png,masks/HEALTHY_0302_mask.png
HEALTHY_0303,HEALTHY,final_test,test,,C0280,G0285,HEALTHY/HEALTHY_0303.png,masks/HEALTHY_0303_mask.png
HEALTHY_0304,HEALTHY,dev,1,,C0281,G0286,HEALTHY/HEALTHY_0304.png,masks/HEALTHY_0304_mask.png
HEALTHY_0305,HEALTHY,dev,2,,C0282,G0287,HEALTHY/HEALTHY_0305.png,masks/HEALTHY_0305_mask.png
HEALTHY_0306,HEALTHY,dev,2,,C0283,G0288,HEALTHY/HEALTHY_0306.png,masks/HEALTHY_0306_mask.png
HEALTHY_0307,HEALTHY,final_test,test,,C0284,G0289,HEALTHY/HEALTHY_0307.png,masks/HEALTHY_0307_mask.png
HEALTHY_0308,HEALTHY,dev,3,,C0285,G0290,HEALTHY/HEALTHY_0308.png,masks/HEALTHY_0308_mask.png
HEALTHY_0309,HEALTHY,dev,4,,C0286,G0291,HEALTHY/HEALTHY_0309.png,masks/HEALTHY_0309_mask.png
HEALTHY_0310,HEALTHY,final_test,test,,C0287,G0292,HEALTHY/HEALTHY_0310.png,masks/HEALTHY_0310_mask.png
HEALTHY_0311,HEALTHY,dev,0,,C0288,G0293,HEALTHY/HEALTHY_0311.png,masks/HEALTHY_0311_mask.png
RT1_0312,RT1,dev,0,,C0104,G0294,RT1/RT1_0312.png,masks/RT1_0312_mask.png
RT1_0313,RT1,final_test,test,,C0289,G0295,RT1/RT1_0313.png,masks/RT1_0313_mask.png
RT1_0314,RT1,dev,1,,C0290,G0296,RT1/RT1_0314.png,masks/RT1_0314_mask.png
RT1_0315,RT1,dev,3,,C0049,G0297,RT1/RT1_0315.png,masks/RT1_0315_mask.png
RT1_0316,RT1,dev,-1,"linked to a test image (same case, source records)",C0050,G0298,RT1/RT1_0316.png,masks/RT1_0316_mask.png
RT1_0317,RT1,dev,2,,C0291,G0299,RT1/RT1_0317.png,masks/RT1_0317_mask.png
RT1_0318,RT1,dev,1,,C0051,G0300,RT1/RT1_0318.png,masks/RT1_0318_mask.png
RT1_0319,RT1,dev,3,,C0292,G0301,RT1/RT1_0319.png,masks/RT1_0319_mask.png
RT1_0320,RT1,dev,4,,C0293,G0302,RT1/RT1_0320.png,masks/RT1_0320_mask.png
RT1_0321,RT1,dev,-1,"linked to a test image (same case, source records)",C0056,G0303,RT1/RT1_0321.png,masks/RT1_0321_mask.png
RT1_0322,RT1,dev,0,,C0294,G0304,RT1/RT1_0322.png,masks/RT1_0322_mask.png
RT1_0323,RT1,dev,2,,C0295,G0305,RT1/RT1_0323.png,masks/RT1_0323_mask.png
RT1_0324,RT1,final_test,test,,C0296,G0306,RT1/RT1_0324.png,masks/RT1_0324_mask.png
RT1_0325,RT1,dev,3,,C0297,G0307,RT1/RT1_0325.png,masks/RT1_0325_mask.png
RT1_0326,RT1,final_test,test,,C0298,G0308,RT1/RT1_0326.png,masks/RT1_0326_mask.png
RT1_0327,RT1,dev,-1,"linked to a test image (same case, source records)",C0064,G0309,RT1/RT1_0327.png,masks/RT1_0327_mask.png
RT1_0328,RT1,dev,4,,C0299,G0310,RT1/RT1_0328.png,masks/RT1_0328_mask.png
RT1_0329,RT1,dev,2,,C0066,G0311,RT1/RT1_0329.png,masks/RT1_0329_mask.png
RT1_0330,RT1,dev,-1,"linked to a test image (same case, source records)",C0069,G0312,RT1/RT1_0330.png,masks/RT1_0330_mask.png
RT1_0331,RT1,dev,2,,C0300,G0313,RT1/RT1_0331.png,masks/RT1_0331_mask.png
RT1_0332,RT1,dev,-1,"linked to a test image (same case, source records)",C0071,G0314,RT1/RT1_0332.png,masks/RT1_0332_mask.png
RT1_0333,RT1,dev,2,,C0301,G0315,RT1/RT1_0333.png,masks/RT1_0333_mask.png
RT1_0334,RT1,dev,0,,C0302,G0316,RT1/RT1_0334.png,masks/RT1_0334_mask.png
RT1_0335,RT1,dev,1,,C0303,G0317,RT1/RT1_0335.png,masks/RT1_0335_mask.png
RT1_0336,RT1,dev,2,,C0304,G0318,RT1/RT1_0336.png,masks/RT1_0336_mask.png
RT1_0337,RT1,dev,2,,C0304,G0318,RT1/RT1_0337.png,masks/RT1_0337_mask.png
RT1_0338,RT1,dev,3,,C0305,G0319,RT1/RT1_0338.png,masks/RT1_0338_mask.png
RT1_0339,RT1,dev,4,,C0306,G0320,RT1/RT1_0339.png,masks/RT1_0339_mask.png
RT1_0340,RT1,dev,0,,C0307,G0321,RT1/RT1_0340.png,masks/RT1_0340_mask.png
RT1_0341,RT1,dev,1,,C0077,G0322,RT1/RT1_0341.png,masks/RT1_0341_mask.png
RT1_0342,RT1,dev,3,,C0308,G0323,RT1/RT1_0342.png,masks/RT1_0342_mask.png
RT1_0343,RT1,final_test,test,,C0309,G0324,RT1/RT1_0343.png,masks/RT1_0343_mask.png
RT1_0344,RT1,dev,4,,C0310,G0325,RT1/RT1_0344.png,masks/RT1_0344_mask.png
RT1_0345,RT1,dev,3,,C0311,G0326,RT1/RT1_0345.png,masks/RT1_0345_mask.png
RT1_0346,RT1,dev,2,,C0105,G0327,RT1/RT1_0346.png,masks/RT1_0346_mask.png
RT1_0347,RT1,dev,0,,C0312,G0328,RT1/RT1_0347.png,masks/RT1_0347_mask.png
RT1_0348,RT1,dev,1,,C0313,G0329,RT1/RT1_0348.png,masks/RT1_0348_mask.png
RT1_0349,RT1,dev,4,,C0314,G0330,RT1/RT1_0349.png,masks/RT1_0349_mask.png
RT1_0350,RT1,dev,-1,near-duplicate of a final_test image,C0141,G0196,RT1/RT1_0350.png,masks/RT1_0350_mask.png
RT1_0351,RT1,dev,4,,C0080,G0331,RT1/RT1_0351.png,masks/RT1_0351_mask.png
RT1_0352,RT1,dev,0,,C0315,G0332,RT1/RT1_0352.png,masks/RT1_0352_mask.png
RT1_0353,RT1,dev,3,,C0316,G0333,RT1/RT1_0353.png,masks/RT1_0353_mask.png
RT1_0354,RT1,dev,1,,C0317,G0334,RT1/RT1_0354.png,masks/RT1_0354_mask.png
RT1_0355,RT1,final_test,test,,C0318,G0335,RT1/RT1_0355.png,masks/RT1_0355_mask.png
RT1_0356,RT1,final_test,test,,C0319,G0336,RT1/RT1_0356.png,masks/RT1_0356_mask.png
RT1_0357,RT1,dev,2,,C0127,G0337,RT1/RT1_0357.png,masks/RT1_0357_mask.png
RT1_0358,RT1,dev,0,,C0320,G0338,RT1/RT1_0358.png,masks/RT1_0358_mask.png
RT1_0359,RT1,final_test,test,,C0321,G0339,RT1/RT1_0359.png,masks/RT1_0359_mask.png
RT1_0360,RT1,dev,1,,C0322,G0340,RT1/RT1_0360.png,masks/RT1_0360_mask.png
RT1_0361,RT1,dev,2,,C0323,G0341,RT1/RT1_0361.png,masks/RT1_0361_mask.png
RT1_0362,RT1,dev,4,,C0087,G0342,RT1/RT1_0362.png,masks/RT1_0362_mask.png
RT1_0363,RT1,dev,4,,C0087,G0342,RT1/RT1_0363.png,masks/RT1_0363_mask.png
RT1_0364,RT1,dev,3,,C0324,G0343,RT1/RT1_0364.png,masks/RT1_0364_mask.png
RT1_0365,RT1,dev,0,,C0325,G0344,RT1/RT1_0365.png,masks/RT1_0365_mask.png
RT1_0366,RT1,final_test,test,,C0090,G0345,RT1/RT1_0366.png,masks/RT1_0366_mask.png
RT1_0367,RT1,dev,-1,near-duplicate of a final_test image,C0090,G0345,RT1/RT1_0367.png,masks/RT1_0367_mask.png
RT1_0368,RT1,dev,2,,C0135,G0346,RT1/RT1_0368.png,masks/RT1_0368_mask.png
RT1_0369,RT1,dev,1,,C0091,G0347,RT1/RT1_0369.png,masks/RT1_0369_mask.png
RT1_0370,RT1,dev,3,,C0326,G0348,RT1/RT1_0370.png,masks/RT1_0370_mask.png
RT1_0371,RT1,dev,4,,C0327,G0349,RT1/RT1_0371.png,masks/RT1_0371_mask.png
RT1_0372,RT1,dev,-1,"linked to a test image (same case, source records)",C0095,G0350,RT1/RT1_0372.png,masks/RT1_0372_mask.png
RT1_0373,RT1,dev,-1,"linked to a test image (same case, source records)",C0096,G0351,RT1/RT1_0373.png,masks/RT1_0373_mask.png
RT1_0374,RT1,final_test,test,,C0328,G0352,RT1/RT1_0374.png,masks/RT1_0374_mask.png
RT1_0375,RT1,dev,1,,C0329,G0353,RT1/RT1_0375.png,masks/RT1_0375_mask.png
RT1_0376,RT1,dev,0,,C0330,G0354,RT1/RT1_0376.png,masks/RT1_0376_mask.png
RT1_0377,RT1,dev,3,,C0331,G0355,RT1/RT1_0377.png,masks/RT1_0377_mask.png
RT1_0378,RT1,dev,-1,near-duplicate of a final_test image,C0141,G0196,RT1/RT1_0378.png,masks/RT1_0378_mask.png
RT1_0379,RT1,dev,-1,near-duplicate of a final_test image,C0141,G0196,RT1/RT1_0379.png,masks/RT1_0379_mask.png
RT1_0380,RT1,dev,4,,C0332,G0356,RT1/RT1_0380.png,masks/RT1_0380_mask.png
RT1_0381,RT1,dev,0,,C0333,G0357,RT1/RT1_0381.png,masks/RT1_0381_mask.png
RT1_0382,RT1,dev,4,,C0334,G0358,RT1/RT1_0382.png,masks/RT1_0382_mask.png
RT1_0383,RT1,dev,1,,C0335,G0359,RT1/RT1_0383.png,masks/RT1_0383_mask.png
RT1_0384,RT1,dev,0,,C0102,G0103,RT1/RT1_0384.png,masks/RT1_0384_mask.png
RT1_0385,RT1,final_test,test,,C0103,G0104,RT1/RT1_0385.png,masks/RT1_0385_mask.png
RT1_0386,RT1,dev,4,,C0106,G0360,RT1/RT1_0386.png,masks/RT1_0386_mask.png
RT1_0387,RT1,final_test,test,,C0336,G0361,RT1/RT1_0387.png,masks/RT1_0387_mask.png
RT1_0388,RT1,final_test,test,,C0337,G0362,RT1/RT1_0388.png,masks/RT1_0388_mask.png
RT1_0389,RT1,dev,4,,C0109,G0363,RT1/RT1_0389.png,masks/RT1_0389_mask.png
RT1_0390,RT1,dev,2,,C0338,G0364,RT1/RT1_0390.png,masks/RT1_0390_mask.png
RT1_0391,RT1,dev,1,,C0339,G0365,RT1/RT1_0391.png,masks/RT1_0391_mask.png
RT1_0392,RT1,dev,0,,C0340,G0366,RT1/RT1_0392.png,masks/RT1_0392_mask.png
RT1_0393,RT1,final_test,test,,C0112,G0367,RT1/RT1_0393.png,masks/RT1_0393_mask.png
RT1_0394,RT1,final_test,test,,C0112,G0367,RT1/RT1_0394.png,masks/RT1_0394_mask.png
RT1_0395,RT1,dev,3,,C0113,G0368,RT1/RT1_0395.png,masks/RT1_0395_mask.png
RT1_0396,RT1,dev,4,,C0341,G0369,RT1/RT1_0396.png,masks/RT1_0396_mask.png
RT1_0397,RT1,dev,-1,near-duplicate of a final_test image,C0114,G0115,RT1/RT1_0397.png,masks/RT1_0397_mask.png
RT1_0398,RT1,dev,2,,C0115,G0116,RT1/RT1_0398.png,masks/RT1_0398_mask.png
RT1_0399,RT1,dev,1,,C0342,G0370,RT1/RT1_0399.png,masks/RT1_0399_mask.png
RT1_0400,RT1,dev,0,,C0117,G0371,RT1/RT1_0400.png,masks/RT1_0400_mask.png
RT1_0401,RT1,dev,2,,C0343,G0372,RT1/RT1_0401.png,masks/RT1_0401_mask.png
RT1_0402,RT1,dev,0,,C0344,G0373,RT1/RT1_0402.png,masks/RT1_0402_mask.png
RT1_0403,RT1,dev,3,,C0345,G0374,RT1/RT1_0403.png,masks/RT1_0403_mask.png
RT1_0404,RT1,dev,4,,C0346,G0375,RT1/RT1_0404.png,masks/RT1_0404_mask.png
RT1_0405,RT1,dev,1,,C0121,G0122,RT1/RT1_0405.png,masks/RT1_0405_mask.png
RT1_0406,RT1,dev,2,,C0347,G0376,RT1/RT1_0406.png,masks/RT1_0406_mask.png
RT1_0407,RT1,dev,1,,C0348,G0377,RT1/RT1_0407.png,masks/RT1_0407_mask.png
RT1_0408,RT1,final_test,test,,C0349,G0378,RT1/RT1_0408.png,masks/RT1_0408_mask.png
RT1_0409,RT1,dev,3,,C0350,G0379,RT1/RT1_0409.png,masks/RT1_0409_mask.png
RT1_0410,RT1,dev,1,,C0351,G0380,RT1/RT1_0410.png,masks/RT1_0410_mask.png
RT1_0411,RT1,final_test,test,,C0352,G0381,RT1/RT1_0411.png,masks/RT1_0411_mask.png
RT1_0412,RT1,final_test,test,,C0141,G0196,RT1/RT1_0412.png,masks/RT1_0412_mask.png
RT1_0413,RT1,final_test,test,,C0353,G0382,RT1/RT1_0413.png,masks/RT1_0413_mask.png
RT1_0414,RT1,dev,2,,C0354,G0383,RT1/RT1_0414.png,masks/RT1_0414_mask.png
RT1_0415,RT1,final_test,test,,C0355,G0384,RT1/RT1_0415.png,masks/RT1_0415_mask.png
RT1_0416,RT1,dev,4,,C0356,G0385,RT1/RT1_0416.png,masks/RT1_0416_mask.png
RT1_0417,RT1,dev,0,,C0357,G0386,RT1/RT1_0417.png,masks/RT1_0417_mask.png
RT1_0418,RT1,final_test,test,,C0358,G0387,RT1/RT1_0418.png,masks/RT1_0418_mask.png
RT1_0419,RT1,dev,3,,C0359,G0388,RT1/RT1_0419.png,masks/RT1_0419_mask.png
RT1_0420,RT1,dev,-1,near-duplicate of a final_test image,C0141,G0196,RT1/RT1_0420.png,masks/RT1_0420_mask.png
RT1_0421,RT1,dev,-1,near-duplicate of a final_test image,C0141,G0196,RT1/RT1_0421.png,masks/RT1_0421_mask.png
RT1_0422,RT1,dev,4,,C0360,G0389,RT1/RT1_0422.png,masks/RT1_0422_mask.png
RT1_0423,RT1,dev,-1,near-duplicate of a final_test image,C0237,G0390,RT1/RT1_0423.png,masks/RT1_0423_mask.png
RT1_0424,RT1,final_test,test,,C0142,G0391,RT1/RT1_0424.png,masks/RT1_0424_mask.png
RT1_0425,RT1,dev,0,,C0361,G0392,RT1/RT1_0425.png,masks/RT1_0425_mask.png
RT1_0426,RT1,final_test,test,,C0144,G0393,RT1/RT1_0426.png,masks/RT1_0426_mask.png
RT1_0427,RT1,dev,1,,C0362,G0394,RT1/RT1_0427.png,masks/RT1_0427_mask.png
RT1_0428,RT1,dev,2,,C0363,G0395,RT1/RT1_0428.png,masks/RT1_0428_mask.png
RT1_0429,RT1,dev,0,,C0364,G0396,RT1/RT1_0429.png,masks/RT1_0429_mask.png
RT1_0430,RT1,final_test,test,,C0146,G0397,RT1/RT1_0430.png,masks/RT1_0430_mask.png
RT1_0431,RT1,final_test,test,,C0147,G0398,RT1/RT1_0431.png,masks/RT1_0431_mask.png
RT1_0432,RT1,dev,3,,C0148,G0399,RT1/RT1_0432.png,masks/RT1_0432_mask.png
RT1_0433,RT1,dev,4,,C0149,G0400,RT1/RT1_0433.png,masks/RT1_0433_mask.png
RT1_0434,RT1,dev,4,,C0229,G0401,RT1/RT1_0434.png,masks/RT1_0434_mask.png
RT1_0435,RT1,dev,1,,C0365,G0402,RT1/RT1_0435.png,masks/RT1_0435_mask.png
RT1_0436,RT1,final_test,test,,C0366,G0403,RT1/RT1_0436.png,masks/RT1_0436_mask.png
RT1_0437,RT1,dev,2,,C0367,G0404,RT1/RT1_0437.png,masks/RT1_0437_mask.png
RT1_0438,RT1,dev,3,,C0368,G0405,RT1/RT1_0438.png,masks/RT1_0438_mask.png
RT1_0439,RT1,final_test,test,,C0369,G0406,RT1/RT1_0439.png,masks/RT1_0439_mask.png
RT1_0440,RT1,dev,0,,C0151,G0407,RT1/RT1_0440.png,masks/RT1_0440_mask.png
RT1_0441,RT1,dev,1,,C0370,G0408,RT1/RT1_0441.png,masks/RT1_0441_mask.png
RT1_0442,RT1,dev,2,,C0371,G0409,RT1/RT1_0442.png,masks/RT1_0442_mask.png
RT1_0443,RT1,dev,3,,C0372,G0410,RT1/RT1_0443.png,masks/RT1_0443_mask.png
RT1_0444,RT1,dev,2,,C0155,G0411,RT1/RT1_0444.png,masks/RT1_0444_mask.png
RT1_0445,RT1,dev,0,,C0373,G0412,RT1/RT1_0445.png,masks/RT1_0445_mask.png
RT1_0446,RT1,final_test,test,,C0374,G0413,RT1/RT1_0446.png,masks/RT1_0446_mask.png
RT1_0447,RT1,dev,1,,C0375,G0414,RT1/RT1_0447.png,masks/RT1_0447_mask.png
RT1_0448,RT1,dev,3,,C0376,G0415,RT1/RT1_0448.png,masks/RT1_0448_mask.png
RT1_0449,RT1,dev,3,,C0377,G0416,RT1/RT1_0449.png,masks/RT1_0449_mask.png
RT1_0450,RT1,dev,2,,C0378,G0417,RT1/RT1_0450.png,masks/RT1_0450_mask.png
RT1_0451,RT1,dev,4,,C0159,G0418,RT1/RT1_0451.png,masks/RT1_0451_mask.png
RT1_0452,RT1,dev,2,,C0160,G0419,RT1/RT1_0452.png,masks/RT1_0452_mask.png
RT1_0453,RT1,dev,0,,C0163,G0420,RT1/RT1_0453.png,masks/RT1_0453_mask.png
RT1_0454,RT1,final_test,test,,C0379,G0421,RT1/RT1_0454.png,masks/RT1_0454_mask.png
RT1_0455,RT1,dev,2,,C0380,G0422,RT1/RT1_0455.png,masks/RT1_0455_mask.png
RT1_0456,RT1,final_test,test,,C0237,G0390,RT1/RT1_0456.png,masks/RT1_0456_mask.png
RT1_0457,RT1,dev,1,,C0165,G0423,RT1/RT1_0457.png,masks/RT1_0457_mask.png
RT1_0458,RT1,final_test,test,,C0381,G0424,RT1/RT1_0458.png,masks/RT1_0458_mask.png
RT1_0459,RT1,dev,4,,C0382,G0425,RT1/RT1_0459.png,masks/RT1_0459_mask.png
RT1_0460,RT1,dev,3,,C0383,G0426,RT1/RT1_0460.png,masks/RT1_0460_mask.png
RT1_0461,RT1,dev,0,,C0384,G0427,RT1/RT1_0461.png,masks/RT1_0461_mask.png
RT1_0462,RT1,dev,1,,C0385,G0428,RT1/RT1_0462.png,masks/RT1_0462_mask.png
RT1_0463,RT1,dev,3,,C0386,G0429,RT1/RT1_0463.png,masks/RT1_0463_mask.png
RT1_0464,RT1,final_test,test,,C0387,G0430,RT1/RT1_0464.png,masks/RT1_0464_mask.png
RT1_0465,RT1,dev,4,,C0388,G0431,RT1/RT1_0465.png,masks/RT1_0465_mask.png
RT1_0466,RT1,dev,2,,C0389,G0432,RT1/RT1_0466.png,masks/RT1_0466_mask.png
RT1_0467,RT1,dev,-1,near-duplicate of a final_test image,C0232,G0235,RT1/RT1_0467.png,masks/RT1_0467_mask.png
RT1_0468,RT1,dev,0,,C0390,G0433,RT1/RT1_0468.png,masks/RT1_0468_mask.png
RT1_0469,RT1,final_test,test,,C0391,G0434,RT1/RT1_0469.png,masks/RT1_0469_mask.png
RT1_0470,RT1,dev,3,,C0392,G0435,RT1/RT1_0470.png,masks/RT1_0470_mask.png
RT1_0471,RT1,final_test,test,,C0393,G0436,RT1/RT1_0471.png,masks/RT1_0471_mask.png
RT1_0472,RT1,dev,4,,C0394,G0437,RT1/RT1_0472.png,masks/RT1_0472_mask.png
RT1_0473,RT1,dev,4,,C0394,G0437,RT1/RT1_0473.png,masks/RT1_0473_mask.png
RT1_0474,RT1,dev,1,,C0395,G0438,RT1/RT1_0474.png,masks/RT1_0474_mask.png
RT1_0475,RT1,dev,0,,C0396,G0439,RT1/RT1_0475.png,masks/RT1_0475_mask.png
RT1_0476,RT1,dev,1,,C0397,G0440,RT1/RT1_0476.png,masks/RT1_0476_mask.png
RT1_0477,RT1,dev,2,,C0398,G0441,RT1/RT1_0477.png,masks/RT1_0477_mask.png
RT1_0478,RT1,dev,-1,"linked to a test image (same case, source records)",C0234,G0442,RT1/RT1_0478.png,masks/RT1_0478_mask.png
RT1_0479,RT1,dev,4,,C0399,G0443,RT1/RT1_0479.png,masks/RT1_0479_mask.png
RT1_0480,RT1,final_test,test,,C0400,G0444,RT1/RT1_0480.png,masks/RT1_0480_mask.png
RT1_0481,RT1,dev,-1,near-duplicate of a final_test image,C0141,G0196,RT1/RT1_0481.png,masks/RT1_0481_mask.png
RT1_0482,RT1,dev,0,,C0196,G0198,RT1/RT1_0482.png,masks/RT1_0482_mask.png
RT1_0483,RT1,dev,0,,C0401,G0445,RT1/RT1_0483.png,masks/RT1_0483_mask.png
RT1_0484,RT1,final_test,test,,C0402,G0446,RT1/RT1_0484.png,masks/RT1_0484_mask.png
RT1_0485,RT1,final_test,test,,C0403,G0447,RT1/RT1_0485.png,masks/RT1_0485_mask.png
RT1_0486,RT1,dev,-1,"linked to a test image (same case, source records)",C0203,G0448,RT1/RT1_0486.png,masks/RT1_0486_mask.png
RT1_0487,RT1,dev,1,,C0404,G0449,RT1/RT1_0487.png,masks/RT1_0487_mask.png
RT1_0488,RT1,dev,3,,C0205,G0450,RT1/RT1_0488.png,masks/RT1_0488_mask.png
RT1_0489,RT1,dev,1,,C0405,G0451,RT1/RT1_0489.png,masks/RT1_0489_mask.png
RT1_0490,RT1,dev,1,,C0208,G0452,RT1/RT1_0490.png,masks/RT1_0490_mask.png
RT1_0491,RT1,dev,0,,C0406,G0453,RT1/RT1_0491.png,masks/RT1_0491_mask.png
RT1_0492,RT1,dev,4,,C0407,G0454,RT1/RT1_0492.png,masks/RT1_0492_mask.png
RT1_0493,RT1,dev,3,,C0408,G0455,RT1/RT1_0493.png,masks/RT1_0493_mask.png
RT1_0494,RT1,dev,2,,C0409,G0456,RT1/RT1_0494.png,masks/RT1_0494_mask.png
RT1_0495,RT1,dev,0,,C0410,G0457,RT1/RT1_0495.png,masks/RT1_0495_mask.png
RT1_0496,RT1,dev,0,,C0410,G0457,RT1/RT1_0496.png,masks/RT1_0496_mask.png
RT1_0497,RT1,final_test,test,,C0411,G0458,RT1/RT1_0497.png,masks/RT1_0497_mask.png
RT1_0498,RT1,final_test,test,,C0411,G0458,RT1/RT1_0498.png,masks/RT1_0498_mask.png
RT1_0499,RT1,dev,2,,C0412,G0459,RT1/RT1_0499.png,masks/RT1_0499_mask.png
RT1_0500,RT1,dev,2,,C0413,G0460,RT1/RT1_0500.png,masks/RT1_0500_mask.png
RT1_0501,RT1,dev,1,,C0414,G0461,RT1/RT1_0501.png,masks/RT1_0501_mask.png
RT1_0502,RT1,dev,3,,C0415,G0462,RT1/RT1_0502.png,masks/RT1_0502_mask.png
RT1_0503,RT1,dev,4,,C0416,G0463,RT1/RT1_0503.png,masks/RT1_0503_mask.png
RT1_0504,RT1,final_test,test,,C0417,G0464,RT1/RT1_0504.png,masks/RT1_0504_mask.png
RT1_0505,RT1,dev,0,,C0418,G0465,RT1/RT1_0505.png,masks/RT1_0505_mask.png
RT1_0506,RT1,dev,1,,C0419,G0466,RT1/RT1_0506.png,masks/RT1_0506_mask.png
RT1_0507,RT1,dev,3,,C0420,G0467,RT1/RT1_0507.png,masks/RT1_0507_mask.png
RT1_0508,RT1,dev,3,,C0420,G0467,RT1/RT1_0508.png,masks/RT1_0508_mask.png
RT1_0509,RT1,dev,4,,C0421,G0468,RT1/RT1_0509.png,masks/RT1_0509_mask.png
RT1_0510,RT1,dev,0,,C0422,G0469,RT1/RT1_0510.png,masks/RT1_0510_mask.png
RT1_0511,RT1,dev,1,,C0423,G0470,RT1/RT1_0511.png,masks/RT1_0511_mask.png
RT1_0512,RT1,final_test,test,,C0424,G0471,RT1/RT1_0512.png,masks/RT1_0512_mask.png
RT1_0513,RT1,final_test,test,,C0425,G0472,RT1/RT1_0513.png,masks/RT1_0513_mask.png
RT1_0514,RT1,dev,1,,C0426,G0473,RT1/RT1_0514.png,masks/RT1_0514_mask.png
RT1_0515,RT1,dev,2,,C0427,G0474,RT1/RT1_0515.png,masks/RT1_0515_mask.png
RT1_0516,RT1,dev,3,,C0428,G0475,RT1/RT1_0516.png,masks/RT1_0516_mask.png
RT1_0517,RT1,dev,4,,C0429,G0476,RT1/RT1_0517.png,masks/RT1_0517_mask.png
RT1_0518,RT1,dev,2,,C0430,G0477,RT1/RT1_0518.png,masks/RT1_0518_mask.png
RT1_0519,RT1,final_test,test,,C0431,G0478,RT1/RT1_0519.png,masks/RT1_0519_mask.png
RT1_0520,RT1,dev,0,,C0432,G0479,RT1/RT1_0520.png,masks/RT1_0520_mask.png
RT1_0521,RT1,final_test,test,,C0433,G0480,RT1/RT1_0521.png,masks/RT1_0521_mask.png
RT1_0522,RT1,dev,3,,C0434,G0481,RT1/RT1_0522.png,masks/RT1_0522_mask.png
RT1_0523,RT1,final_test,test,,C0435,G0482,RT1/RT1_0523.png,masks/RT1_0523_mask.png
RT1_0524,RT1,dev,4,,C0436,G0483,RT1/RT1_0524.png,masks/RT1_0524_mask.png
RT1_0525,RT1,dev,4,,C0437,G0484,RT1/RT1_0525.png,masks/RT1_0525_mask.png
RT1_0526,RT1,dev,0,,C0438,G0485,RT1/RT1_0526.png,masks/RT1_0526_mask.png
RT1_0527,RT1,dev,1,,C0439,G0486,RT1/RT1_0527.png,masks/RT1_0527_mask.png
RT1_0528,RT1,dev,2,,C0440,G0487,RT1/RT1_0528.png,masks/RT1_0528_mask.png
RT1_0529,RT1,dev,3,,C0441,G0488,RT1/RT1_0529.png,masks/RT1_0529_mask.png
RT1_0530,RT1,dev,0,,C0442,G0489,RT1/RT1_0530.png,masks/RT1_0530_mask.png
RT1_0531,RT1,final_test,test,,C0443,G0490,RT1/RT1_0531.png,masks/RT1_0531_mask.png
RT1_0532,RT1,final_test,test,,C0444,G0491,RT1/RT1_0532.png,masks/RT1_0532_mask.png
RT1_0533,RT1,dev,3,,C0445,G0492,RT1/RT1_0533.png,masks/RT1_0533_mask.png
RT1_0534,RT1,dev,1,,C0446,G0493,RT1/RT1_0534.png,masks/RT1_0534_mask.png
RT1_0535,RT1,dev,1,,C0447,G0494,RT1/RT1_0535.png,masks/RT1_0535_mask.png
RT1_0536,RT1,dev,1,,C0448,G0495,RT1/RT1_0536.png,masks/RT1_0536_mask.png
RT1_0537,RT1,dev,4,,C0449,G0496,RT1/RT1_0537.png,masks/RT1_0537_mask.png
RT1_0538,RT1,dev,4,,C0450,G0497,RT1/RT1_0538.png,masks/RT1_0538_mask.png
RT1_0539,RT1,dev,2,,C0451,G0498,RT1/RT1_0539.png,masks/RT1_0539_mask.png
RT1_0540,RT1,final_test,test,,C0452,G0499,RT1/RT1_0540.png,masks/RT1_0540_mask.png
RT1_0541,RT1,final_test,test,,C0453,G0500,RT1/RT1_0541.png,masks/RT1_0541_mask.png
RT1_0542,RT1,dev,3,,C0454,G0501,RT1/RT1_0542.png,masks/RT1_0542_mask.png
RT1_0543,RT1,dev,0,,C0455,G0502,RT1/RT1_0543.png,masks/RT1_0543_mask.png
RT1_0544,RT1,dev,4,,C0456,G0503,RT1/RT1_0544.png,masks/RT1_0544_mask.png
RT1_0545,RT1,dev,-1,near-duplicate of a final_test image,C0237,G0390,RT1/RT1_0545.png,masks/RT1_0545_mask.png
RT1_0546,RT1,dev,2,,C0457,G0504,RT1/RT1_0546.png,masks/RT1_0546_mask.png
RT1_0547,RT1,dev,2,,C0458,G0505,RT1/RT1_0547.png,masks/RT1_0547_mask.png
RT1_0548,RT1,dev,0,,C0238,G0506,RT1/RT1_0548.png,masks/RT1_0548_mask.png
RT1_0549,RT1,dev,1,,C0459,G0507,RT1/RT1_0549.png,masks/RT1_0549_mask.png
RT1_0550,RT1,dev,2,,C0430,G0477,RT1/RT1_0550.png,masks/RT1_0550_mask.png
RT1_0551,RT1,final_test,test,,C0460,G0508,RT1/RT1_0551.png,masks/RT1_0551_mask.png
RT1_0552,RT1,dev,3,,C0461,G0509,RT1/RT1_0552.png,masks/RT1_0552_mask.png
RT1_0553,RT1,dev,4,,C0462,G0510,RT1/RT1_0553.png,masks/RT1_0553_mask.png
RT1_0554,RT1,dev,1,,C0463,G0511,RT1/RT1_0554.png,masks/RT1_0554_mask.png
RT1_0555,RT1,dev,0,,C0464,G0512,RT1/RT1_0555.png,masks/RT1_0555_mask.png
RT1_0556,RT1,dev,2,,C0465,G0513,RT1/RT1_0556.png,masks/RT1_0556_mask.png
RT1_0557,RT1,dev,3,,C0466,G0514,RT1/RT1_0557.png,masks/RT1_0557_mask.png
RT1_0558,RT1,dev,2,,C0467,G0515,RT1/RT1_0558.png,masks/RT1_0558_mask.png
RT1_0559,RT1,final_test,test,,C0468,G0516,RT1/RT1_0559.png,masks/RT1_0559_mask.png
RT1_0560,RT1,dev,4,,C0469,G0517,RT1/RT1_0560.png,masks/RT1_0560_mask.png
RT1_0561,RT1,dev,3,,C0470,G0518,RT1/RT1_0561.png,masks/RT1_0561_mask.png
RT1_0562,RT1,dev,0,,C0471,G0519,RT1/RT1_0562.png,masks/RT1_0562_mask.png
RT1_0563,RT1,dev,4,,C0472,G0520,RT1/RT1_0563.png,masks/RT1_0563_mask.png
RT1_0564,RT1,final_test,test,,C0473,G0521,RT1/RT1_0564.png,masks/RT1_0564_mask.png
RT1_0565,RT1,dev,1,,C0474,G0522,RT1/RT1_0565.png,masks/RT1_0565_mask.png
RT1_0566,RT1,final_test,test,,C0475,G0523,RT1/RT1_0566.png,masks/RT1_0566_mask.png
RT1_0567,RT1,dev,1,,C0476,G0524,RT1/RT1_0567.png,masks/RT1_0567_mask.png
RT1_0568,RT1,final_test,test,,C0477,G0525,RT1/RT1_0568.png,masks/RT1_0568_mask.png
RT1_0569,RT1,dev,0,,C0478,G0526,RT1/RT1_0569.png,masks/RT1_0569_mask.png
RT1_0570,RT1,dev,3,,C0265,G0269,RT1/RT1_0570.png,masks/RT1_0570_mask.png
RT1_0571,RT1,dev,0,,C0479,G0527,RT1/RT1_0571.png,masks/RT1_0571_mask.png
RT1_0572,RT1,dev,1,,C0480,G0528,RT1/RT1_0572.png,masks/RT1_0572_mask.png
RT1_0573,RT1,dev,2,,C0481,G0529,RT1/RT1_0573.png,masks/RT1_0573_mask.png
RT1_0574,RT1,dev,3,,C0482,G0530,RT1/RT1_0574.png,masks/RT1_0574_mask.png
RT1_0575,RT1,dev,4,,C0483,G0531,RT1/RT1_0575.png,masks/RT1_0575_mask.png
RT1_0576,RT1,final_test,test,,C0272,G0532,RT1/RT1_0576.png,masks/RT1_0576_mask.png
RT1_0577,RT1,dev,2,,C0484,G0533,RT1/RT1_0577.png,masks/RT1_0577_mask.png
RT1_0578,RT1,dev,4,,C0485,G0534,RT1/RT1_0578.png,masks/RT1_0578_mask.png
RT1_0579,RT1,dev,3,,C0486,G0535,RT1/RT1_0579.png,masks/RT1_0579_mask.png
RT1_0580,RT1,dev,2,,C0487,G0536,RT1/RT1_0580.png,masks/RT1_0580_mask.png
RT1_0581,RT1,dev,0,,C0488,G0537,RT1/RT1_0581.png,masks/RT1_0581_mask.png
RT1_0582,RT1,dev,1,,C0276,G0538,RT1/RT1_0582.png,masks/RT1_0582_mask.png
RT1_0583,RT1,dev,3,,C0489,G0539,RT1/RT1_0583.png,masks/RT1_0583_mask.png
RT1_0584,RT1,dev,0,,C0490,G0540,RT1/RT1_0584.png,masks/RT1_0584_mask.png
RT1_0585,RT1,dev,4,,C0491,G0541,RT1/RT1_0585.png,masks/RT1_0585_mask.png
RT1_0586,RT1,dev,1,,C0492,G0542,RT1/RT1_0586.png,masks/RT1_0586_mask.png
RT1_0587,RT1,dev,2,,C0493,G0543,RT1/RT1_0587.png,masks/RT1_0587_mask.png
RT1_0588,RT1,dev,3,,C0494,G0544,RT1/RT1_0588.png,masks/RT1_0588_mask.png
RT1_0589,RT1,dev,4,,C0495,G0545,RT1/RT1_0589.png,masks/RT1_0589_mask.png
RT1_0590,RT1,dev,0,,C0496,G0546,RT1/RT1_0590.png,masks/RT1_0590_mask.png
RT1_0591,RT1,final_test,test,,C0497,G0547,RT1/RT1_0591.png,masks/RT1_0591_mask.png
RT1_0592,RT1,dev,1,,C0498,G0548,RT1/RT1_0592.png,masks/RT1_0592_mask.png
RT1_0593,RT1,dev,4,,C0499,G0549,RT1/RT1_0593.png,masks/RT1_0593_mask.png
RT1_0594,RT1,final_test,test,,C0500,G0550,RT1/RT1_0594.png,masks/RT1_0594_mask.png
RT1_0595,RT1,dev,2,,C0501,G0551,RT1/RT1_0595.png,masks/RT1_0595_mask.png
RT1_0596,RT1,final_test,test,,C0502,G0552,RT1/RT1_0596.png,masks/RT1_0596_mask.png
RT1_0597,RT1,dev,0,,C0503,G0553,RT1/RT1_0597.png,masks/RT1_0597_mask.png
RT1_0598,RT1,dev,3,,C0504,G0554,RT1/RT1_0598.png,masks/RT1_0598_mask.png
RT1_0599,RT1,dev,1,,C0505,G0555,RT1/RT1_0599.png,masks/RT1_0599_mask.png
RT1_0600,RT1,dev,4,,C0506,G0556,RT1/RT1_0600.png,masks/RT1_0600_mask.png
RT1_0601,RT1,final_test,test,,C0016,G0557,RT1/RT1_0601.png,masks/RT1_0601_mask.png
RT1_0602,RT1,final_test,test,,C0507,G0558,RT1/RT1_0602.png,masks/RT1_0602_mask.png
RT1_0603,RT1,dev,1,,C0508,G0559,RT1/RT1_0603.png,masks/RT1_0603_mask.png
RT1_0604,RT1,final_test,test,,C0509,G0560,RT1/RT1_0604.png,masks/RT1_0604_mask.png
RT1_0605,RT1,dev,2,,C0510,G0561,RT1/RT1_0605.png,masks/RT1_0605_mask.png
RT1_0606,RT1,dev,3,,C0511,G0562,RT1/RT1_0606.png,masks/RT1_0606_mask.png
RT1_0607,RT1,dev,0,,C0512,G0563,RT1/RT1_0607.png,masks/RT1_0607_mask.png
RT1_0608,RT1,dev,2,,C0513,G0564,RT1/RT1_0608.png,masks/RT1_0608_mask.png
RT1_0609,RT1,dev,0,,C0037,G0565,RT1/RT1_0609.png,masks/RT1_0609_mask.png
RT1_0610,RT1,dev,4,,C0514,G0566,RT1/RT1_0610.png,masks/RT1_0610_mask.png
RT1_0611,RT1,dev,0,,C0515,G0567,RT1/RT1_0611.png,masks/RT1_0611_mask.png
RT1_0612,RT1,dev,1,,C0516,G0568,RT1/RT1_0612.png,masks/RT1_0612_mask.png
RT1_0613,RT1,dev,2,,C0517,G0569,RT1/RT1_0613.png,masks/RT1_0613_mask.png
RT1_0614,RT1,dev,2,,C0518,G0570,RT1/RT1_0614.png,masks/RT1_0614_mask.png
RT1_0615,RT1,final_test,test,,C0519,G0571,RT1/RT1_0615.png,masks/RT1_0615_mask.png
RT1_0616,RT1,final_test,test,,C0520,G0572,RT1/RT1_0616.png,masks/RT1_0616_mask.png
RT1_0617,RT1,dev,3,,C0521,G0573,RT1/RT1_0617.png,masks/RT1_0617_mask.png
RT1_0618,RT1,dev,4,,C0522,G0574,RT1/RT1_0618.png,masks/RT1_0618_mask.png
RT1_0619,RT1,dev,0,,C0523,G0575,RT1/RT1_0619.png,masks/RT1_0619_mask.png
RT1_0620,RT1,dev,3,,C0524,G0576,RT1/RT1_0620.png,masks/RT1_0620_mask.png
RT2_0621,RT2,final_test,test,,C0525,G0577,RT2/RT2_0621.png,masks/RT2_0621_mask.png
RT2_0622,RT2,dev,1,,C0526,G0578,RT2/RT2_0622.png,masks/RT2_0622_mask.png
RT2_0623,RT2,final_test,test,,C0527,G0579,RT2/RT2_0623.png,masks/RT2_0623_mask.png
RT2_0624,RT2,dev,4,,C0528,G0580,RT2/RT2_0624.png,masks/RT2_0624_mask.png
RT2_0625,RT2,dev,1,,C0529,G0581,RT2/RT2_0625.png,masks/RT2_0625_mask.png
RT2_0626,RT2,dev,0,,C0530,G0582,RT2/RT2_0626.png,masks/RT2_0626_mask.png
RT2_0627,RT2,dev,2,,C0531,G0583,RT2/RT2_0627.png,masks/RT2_0627_mask.png
RT2_0628,RT2,dev,3,,C0532,G0584,RT2/RT2_0628.png,masks/RT2_0628_mask.png
RT2_0629,RT2,dev,4,,C0533,G0585,RT2/RT2_0629.png,masks/RT2_0629_mask.png
RT2_0630,RT2,dev,0,,C0534,G0586,RT2/RT2_0630.png,masks/RT2_0630_mask.png
RT2_0631,RT2,dev,1,,C0535,G0587,RT2/RT2_0631.png,masks/RT2_0631_mask.png
RT2_0632,RT2,dev,2,,C0536,G0588,RT2/RT2_0632.png,masks/RT2_0632_mask.png
RT2_0633,RT2,dev,3,,C0537,G0589,RT2/RT2_0633.png,masks/RT2_0633_mask.png
RT2_0634,RT2,dev,4,,C0538,G0590,RT2/RT2_0634.png,masks/RT2_0634_mask.png
RT2_0635,RT2,final_test,test,,C0539,G0591,RT2/RT2_0635.png,masks/RT2_0635_mask.png
RT2_0636,RT2,dev,2,,C0540,G0592,RT2/RT2_0636.png,masks/RT2_0636_mask.png
RT2_0637,RT2,dev,3,,C0541,G0593,RT2/RT2_0637.png,masks/RT2_0637_mask.png
RT2_0638,RT2,dev,1,,C0542,G0594,RT2/RT2_0638.png,masks/RT2_0638_mask.png
RT2_0639,RT2,final_test,test,,C0543,G0595,RT2/RT2_0639.png,masks/RT2_0639_mask.png
RT2_0640,RT2,dev,0,,C0544,G0596,RT2/RT2_0640.png,masks/RT2_0640_mask.png
RT2_0641,RT2,dev,0,,C0545,G0597,RT2/RT2_0641.png,masks/RT2_0641_mask.png
RT2_0642,RT2,dev,2,,C0546,G0598,RT2/RT2_0642.png,masks/RT2_0642_mask.png
RT2_0643,RT2,dev,4,,C0547,G0599,RT2/RT2_0643.png,masks/RT2_0643_mask.png
RT2_0644,RT2,final_test,test,,C0548,G0600,RT2/RT2_0644.png,masks/RT2_0644_mask.png
RT2_0645,RT2,final_test,test,,C0549,G0601,RT2/RT2_0645.png,masks/RT2_0645_mask.png
RT2_0646,RT2,dev,3,,C0550,G0602,RT2/RT2_0646.png,masks/RT2_0646_mask.png
RT2_0647,RT2,final_test,test,,C0551,G0603,RT2/RT2_0647.png,masks/RT2_0647_mask.png
RT2_0648,RT2,dev,1,,C0552,G0604,RT2/RT2_0648.png,masks/RT2_0648_mask.png
RT2_0649,RT2,final_test,test,,C0553,G0605,RT2/RT2_0649.png,masks/RT2_0649_mask.png
RT2_0650,RT2,dev,4,,C0554,G0606,RT2/RT2_0650.png,masks/RT2_0650_mask.png
RT2_0651,RT2,final_test,test,,C0555,G0607,RT2/RT2_0651.png,masks/RT2_0651_mask.png
RT2_0652,RT2,dev,0,,C0556,G0608,RT2/RT2_0652.png,masks/RT2_0652_mask.png
RT2_0653,RT2,dev,1,,C0557,G0609,RT2/RT2_0653.png,masks/RT2_0653_mask.png
RT2_0654,RT2,final_test,test,,C0558,G0610,RT2/RT2_0654.png,masks/RT2_0654_mask.png
RT2_0655,RT2,dev,4,,C0559,G0611,RT2/RT2_0655.png,masks/RT2_0655_mask.png
RT2_0656,RT2,dev,2,,C0560,G0612,RT2/RT2_0656.png,masks/RT2_0656_mask.png
RT2_0657,RT2,final_test,test,,C0561,G0613,RT2/RT2_0657.png,masks/RT2_0657_mask.png
RT2_0658,RT2,final_test,test,,C0562,G0614,RT2/RT2_0658.png,masks/RT2_0658_mask.png
RT2_0659,RT2,dev,1,,C0563,G0615,RT2/RT2_0659.png,masks/RT2_0659_mask.png
RT2_0660,RT2,dev,0,,C0564,G0616,RT2/RT2_0660.png,masks/RT2_0660_mask.png
RT2_0661,RT2,final_test,test,,C0565,G0617,RT2/RT2_0661.png,masks/RT2_0661_mask.png
RT2_0662,RT2,dev,4,,C0566,G0618,RT2/RT2_0662.png,masks/RT2_0662_mask.png
RT2_0663,RT2,dev,3,,C0567,G0619,RT2/RT2_0663.png,masks/RT2_0663_mask.png
RT2_0664,RT2,dev,3,,C0568,G0620,RT2/RT2_0664.png,masks/RT2_0664_mask.png
RT2_0665,RT2,dev,0,,C0569,G0621,RT2/RT2_0665.png,masks/RT2_0665_mask.png
RT2_0666,RT2,final_test,test,,C0570,G0622,RT2/RT2_0666.png,masks/RT2_0666_mask.png
RT2_0667,RT2,dev,2,,C0571,G0623,RT2/RT2_0667.png,masks/RT2_0667_mask.png
RT2_0668,RT2,dev,0,,C0572,G0624,RT2/RT2_0668.png,masks/RT2_0668_mask.png
RT2_0669,RT2,dev,1,,C0573,G0625,RT2/RT2_0669.png,masks/RT2_0669_mask.png
RT2_0670,RT2,final_test,test,,C0574,G0626,RT2/RT2_0670.png,masks/RT2_0670_mask.png
RT2_0671,RT2,dev,2,,C0575,G0627,RT2/RT2_0671.png,masks/RT2_0671_mask.png
RT2_0672,RT2,dev,3,,C0576,G0628,RT2/RT2_0672.png,masks/RT2_0672_mask.png
RT2_0673,RT2,dev,4,,C0577,G0629,RT2/RT2_0673.png,masks/RT2_0673_mask.png
RT2_0674,RT2,dev,1,,C0578,G0630,RT2/RT2_0674.png,masks/RT2_0674_mask.png
RT2_0675,RT2,dev,0,,C0579,G0631,RT2/RT2_0675.png,masks/RT2_0675_mask.png
RT2_0676,RT2,dev,3,,C0580,G0632,RT2/RT2_0676.png,masks/RT2_0676_mask.png
RT2_0677,RT2,dev,4,,C0581,G0633,RT2/RT2_0677.png,masks/RT2_0677_mask.png
RT2_0678,RT2,dev,2,,C0582,G0634,RT2/RT2_0678.png,masks/RT2_0678_mask.png
RT2_0679,RT2,dev,2,,C0583,G0635,RT2/RT2_0679.png,masks/RT2_0679_mask.png
RT2_0680,RT2,dev,0,,C0584,G0636,RT2/RT2_0680.png,masks/RT2_0680_mask.png
RT2_0681,RT2,final_test,test,,C0585,G0637,RT2/RT2_0681.png,masks/RT2_0681_mask.png
RT2_0682,RT2,dev,1,,C0586,G0638,RT2/RT2_0682.png,masks/RT2_0682_mask.png
RT2_0683,RT2,dev,3,,C0587,G0639,RT2/RT2_0683.png,masks/RT2_0683_mask.png
RT2_0684,RT2,dev,4,,C0588,G0640,RT2/RT2_0684.png,masks/RT2_0684_mask.png
RT2_0685,RT2,dev,1,,C0589,G0641,RT2/RT2_0685.png,masks/RT2_0685_mask.png
RT2_0686,RT2,dev,2,,C0590,G0642,RT2/RT2_0686.png,masks/RT2_0686_mask.png
RT2_0687,RT2,dev,2,,C0591,G0643,RT2/RT2_0687.png,masks/RT2_0687_mask.png
RT2_0688,RT2,dev,3,,C0592,G0644,RT2/RT2_0688.png,masks/RT2_0688_mask.png
RT2_0689,RT2,dev,4,,C0593,G0645,RT2/RT2_0689.png,masks/RT2_0689_mask.png
RT2_0690,RT2,dev,0,,C0594,G0646,RT2/RT2_0690.png,masks/RT2_0690_mask.png
RT2_0691,RT2,dev,4,,C0595,G0647,RT2/RT2_0691.png,masks/RT2_0691_mask.png
RT2_0692,RT2,dev,1,,C0596,G0648,RT2/RT2_0692.png,masks/RT2_0692_mask.png
RT2_0693,RT2,dev,0,,C0597,G0649,RT2/RT2_0693.png,masks/RT2_0693_mask.png
RT2_0694,RT2,dev,4,,C0598,G0650,RT2/RT2_0694.png,masks/RT2_0694_mask.png
RT2_0695,RT2,final_test,test,,C0599,G0651,RT2/RT2_0695.png,masks/RT2_0695_mask.png
RT2_0696,RT2,dev,2,,C0600,G0652,RT2/RT2_0696.png,masks/RT2_0696_mask.png
RT2_0697,RT2,dev,3,,C0601,G0653,RT2/RT2_0697.png,masks/RT2_0697_mask.png
RT2_0698,RT2,final_test,test,,C0602,G0654,RT2/RT2_0698.png,masks/RT2_0698_mask.png
RT2_0699,RT2,dev,1,,C0603,G0655,RT2/RT2_0699.png,masks/RT2_0699_mask.png
RT2_0700,RT2,dev,0,,C0604,G0656,RT2/RT2_0700.png,masks/RT2_0700_mask.png
RT2_0701,RT2,dev,1,,C0605,G0657,RT2/RT2_0701.png,masks/RT2_0701_mask.png
RT2_0702,RT2,dev,-1,"linked to a test image (same case, source records)",C0013,G0658,RT2/RT2_0702.png,masks/RT2_0702_mask.png
RT2_0703,RT2,dev,4,,C0606,G0659,RT2/RT2_0703.png,masks/RT2_0703_mask.png
RT2_0704,RT2,dev,3,,C0607,G0660,RT2/RT2_0704.png,masks/RT2_0704_mask.png
RT2_0705,RT2,dev,0,,C0604,G0656,RT2/RT2_0705.png,masks/RT2_0705_mask.png
RT2_0706,RT2,dev,0,,C0608,G0661,RT2/RT2_0706.png,masks/RT2_0706_mask.png
RT2_0707,RT2,dev,2,,C0609,G0662,RT2/RT2_0707.png,masks/RT2_0707_mask.png
RT2_0708,RT2,dev,1,,C0610,G0663,RT2/RT2_0708.png,masks/RT2_0708_mask.png
RT2_0709,RT2,dev,3,,C0611,G0664,RT2/RT2_0709.png,masks/RT2_0709_mask.png
RT2_0710,RT2,dev,4,,C0612,G0665,RT2/RT2_0710.png,masks/RT2_0710_mask.png
RT2_0711,RT2,dev,2,,C0613,G0666,RT2/RT2_0711.png,masks/RT2_0711_mask.png
RT2_0712,RT2,dev,0,,C0614,G0667,RT2/RT2_0712.png,masks/RT2_0712_mask.png
RT2_0713,RT2,final_test,test,,C0615,G0668,RT2/RT2_0713.png,masks/RT2_0713_mask.png
RT2_0714,RT2,final_test,test,,C0616,G0669,RT2/RT2_0714.png,masks/RT2_0714_mask.png
RT2_0715,RT2,dev,2,,C0617,G0670,RT2/RT2_0715.png,masks/RT2_0715_mask.png
RT2_0716,RT2,final_test,test,,C0618,G0671,RT2/RT2_0716.png,masks/RT2_0716_mask.png
RT2_0717,RT2,dev,4,,C0619,G0672,RT2/RT2_0717.png,masks/RT2_0717_mask.png
RT2_0718,RT2,dev,1,,C0620,G0673,RT2/RT2_0718.png,masks/RT2_0718_mask.png
RT2_0719,RT2,dev,3,,C0621,G0674,RT2/RT2_0719.png,masks/RT2_0719_mask.png
RT2_0720,RT2,dev,4,,C0622,G0675,RT2/RT2_0720.png,masks/RT2_0720_mask.png
RT2_0721,RT2,dev,0,,C0623,G0676,RT2/RT2_0721.png,masks/RT2_0721_mask.png
RT2_0722,RT2,dev,1,,C0624,G0677,RT2/RT2_0722.png,masks/RT2_0722_mask.png
RT2_0723,RT2,final_test,test,,C0625,G0678,RT2/RT2_0723.png,masks/RT2_0723_mask.png
RT2_0724,RT2,dev,2,,C0626,G0679,RT2/RT2_0724.png,masks/RT2_0724_mask.png
RT2_0725,RT2,dev,0,,C0627,G0680,RT2/RT2_0725.png,masks/RT2_0725_mask.png
RT2_0726,RT2,final_test,test,,C0628,G0681,RT2/RT2_0726.png,masks/RT2_0726_mask.png
RT2_0727,RT2,dev,4,,C0629,G0682,RT2/RT2_0727.png,masks/RT2_0727_mask.png
RT2_0728,RT2,dev,3,,C0630,G0683,RT2/RT2_0728.png,masks/RT2_0728_mask.png
RT2_0729,RT2,dev,2,,C0631,G0684,RT2/RT2_0729.png,masks/RT2_0729_mask.png
RT2_0730,RT2,dev,1,,C0632,G0685,RT2/RT2_0730.png,masks/RT2_0730_mask.png
RT2_0731,RT2,final_test,test,,C0633,G0686,RT2/RT2_0731.png,masks/RT2_0731_mask.png
RT2_0732,RT2,dev,3,,C0634,G0687,RT2/RT2_0732.png,masks/RT2_0732_mask.png
RT2_0733,RT2,dev,1,,C0635,G0688,RT2/RT2_0733.png,masks/RT2_0733_mask.png
RT2_0734,RT2,dev,0,,C0636,G0689,RT2/RT2_0734.png,masks/RT2_0734_mask.png
RT2_0735,RT2,final_test,test,,C0637,G0690,RT2/RT2_0735.png,masks/RT2_0735_mask.png
RT2_0736,RT2,dev,1,,C0638,G0691,RT2/RT2_0736.png,masks/RT2_0736_mask.png
RT2_0737,RT2,dev,2,,C0639,G0692,RT2/RT2_0737.png,masks/RT2_0737_mask.png
RT2_0738,RT2,final_test,test,,C0640,G0693,RT2/RT2_0738.png,masks/RT2_0738_mask.png
RT2_0739,RT2,dev,3,,C0641,G0694,RT2/RT2_0739.png,masks/RT2_0739_mask.png
RT2_0740,RT2,dev,4,,C0642,G0695,RT2/RT2_0740.png,masks/RT2_0740_mask.png
RT2_0741,RT2,dev,0,,C0643,G0696,RT2/RT2_0741.png,masks/RT2_0741_mask.png
RT2_0742,RT2,dev,2,,C0644,G0697,RT2/RT2_0742.png,masks/RT2_0742_mask.png
RT2_0743,RT2,dev,3,,C0645,G0698,RT2/RT2_0743.png,masks/RT2_0743_mask.png
RT2_0744,RT2,dev,4,,C0646,G0699,RT2/RT2_0744.png,masks/RT2_0744_mask.png
RT2_0745,RT2,dev,3,,C0647,G0700,RT2/RT2_0745.png,masks/RT2_0745_mask.png
RT2_0746,RT2,dev,0,,C0648,G0701,RT2/RT2_0746.png,masks/RT2_0746_mask.png
RT2_0747,RT2,final_test,test,,C0649,G0702,RT2/RT2_0747.png,masks/RT2_0747_mask.png
RT2_0748,RT2,final_test,test,,C0650,G0703,RT2/RT2_0748.png,masks/RT2_0748_mask.png
RT2_0749,RT2,final_test,test,,C0651,G0704,RT2/RT2_0749.png,masks/RT2_0749_mask.png
RT2_0750,RT2,final_test,test,,C0652,G0705,RT2/RT2_0750.png,masks/RT2_0750_mask.png
RT2_0751,RT2,dev,4,,C0653,G0706,RT2/RT2_0751.png,masks/RT2_0751_mask.png
RT2_0752,RT2,dev,2,,C0654,G0707,RT2/RT2_0752.png,masks/RT2_0752_mask.png
RT2_0753,RT2,final_test,test,,C0655,G0708,RT2/RT2_0753.png,masks/RT2_0753_mask.png
RT2_0754,RT2,final_test,test,,C0656,G0709,RT2/RT2_0754.png,masks/RT2_0754_mask.png
RT2_0755,RT2,final_test,test,,C0657,G0710,RT2/RT2_0755.png,masks/RT2_0755_mask.png
RT2_0756,RT2,final_test,test,,C0658,G0711,RT2/RT2_0756.png,masks/RT2_0756_mask.png
RT2_0757,RT2,dev,1,,C0659,G0712,RT2/RT2_0757.png,masks/RT2_0757_mask.png
RT2_0758,RT2,dev,0,,C0660,G0713,RT2/RT2_0758.png,masks/RT2_0758_mask.png
RT2_0759,RT2,dev,1,,C0661,G0714,RT2/RT2_0759.png,masks/RT2_0759_mask.png
RT2_0760,RT2,dev,2,,C0662,G0715,RT2/RT2_0760.png,masks/RT2_0760_mask.png
RT2_0761,RT2,final_test,test,,C0663,G0716,RT2/RT2_0761.png,masks/RT2_0761_mask.png
RT2_0762,RT2,dev,3,,C0664,G0717,RT2/RT2_0762.png,masks/RT2_0762_mask.png
RT2_0763,RT2,final_test,test,,C0665,G0718,RT2/RT2_0763.png,masks/RT2_0763_mask.png
RT2_0764,RT2,dev,4,,C0666,G0719,RT2/RT2_0764.png,masks/RT2_0764_mask.png
RT2_0765,RT2,dev,0,,C0667,G0720,RT2/RT2_0765.png,masks/RT2_0765_mask.png
RT2_0766,RT2,final_test,test,,C0668,G0721,RT2/RT2_0766.png,masks/RT2_0766_mask.png
RT2_0767,RT2,final_test,test,,C0669,G0722,RT2/RT2_0767.png,masks/RT2_0767_mask.png
RT2_0768,RT2,dev,2,,C0670,G0723,RT2/RT2_0768.png,masks/RT2_0768_mask.png
RT2_0769,RT2,dev,1,,C0671,G0724,RT2/RT2_0769.png,masks/RT2_0769_mask.png
RT2_0770,RT2,dev,3,,C0672,G0725,RT2/RT2_0770.png,masks/RT2_0770_mask.png
RT2_0771,RT2,dev,4,,C0673,G0726,RT2/RT2_0771.png,masks/RT2_0771_mask.png
RT2_0772,RT2,dev,0,,C0674,G0727,RT2/RT2_0772.png,masks/RT2_0772_mask.png
RT2_0773,RT2,dev,3,,C0675,G0728,RT2/RT2_0773.png,masks/RT2_0773_mask.png
RT2_0774,RT2,final_test,test,,C0676,G0729,RT2/RT2_0774.png,masks/RT2_0774_mask.png
RT2_0775,RT2,dev,1,,C0677,G0730,RT2/RT2_0775.png,masks/RT2_0775_mask.png
RT2_0776,RT2,dev,4,,C0678,G0731,RT2/RT2_0776.png,masks/RT2_0776_mask.png
RT2_0777,RT2,dev,2,,C0679,G0732,RT2/RT2_0777.png,masks/RT2_0777_mask.png
RT2_0778,RT2,dev,2,,C0680,G0733,RT2/RT2_0778.png,masks/RT2_0778_mask.png
RT2_0779,RT2,dev,0,,C0681,G0734,RT2/RT2_0779.png,masks/RT2_0779_mask.png
RT2_0780,RT2,dev,4,,C0682,G0735,RT2/RT2_0780.png,masks/RT2_0780_mask.png
RT2_0781,RT2,dev,-1,near-duplicate of a final_test image,C0141,G0196,RT2/RT2_0781.png,masks/RT2_0781_mask.png
RT2_0782,RT2,final_test,test,,C0683,G0736,RT2/RT2_0782.png,masks/RT2_0782_mask.png
RT2_0783,RT2,dev,1,,C0684,G0737,RT2/RT2_0783.png,masks/RT2_0783_mask.png
RT2_0784,RT2,dev,3,,C0685,G0738,RT2/RT2_0784.png,masks/RT2_0784_mask.png
RT2_0785,RT2,dev,1,,C0686,G0739,RT2/RT2_0785.png,masks/RT2_0785_mask.png
RT2_0786,RT2,dev,3,,C0687,G0740,RT2/RT2_0786.png,masks/RT2_0786_mask.png
RT2_0787,RT2,dev,0,,C0688,G0741,RT2/RT2_0787.png,masks/RT2_0787_mask.png
RT2_0788,RT2,final_test,test,,C0689,G0742,RT2/RT2_0788.png,masks/RT2_0788_mask.png
RT2_0789,RT2,dev,2,,C0690,G0743,RT2/RT2_0789.png,masks/RT2_0789_mask.png
RT2_0790,RT2,dev,4,,C0691,G0744,RT2/RT2_0790.png,masks/RT2_0790_mask.png
RT2_0791,RT2,dev,0,,C0692,G0745,RT2/RT2_0791.png,masks/RT2_0791_mask.png
RT2_0792,RT2,dev,1,,C0693,G0746,RT2/RT2_0792.png,masks/RT2_0792_mask.png
RT2_0793,RT2,dev,3,,C0694,G0747,RT2/RT2_0793.png,masks/RT2_0793_mask.png
RT2_0794,RT2,dev,2,,C0695,G0748,RT2/RT2_0794.png,masks/RT2_0794_mask.png
RT2_0795,RT2,dev,4,,C0696,G0749,RT2/RT2_0795.png,masks/RT2_0795_mask.png
RT2_0796,RT2,final_test,test,,C0697,G0750,RT2/RT2_0796.png,masks/RT2_0796_mask.png
RT2_0797,RT2,dev,0,,C0698,G0751,RT2/RT2_0797.png,masks/RT2_0797_mask.png
RT2_0798,RT2,dev,1,,C0516,G0568,RT2/RT2_0798.png,masks/RT2_0798_mask.png
RT2_0799,RT2,dev,1,,C0699,G0752,RT2/RT2_0799.png,masks/RT2_0799_mask.png
RT2_0800,RT2,dev,2,,C0700,G0753,RT2/RT2_0800.png,masks/RT2_0800_mask.png
RT2_0801,RT2,dev,4,,C0701,G0754,RT2/RT2_0801.png,masks/RT2_0801_mask.png
RT2_0802,RT2,dev,3,,C0702,G0755,RT2/RT2_0802.png,masks/RT2_0802_mask.png
RT2_0803,RT2,dev,0,,C0703,G0756,RT2/RT2_0803.png,masks/RT2_0803_mask.png
RT2_0804,RT2,dev,1,,C0704,G0757,RT2/RT2_0804.png,masks/RT2_0804_mask.png
RT2_0805,RT2,dev,3,,C0705,G0758,RT2/RT2_0805.png,masks/RT2_0805_mask.png
RT2_0806,RT2,dev,4,,C0706,G0759,RT2/RT2_0806.png,masks/RT2_0806_mask.png
RT2_0807,RT2,dev,1,,C0707,G0760,RT2/RT2_0807.png,masks/RT2_0807_mask.png
RT2_0808,RT2,dev,2,,C0708,G0761,RT2/RT2_0808.png,masks/RT2_0808_mask.png
RT2_0809,RT2,dev,0,,C0709,G0762,RT2/RT2_0809.png,masks/RT2_0809_mask.png
RT2_0810,RT2,dev,2,,C0710,G0763,RT2/RT2_0810.png,masks/RT2_0810_mask.png
RT2_0811,RT2,dev,3,,C0711,G0764,RT2/RT2_0811.png,masks/RT2_0811_mask.png
RT2_0812,RT2,dev,4,,C0712,G0765,RT2/RT2_0812.png,masks/RT2_0812_mask.png
RT2_0813,RT2,dev,1,,C0713,G0766,RT2/RT2_0813.png,masks/RT2_0813_mask.png
RT2_0814,RT2,dev,2,,C0714,G0767,RT2/RT2_0814.png,masks/RT2_0814_mask.png
RT2_0815,RT2,final_test,test,,C0715,G0768,RT2/RT2_0815.png,masks/RT2_0815_mask.png
RT2_0816,RT2,dev,-1,near-duplicate of a final_test image,C0715,G0768,RT2/RT2_0816.png,masks/RT2_0816_mask.png
RT2_0817,RT2,dev,0,,C0716,G0769,RT2/RT2_0817.png,masks/RT2_0817_mask.png
RT2_0818,RT2,dev,-1,near-duplicate of a final_test image,C0715,G0768,RT2/RT2_0818.png,masks/RT2_0818_mask.png
RT2_0819,RT2,dev,3,,C0717,G0770,RT2/RT2_0819.png,masks/RT2_0819_mask.png
RT2_0820,RT2,final_test,test,,C0718,G0771,RT2/RT2_0820.png,masks/RT2_0820_mask.png
RT2_0821,RT2,dev,4,,C0719,G0772,RT2/RT2_0821.png,masks/RT2_0821_mask.png
RT2_0822,RT2,dev,0,,C0720,G0773,RT2/RT2_0822.png,masks/RT2_0822_mask.png
RT2_0823,RT2,dev,4,,C0721,G0774,RT2/RT2_0823.png,masks/RT2_0823_mask.png
RT2_0824,RT2,dev,1,,C0722,G0775,RT2/RT2_0824.png,masks/RT2_0824_mask.png
RT2_0825,RT2,dev,3,,C0723,G0776,RT2/RT2_0825.png,masks/RT2_0825_mask.png
RT2_0826,RT2,dev,2,,C0724,G0777,RT2/RT2_0826.png,masks/RT2_0826_mask.png
RT2_0827,RT2,dev,0,,C0725,G0778,RT2/RT2_0827.png,masks/RT2_0827_mask.png
RT2_0828,RT2,final_test,test,,C0726,G0779,RT2/RT2_0828.png,masks/RT2_0828_mask.png
RT2_0829,RT2,dev,1,,C0727,G0780,RT2/RT2_0829.png,masks/RT2_0829_mask.png
RT2_0830,RT2,dev,3,,C0728,G0781,RT2/RT2_0830.png,masks/RT2_0830_mask.png
RT2_0831,RT2,dev,2,,C0729,G0782,RT2/RT2_0831.png,masks/RT2_0831_mask.png
RT2_0832,RT2,final_test,test,,C0730,G0783,RT2/RT2_0832.png,masks/RT2_0832_mask.png
RT2_0833,RT2,final_test,test,,C0731,G0784,RT2/RT2_0833.png,masks/RT2_0833_mask.png
RT2_0834,RT2,dev,4,,C0732,G0785,RT2/RT2_0834.png,masks/RT2_0834_mask.png
RT2_0835,RT2,dev,0,,C0733,G0786,RT2/RT2_0835.png,masks/RT2_0835_mask.png
RT2_0836,RT2,dev,1,,C0734,G0787,RT2/RT2_0836.png,masks/RT2_0836_mask.png
RT2_0837,RT2,dev,2,,C0735,G0788,RT2/RT2_0837.png,masks/RT2_0837_mask.png
RT2_0838,RT2,dev,3,,C0736,G0789,RT2/RT2_0838.png,masks/RT2_0838_mask.png
RT2_0839,RT2,dev,0,,C0737,G0790,RT2/RT2_0839.png,masks/RT2_0839_mask.png
RT2_0840,RT2,dev,4,,C0738,G0791,RT2/RT2_0840.png,masks/RT2_0840_mask.png
RT2_0841,RT2,dev,1,,C0739,G0792,RT2/RT2_0841.png,masks/RT2_0841_mask.png
RT2_0842,RT2,dev,4,,C0740,G0793,RT2/RT2_0842.png,masks/RT2_0842_mask.png
RT2_0843,RT2,dev,2,,C0741,G0794,RT2/RT2_0843.png,masks/RT2_0843_mask.png
RT2_0844,RT2,dev,3,,C0742,G0795,RT2/RT2_0844.png,masks/RT2_0844_mask.png
RT2_0845,RT2,dev,0,,C0743,G0796,RT2/RT2_0845.png,masks/RT2_0845_mask.png
RT2_0846,RT2,dev,2,,C0744,G0797,RT2/RT2_0846.png,masks/RT2_0846_mask.png
RT2_0847,RT2,final_test,test,,C0745,G0798,RT2/RT2_0847.png,masks/RT2_0847_mask.png
RT2_0848,RT2,dev,1,,C0746,G0799,RT2/RT2_0848.png,masks/RT2_0848_mask.png
RT2_0849,RT2,dev,3,,C0747,G0800,RT2/RT2_0849.png,masks/RT2_0849_mask.png
RT2_0850,RT2,dev,4,,C0748,G0801,RT2/RT2_0850.png,masks/RT2_0850_mask.png
RT2_0851,RT2,dev,1,,C0749,G0802,RT2/RT2_0851.png,masks/RT2_0851_mask.png
RT2_0852,RT2,dev,0,,C0750,G0803,RT2/RT2_0852.png,masks/RT2_0852_mask.png
RT2_0853,RT2,dev,2,,C0751,G0804,RT2/RT2_0853.png,masks/RT2_0853_mask.png
RT2_0854,RT2,dev,3,,C0752,G0805,RT2/RT2_0854.png,masks/RT2_0854_mask.png
RT2_0855,RT2,dev,4,,C0753,G0806,RT2/RT2_0855.png,masks/RT2_0855_mask.png
RT2_0856,RT2,dev,1,,C0754,G0807,RT2/RT2_0856.png,masks/RT2_0856_mask.png
RT2_0857,RT2,dev,4,,C0753,G0806,RT2/RT2_0857.png,masks/RT2_0857_mask.png
RT2_0858,RT2,dev,2,,C0755,G0808,RT2/RT2_0858.png,masks/RT2_0858_mask.png
RT2_0859,RT2,dev,0,,C0756,G0809,RT2/RT2_0859.png,masks/RT2_0859_mask.png
RT2_0860,RT2,dev,4,,C0757,G0810,RT2/RT2_0860.png,masks/RT2_0860_mask.png
RT2_0861,RT2,dev,1,,C0758,G0811,RT2/RT2_0861.png,masks/RT2_0861_mask.png
RT2_0862,RT2,final_test,test,,C0759,G0812,RT2/RT2_0862.png,masks/RT2_0862_mask.png
RT3_0863,RT3,dev,3,,C0760,G0813,RT3/RT3_0863.png,masks/RT3_0863_mask.png
RT3_0864,RT3,dev,0,,C0761,G0814,RT3/RT3_0864.png,masks/RT3_0864_mask.png
RT3_0865,RT3,dev,2,,C0762,G0815,RT3/RT3_0865.png,masks/RT3_0865_mask.png
RT3_0866,RT3,dev,2,,C0763,G0816,RT3/RT3_0866.png,masks/RT3_0866_mask.png
RT3_0867,RT3,final_test,test,,C0764,G0817,RT3/RT3_0867.png,masks/RT3_0867_mask.png
RT3_0868,RT3,dev,3,,C0765,G0818,RT3/RT3_0868.png,masks/RT3_0868_mask.png
RT3_0869,RT3,dev,4,,C0766,G0819,RT3/RT3_0869.png,masks/RT3_0869_mask.png
RT3_0870,RT3,dev,4,,C0767,G0820,RT3/RT3_0870.png,masks/RT3_0870_mask.png
RT3_0871,RT3,dev,0,,C0768,G0821,RT3/RT3_0871.png,masks/RT3_0871_mask.png
RT3_0872,RT3,dev,1,,C0769,G0822,RT3/RT3_0872.png,masks/RT3_0872_mask.png
RT3_0873,RT3,final_test,test,,C0770,G0823,RT3/RT3_0873.png,masks/RT3_0873_mask.png
RT3_0874,RT3,final_test,test,,C0771,G0824,RT3/RT3_0874.png,masks/RT3_0874_mask.png
RT3_0875,RT3,dev,3,,C0772,G0825,RT3/RT3_0875.png,masks/RT3_0875_mask.png
RT3_0876,RT3,dev,0,,C0773,G0826,RT3/RT3_0876.png,masks/RT3_0876_mask.png
RT3_0877,RT3,dev,1,,C0774,G0827,RT3/RT3_0877.png,masks/RT3_0877_mask.png
RT3_0878,RT3,dev,2,,C0775,G0828,RT3/RT3_0878.png,masks/RT3_0878_mask.png
RT3_0879,RT3,dev,3,,C0776,G0829,RT3/RT3_0879.png,masks/RT3_0879_mask.png
RT3_0880,RT3,dev,4,,C0777,G0830,RT3/RT3_0880.png,masks/RT3_0880_mask.png
RT3_0881,RT3,final_test,test,,C0778,G0831,RT3/RT3_0881.png,masks/RT3_0881_mask.png
RT3_0882,RT3,dev,0,,C0779,G0832,RT3/RT3_0882.png,masks/RT3_0882_mask.png
RT3_0883,RT3,final_test,test,,C0780,G0833,RT3/RT3_0883.png,masks/RT3_0883_mask.png
RT3_0884,RT3,dev,1,,C0781,G0834,RT3/RT3_0884.png,masks/RT3_0884_mask.png
RT3_0885,RT3,dev,2,,C0782,G0835,RT3/RT3_0885.png,masks/RT3_0885_mask.png
RT3_0886,RT3,dev,3,,C0783,G0836,RT3/RT3_0886.png,masks/RT3_0886_mask.png
RT3_0887,RT3,dev,4,,C0784,G0837,RT3/RT3_0887.png,masks/RT3_0887_mask.png
RT3_0888,RT3,final_test,test,,C0785,G0838,RT3/RT3_0888.png,masks/RT3_0888_mask.png
RT3_0889,RT3,dev,2,,C0786,G0839,RT3/RT3_0889.png,masks/RT3_0889_mask.png
RT3_0890,RT3,dev,3,,C0787,G0840,RT3/RT3_0890.png,masks/RT3_0890_mask.png
RT3_0891,RT3,dev,0,,C0788,G0841,RT3/RT3_0891.png,masks/RT3_0891_mask.png
RT3_0892,RT3,dev,2,,C0789,G0842,RT3/RT3_0892.png,masks/RT3_0892_mask.png
RT3_0893,RT3,final_test,test,,C0790,G0843,RT3/RT3_0893.png,masks/RT3_0893_mask.png
RT3_0894,RT3,final_test,test,,C0791,G0844,RT3/RT3_0894.png,masks/RT3_0894_mask.png
RT3_0895,RT3,dev,4,,C0792,G0845,RT3/RT3_0895.png,masks/RT3_0895_mask.png
RT3_0896,RT3,dev,1,,C0793,G0846,RT3/RT3_0896.png,masks/RT3_0896_mask.png
RT3_0897,RT3,final_test,test,,C0794,G0847,RT3/RT3_0897.png,masks/RT3_0897_mask.png
RT3_0898,RT3,dev,0,,C0795,G0848,RT3/RT3_0898.png,masks/RT3_0898_mask.png
RT3_0899,RT3,dev,1,,C0796,G0849,RT3/RT3_0899.png,masks/RT3_0899_mask.png
RT3_0900,RT3,dev,3,,C0797,G0850,RT3/RT3_0900.png,masks/RT3_0900_mask.png
RT3_0901,RT3,dev,0,,C0798,G0851,RT3/RT3_0901.png,masks/RT3_0901_mask.png
RT3_0902,RT3,dev,4,,C0799,G0852,RT3/RT3_0902.png,masks/RT3_0902_mask.png
RT3_0903,RT3,dev,2,,C0800,G0853,RT3/RT3_0903.png,masks/RT3_0903_mask.png
RT3_0904,RT3,dev,3,,C0801,G0854,RT3/RT3_0904.png,masks/RT3_0904_mask.png
RT3_0905,RT3,final_test,test,,C0802,G0855,RT3/RT3_0905.png,masks/RT3_0905_mask.png
RT3_0906,RT3,dev,1,,C0803,G0856,RT3/RT3_0906.png,masks/RT3_0906_mask.png
RT3_0907,RT3,dev,4,,C0804,G0857,RT3/RT3_0907.png,masks/RT3_0907_mask.png
RT3_0908,RT3,dev,1,,C0805,G0858,RT3/RT3_0908.png,masks/RT3_0908_mask.png
RT3_0909,RT3,dev,0,,C0806,G0859,RT3/RT3_0909.png,masks/RT3_0909_mask.png
RT3_0910,RT3,dev,2,,C0807,G0860,RT3/RT3_0910.png,masks/RT3_0910_mask.png
RT3_0911,RT3,dev,3,,C0808,G0861,RT3/RT3_0911.png,masks/RT3_0911_mask.png
RT3_0912,RT3,dev,3,,C0809,G0862,RT3/RT3_0912.png,masks/RT3_0912_mask.png
RT3_0913,RT3,final_test,test,,C0810,G0863,RT3/RT3_0913.png,masks/RT3_0913_mask.png
RT3_0914,RT3,dev,4,,C0811,G0864,RT3/RT3_0914.png,masks/RT3_0914_mask.png
RT3_0915,RT3,final_test,test,,C0812,G0865,RT3/RT3_0915.png,masks/RT3_0915_mask.png
RT3_0916,RT3,dev,4,,C0813,G0866,RT3/RT3_0916.png,masks/RT3_0916_mask.png
RT3_0917,RT3,dev,0,,C0814,G0867,RT3/RT3_0917.png,masks/RT3_0917_mask.png
RT3_0918,RT3,final_test,test,,C0815,G0868,RT3/RT3_0918.png,masks/RT3_0918_mask.png
RT3_0919,RT3,dev,1,,C0816,G0869,RT3/RT3_0919.png,masks/RT3_0919_mask.png
RT3_0920,RT3,final_test,test,,C0817,G0870,RT3/RT3_0920.png,masks/RT3_0920_mask.png
RT3_0921,RT3,dev,2,,C0818,G0871,RT3/RT3_0921.png,masks/RT3_0921_mask.png
RT3_0922,RT3,dev,0,,C0819,G0872,RT3/RT3_0922.png,masks/RT3_0922_mask.png
RT3_0923,RT3,dev,1,,C0820,G0873,RT3/RT3_0923.png,masks/RT3_0923_mask.png
RT3_0924,RT3,dev,3,,C0821,G0874,RT3/RT3_0924.png,masks/RT3_0924_mask.png
RT3_0925,RT3,dev,2,,C0822,G0875,RT3/RT3_0925.png,masks/RT3_0925_mask.png
RT3_0926,RT3,dev,4,,C0823,G0876,RT3/RT3_0926.png,masks/RT3_0926_mask.png
RT3_0927,RT3,dev,0,,C0824,G0877,RT3/RT3_0927.png,masks/RT3_0927_mask.png
RT3_0928,RT3,dev,4,,C0825,G0878,RT3/RT3_0928.png,masks/RT3_0928_mask.png
RT3_0929,RT3,final_test,test,,C0826,G0879,RT3/RT3_0929.png,masks/RT3_0929_mask.png
RT3_0930,RT3,dev,1,,C0827,G0880,RT3/RT3_0930.png,masks/RT3_0930_mask.png
RT3_0931,RT3,dev,1,,C0828,G0881,RT3/RT3_0931.png,masks/RT3_0931_mask.png
RT3_0932,RT3,dev,2,,C0829,G0882,RT3/RT3_0932.png,masks/RT3_0932_mask.png
RT3_0933,RT3,dev,4,,C0830,G0883,RT3/RT3_0933.png,masks/RT3_0933_mask.png
RT3_0934,RT3,final_test,test,,C0831,G0884,RT3/RT3_0934.png,masks/RT3_0934_mask.png
RT3_0935,RT3,dev,3,,C0832,G0885,RT3/RT3_0935.png,masks/RT3_0935_mask.png
RT3_0936,RT3,dev,0,,C0833,G0886,RT3/RT3_0936.png,masks/RT3_0936_mask.png
RT3_0937,RT3,dev,2,,C0834,G0887,RT3/RT3_0937.png,masks/RT3_0937_mask.png
RT3_0938,RT3,dev,0,,C0835,G0888,RT3/RT3_0938.png,masks/RT3_0938_mask.png
RT3_0939,RT3,dev,3,,C0836,G0889,RT3/RT3_0939.png,masks/RT3_0939_mask.png
RT3_0940,RT3,dev,3,,C0837,G0890,RT3/RT3_0940.png,masks/RT3_0940_mask.png
RT3_0941,RT3,final_test,test,,C0838,G0891,RT3/RT3_0941.png,masks/RT3_0941_mask.png
RT3_0942,RT3,dev,2,,C0839,G0892,RT3/RT3_0942.png,masks/RT3_0942_mask.png
RT3_0943,RT3,dev,1,,C0840,G0893,RT3/RT3_0943.png,masks/RT3_0943_mask.png
RT3_0944,RT3,dev,4,,C0841,G0894,RT3/RT3_0944.png,masks/RT3_0944_mask.png
RT3_0945,RT3,dev,0,,C0842,G0895,RT3/RT3_0945.png,masks/RT3_0945_mask.png
RT3_0946,RT3,dev,1,,C0843,G0896,RT3/RT3_0946.png,masks/RT3_0946_mask.png
RT3_0947,RT3,dev,1,,C0844,G0897,RT3/RT3_0947.png,masks/RT3_0947_mask.png
RT3_0948,RT3,dev,0,,C0845,G0898,RT3/RT3_0948.png,masks/RT3_0948_mask.png
RT3_0949,RT3,dev,2,,C0846,G0899,RT3/RT3_0949.png,masks/RT3_0949_mask.png
RT3_0950,RT3,dev,3,,C0847,G0900,RT3/RT3_0950.png,masks/RT3_0950_mask.png
RT3_0951,RT3,dev,3,,C0848,G0901,RT3/RT3_0951.png,masks/RT3_0951_mask.png
RT3_0952,RT3,dev,4,,C0849,G0902,RT3/RT3_0952.png,masks/RT3_0952_mask.png
RT3_0953,RT3,dev,0,,C0850,G0903,RT3/RT3_0953.png,masks/RT3_0953_mask.png
RT3_0954,RT3,dev,4,,C0851,G0904,RT3/RT3_0954.png,masks/RT3_0954_mask.png
RT3_0955,RT3,dev,2,,C0852,G0905,RT3/RT3_0955.png,masks/RT3_0955_mask.png
RT3_0956,RT3,final_test,test,,C0853,G0906,RT3/RT3_0956.png,masks/RT3_0956_mask.png
RT3_0957,RT3,final_test,test,,C0854,G0907,RT3/RT3_0957.png,masks/RT3_0957_mask.png
RT3_0958,RT3,dev,3,,C0855,G0908,RT3/RT3_0958.png,masks/RT3_0958_mask.png
RT3_0959,RT3,dev,4,,C0856,G0909,RT3/RT3_0959.png,masks/RT3_0959_mask.png
RT3_0960,RT3,dev,0,,C0857,G0910,RT3/RT3_0960.png,masks/RT3_0960_mask.png
RT3_0961,RT3,final_test,test,,C0858,G0911,RT3/RT3_0961.png,masks/RT3_0961_mask.png

In [ ]:
os.environ["GR_SPLIT"] = f"{W}/src/folds.csv"; print("split:", os.environ["GR_SPLIT"])

## Source code
The modules of `src/`.

In [ ]:
%%writefile {W}/src/common.py
# -*- coding: utf-8 -*-
"""Shared definitions for the final run: paths, data split, preprocessing / augmentation, model parts, metrics. Two training back-ends share these definitions: * kaggle_train.py (OFFICIAL run, Kaggle GPU): full backbone on the GPU, frozen part under no_grad, ONLINE augmentation; Not used for the reported results."""
import os, io, json, hashlib, random
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter

# ---------------------------------------------------------------- paths
ROOT = os.environ.get("GR_ROOT", os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))
PKG = os.environ.get("GR_PKG", ROOT)
IMG_ROOT = os.environ.get("GR_IMG_ROOT", os.path.join(ROOT, "images"))
SPLIT_CSV = os.environ.get("GR_SPLIT", os.path.join(ROOT, "data", "folds.csv"))
WORK = os.environ.get("GR_WORK", "./work")      # caches + checkpoints (not in the package)
RESULTS = os.path.join(PKG, "results")
FIGS = os.path.join(PKG, "figures")
for d in (WORK, RESULTS, FIGS): os.makedirs(d, exist_ok=True)

CLASSES = ["HEALTHY", "RT1", "RT2", "RT3"]
NUM = 4
IMG = 224
K_AUG = 8                       # offline augmented views per development image (per configuration)
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
# which cached backbone(s) each model uses

HP = dict(batch=32, stage1_lr=1e-3, stage1_max_epochs=15, stage2_head_lr=1e-4, stage2_backbone_lr=5e-5,
          llrd=0.75, stage2_max_epochs=15, patience=5, weight_decay=0.05, dropout=0.3, head_hidden=256,
          fusion_dim=256, fusion_heads=4, fusion_layers=1, class_weights="inverse frequency (training folds)",
          roi_crop=0.68, ctx_prob=0.6, rot_deg=10, bright=0.2, contrast=0.2, k_aug=K_AUG,
          bootstrap=2000, bootstrap_seed=2026)


# ---------------------------------------------------------------- data split
def load_split():
    import pandas as pd
    d = pd.read_csv(SPLIT_CSV)
    d["path"] = [os.path.join(IMG_ROOT, p.replace("\\", "/")) for p in d["image"]]
    d["y"] = [CLASSES.index(c) for c in d["cls"]]
    dev = d[d["fold"].isin(["0", "1", "2", "3", "4"])].reset_index(drop=True)
    dev["fold"] = dev["fold"].astype(int)
    test = d[d["fold"] == "test"].reset_index(drop=True)
    return d, dev, test


def load_rgb(path):
    return np.asarray(Image.open(path).convert("RGB"))


# ---------------------------------------------------------------- preprocessing
def base_view(rgb, config):
    """Deterministic preprocessing used for evaluation: resize to 224 (+ central ROI crop 68% for 'robust')."""
    x = np.asarray(Image.fromarray(rgb).resize((IMG, IMG), Image.BILINEAR))
    if config == "robust":
        c = int(IMG * HP["roi_crop"]); t = (IMG - c) // 2
        x = np.asarray(Image.fromarray(x[t:t + c, t:t + c]).resize((IMG, IMG), Image.BILINEAR))
    return x


# 16 exposed-root candidate, 32 lip/mucosa. Clinical region R = teeth | gingiva | margin | interdental | root.
MASK_ROOT = os.environ.get("GR_MASK_ROOT", os.path.join(ROOT, "data"))


def load_mask(row):
    p = os.path.join(MASK_ROOT, str(row.mask).replace("\\", "/"))
    if not os.path.exists(p): p = os.path.join(MASK_ROOT, "masks", os.path.basename(p))
    return np.asarray(Image.open(p))


def _rng(*key):
    h = hashlib.sha256("|".join(map(str, key)).encode()).hexdigest()
    return np.random.default_rng(int(h[:16], 16)), random.Random(int(h[16:32], 16))


def std_augment(x, r):
    """Flip, small rotation, brightness/contrast. No hue/saturation change (gingival colour is clinical)."""
    pil = Image.fromarray(x)
    if r.random() < 0.5: pil = pil.transpose(Image.FLIP_LEFT_RIGHT)
    ang = r.uniform(-HP["rot_deg"], HP["rot_deg"])
    import cv2
    a = np.asarray(pil)
    M = cv2.getRotationMatrix2D((IMG / 2, IMG / 2), ang, 1.0)
    a = cv2.warpAffine(a, M, (IMG, IMG), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101).astype(np.float32)
    a = a * r.uniform(1 - HP["bright"], 1 + HP["bright"])
    m = a.mean(); a = (a - m) * r.uniform(1 - HP["contrast"], 1 + HP["contrast"]) + m
    return np.clip(a, 0, 255).astype(np.uint8)




def normalize(batch_uint8):
    """uint8 NHWC -> float32 NCHW, ImageNet mean/std."""
    x = batch_uint8.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return np.ascontiguousarray(x.transpose(0, 3, 1, 2))


# ---------------------------------------------------------------- model parts


import torch
import torch.nn as nn


# ---------------------------------------------------------------- cache access


# ---------------------------------------------------------------- metrics
def per_class_counts(cm):
    out = {}
    tot = cm.sum()
    for i, c in enumerate(CLASSES):
        tp = cm[i, i]; fn = cm[i].sum() - tp; fp = cm[:, i].sum() - tp; tn = tot - tp - fn - fp
        out[c] = dict(tp=int(tp), fn=int(fn), fp=int(fp), tn=int(tn))
    return out


def metrics(y, P):
    """All A5 metrics from labels y (n,) and probabilities P (n, 4)."""
    from sklearn.metrics import (balanced_accuracy_score, f1_score, cohen_kappa_score, roc_auc_score,
                                 confusion_matrix, log_loss)
    y = np.asarray(y); P = np.asarray(P, dtype=np.float64); pred = P.argmax(1)
    cm = confusion_matrix(y, pred, labels=range(NUM))
    pc = {}
    for c, k in per_class_counts(cm).items():
        i = CLASSES.index(c); tp, fn, fp, tn = k["tp"], k["fn"], k["fp"], k["tn"]
        sens = tp / max(1, tp + fn); spec = tn / max(1, tn + fp); ppv = tp / max(1, tp + fp); npv = tn / max(1, tn + fn)
        f1 = 2 * ppv * sens / max(1e-12, ppv + sens)
        try: auc = float(roc_auc_score((y == i).astype(int), P[:, i]))
        except ValueError: auc = float("nan")
        pc[c] = dict(n=int(tp + fn), per_class_accuracy=(tp + tn) / max(1, tp + tn + fp + fn), sensitivity=sens,
                     specificity=spec, ppv=ppv, npv=npv, f1=f1, auc=auc, **k)
    try: macro_auc = float(roc_auc_score(np.eye(NUM)[y], P, average="macro"))
    except ValueError: macro_auc = float("nan")
    return dict(n=int(len(y)), accuracy=float((pred == y).mean()),
                balanced_accuracy=float(balanced_accuracy_score(y, pred)),
                macro_f1=float(f1_score(y, pred, average="macro", labels=range(NUM), zero_division=0)),
                weighted_f1=float(f1_score(y, pred, average="weighted", labels=range(NUM), zero_division=0)),
                macro_precision=float(np.mean([pc[c]["ppv"] for c in CLASSES])),
                macro_recall=float(np.mean([pc[c]["sensitivity"] for c in CLASSES])),
                kappa=float(cohen_kappa_score(y, pred)),
                kappa_quadratic=float(cohen_kappa_score(y, pred, weights="quadratic")),
                macro_auc=macro_auc, log_loss=float(log_loss(y, np.clip(P, 1e-7, 1) / np.clip(P, 1e-7, 1).sum(1, keepdims=True), labels=range(NUM))),
                per_class=pc, confusion_matrix=cm.tolist())


def bootstrap_ci(y, P, fns, n=None, seed=None):
    """Percentile 95% CI by resampling test images. fns: dict name -> f(y, P) -> float."""
    n = n or HP["bootstrap"]; rng = np.random.default_rng(HP["bootstrap_seed"] if seed is None else seed)
    y = np.asarray(y); P = np.asarray(P); vals = {k: [] for k in fns}
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        for k, f in fns.items():
            try: vals[k].append(f(y[idx], P[idx]))
            except ValueError: pass
    return {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] for k, v in vals.items()}


def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f: json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path):
    with open(path, encoding="utf-8") as f: return json.load(f)

In [ ]:
%%writefile {W}/src/data_io.py
# -*- coding: utf-8 -*-
"""Expects `common` to be importable and its paths patched (done by the notebook)."""
import os, time, math, json, hashlib, shutil
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import common as C


def load_all_rgbs(df):
    return {r.image_id: C.load_rgb(r.path) for r in df.itertuples()}

In [ ]:
%%writefile {W}/src/fusion.py
# -*- coding: utf-8 -*-
"""Data loading and augmentation are reused from common.py / kaggle_train.py (unchanged)."""
import os, time, math, json, hashlib, shutil, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
import common as C
from data_io import load_all_rgbs

DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = os.environ.get("GR_FUSION_OUT", os.path.join(C.WORK, "fusion"))
RUNS = os.path.join(OUT, "runs"); os.makedirs(RUNS, exist_ok=True)
PRETRAINED = os.environ.get("GR_PRETRAINED", "1") == "1"
D, HEADS, DROP = 256, 4, 0.3

CNNS = {"mobilenetv2": "mobilenetv2_100.ra_in1k", "efficientnetb0": "efficientnet_b0.ra_in1k",
        "resnet50": "resnet50.a1_in1k", "convnext_tiny": "convnext_tiny.fb_in1k"}
VITS = {"dinov2_small": "vit_small_patch14_dinov2.lvd142m", "vit_small": "vit_small_patch16_224.augreg_in21k_ft_in1k",
        "deit_small": "deit_small_patch16_224.fb_in1k", "swin_tiny": "swin_tiny_patch4_window7_224.ms_in1k"}
TAILS = {"mobilenetv2": [["blocks.6"], ["conv_head", "bn2"]], "efficientnetb0": [["blocks.6"], ["conv_head", "bn2"]],
         "resnet50": [["layer4.0"], ["layer4.1"], ["layer4.2"]],
         "convnext_tiny": [["stages.3.downsample", "stages.3.blocks.0"], ["stages.3.blocks.1"], ["stages.3.blocks.2"]],
         "dinov2_small": [["blocks.10"], ["blocks.11", "norm"]], "vit_small": [["blocks.10"], ["blocks.11", "norm"]],
         "deit_small": [["blocks.10"], ["blocks.11", "norm"]],
         "swin_tiny": [["layers.3.downsample", "layers.3.blocks.0"], ["layers.3.blocks.1", "norm"]]}


# ------------------------------------------------------------------ model parts
class Branch(nn.Module):
    """A timm backbone returning a token sequence (B, N, C)."""

    def __init__(self, key):
        super().__init__()
        import timm
        self.key = key; tid = {**CNNS, **VITS}[key]
        self.m = timm.create_model(tid, pretrained=PRETRAINED, num_classes=0, **({"img_size": 224} if "dinov2" in key else {}))
        self.kind = "cnn" if key in CNNS else ("swin" if "swin" in key else "vit")
        self.dim = self.m.num_features
        for p in self.m.parameters(): p.requires_grad = False

    def tokens(self, x):
        f = self.m.forward_features(x)
        if self.kind == "cnn": return f.flatten(2).transpose(1, 2)            # (B, 49, C)
        if self.kind == "swin": return f.reshape(f.shape[0], -1, f.shape[-1])  # (B, 49, C)
        return f[:, self.m.num_prefix_tokens:]                                  # (B, N, C) patch tokens

    def tail_groups(self):
        named = dict(self.m.named_parameters()); groups = []
        for prefixes in TAILS[self.key]:
            groups.append([p for n, p in named.items() if any(n == q or n.startswith(q + ".") for q in prefixes)])
        assert all(groups), f"empty tail group in {self.key}"
        return groups

    def bn_eval(self):
        for mod in self.m.modules():
            if isinstance(mod, nn.modules.batchnorm._BatchNorm): mod.eval()


class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.q = nn.Parameter(torch.zeros(1, 1, d)); nn.init.trunc_normal_(self.q, std=0.02)
        self.att = nn.MultiheadAttention(d, HEADS, dropout=0.1, batch_first=True); self.ln = nn.LayerNorm(d)

    def forward(self, x):
        o, _ = self.att(self.q.expand(x.shape[0], -1, -1), x, x); return self.ln(o[:, 0])


class Proj(nn.Module):
    def __init__(self, c, n_pos=None):
        super().__init__()
        self.lin = nn.Sequential(nn.Linear(c, D), nn.LayerNorm(D))
        self.pos = None
        if n_pos: self.pos = nn.Parameter(torch.zeros(1, n_pos, D)); nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, t):
        t = self.lin(t); return t + self.pos if self.pos is not None and self.pos.shape[1] == t.shape[1] else t


# ------------------------------------------------------------------ one run

In [ ]:
%%writefile {W}/src/training.py
# -*- coding: utf-8 -*-
"""data loading and the old 'robust' pipeline from common.py / kaggle_train.py. Masks are not used."""
import os, io, time, math, json, hashlib, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from scipy.ndimage import gaussian_filter
import common as C
from data_io import load_all_rgbs
import fusion as V

DEV = V.DEV
OUT = os.environ.get("GR_OUT", os.path.join(C.WORK, "output"))
RUNS = os.path.join(OUT, "runs"); os.makedirs(RUNS, exist_ok=True)
QUICK = int(os.environ.get("GR_QUICK", "0"))
DEADLINE = float(os.environ.get("GR_DEADLINE_H", "11.3")) * 3600
T0 = time.time()
SEEDS = [0, 1, 2, 3, 4]
IMG, FILL = C.IMG, (124, 116, 104)
INNER_FRAC, ORD_W = 0.10, 0.1

DEEP = {"dinov2_small": [["blocks.8"], ["blocks.9"], ["blocks.10"], ["blocks.11", "norm"]],
        "convnext_tiny": [["stages.2.blocks.6"], ["stages.2.blocks.7"], ["stages.2.blocks.8"],
                          ["stages.3.downsample", "stages.3.blocks.0"], ["stages.3.blocks.1"], ["stages.3.blocks.2"]]}
INIT = {}


def _tail_groups(self):
    named = dict(self.m.named_parameters()); groups = []
    for prefixes in getattr(self, "tails", V.TAILS[self.key]):
        groups.append([p for n, p in named.items() if any(n == q or n.startswith(q + ".") for q in prefixes)])
    assert all(groups), f"empty tail group in {self.key}"
    return groups


V.Branch.tail_groups = _tail_groups


def branch(key):
    b = V.Branch(key); b.tails = DEEP[key]; return b


# ------------------------------------------------------------------ preprocessing ('full')
def _place(rgb, scale, r=None):
    h, w = rgb.shape[:2]; s = IMG * scale / max(h, w)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    im = Image.fromarray(rgb).resize((nw, nh), Image.BICUBIC)
    x0, y0 = ((IMG - nw) // 2, (IMG - nh) // 2) if r is None else (r.randint(0, IMG - nw), r.randint(0, IMG - nh))
    canvas = Image.new("RGB", (IMG, IMG), FILL); canvas.paste(im, (x0, y0))
    return np.asarray(canvas)


def full_view(rgb): return _place(rgb, 1.0)


def full_aug(rgb, r):
    x = _place(rgb, r.uniform(0.85, 1.0), r)
    if r.random() < 0.5:
        b = io.BytesIO(); Image.fromarray(x).save(b, "JPEG", quality=r.randint(60, 95)); x = np.asarray(Image.open(b).convert("RGB"))
    if r.random() < 0.3:
        s = r.uniform(0.3, 0.8); x = np.clip(gaussian_filter(x.astype(np.float32), sigma=(s, s, 0)), 0, 255).astype(np.uint8)
    return C.std_augment(x, r)


class FullSet(Dataset):
    def __init__(self, df, rgbs, train, seed):
        self.df, self.rgbs, self.train, self.seed, self.epoch = df.reset_index(drop=True), rgbs, train, seed, 0

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]; rgb = self.rgbs[r.image_id]
        x = full_aug(rgb, C._rng("full_aug", r.image_id, self.seed, self.epoch)[1]) if self.train else full_view(rgb)
        return torch.from_numpy(C.normalize(x[None])[0]), int(r.y)


def make_set(model, df, rgbs, train, seed):
    """Dataset constructor; set by experiments.py."""
    raise NotImplementedError(model)


# ------------------------------------------------------------------ models
def dino_feat(b, x):
    f = b.m.forward_features(x); k = b.m.num_prefix_tokens
    return torch.cat([f[:, 0], f[:, k:].mean(1)], 1), f[:, k:]      # (B, 768) global, (B, 256, 384) patches


def linear_head(d):
    return nn.Sequential(nn.LayerNorm(d), nn.Dropout(V.DROP), nn.Linear(d, C.NUM))


class SingleDino(nn.Module):
    def __init__(self):
        super().__init__(); self.bv = branch("dinov2_small"); self.head = linear_head(2 * self.bv.dim)

    def forward(self, x): return self.head(dino_feat(self.bv, x)[0])

    def branches(self): return [self.bv]


class Residual(nn.Module):
    """logits = z_DINO (the SingleDino head) + dz_partner; dz starts at exactly zero."""

    def __init__(self, partner):
        super().__init__()
        self.bv = branch("dinov2_small"); self.head = linear_head(2 * self.bv.dim)
        self.bp = branch(partner)
        self.pp = V.Proj(self.bp.dim, 49 if self.bp.kind == "cnn" else None); self.pv = V.Proj(self.bv.dim)
        self.att = nn.MultiheadAttention(V.D, V.HEADS, dropout=0.1, batch_first=True)
        self.ln1, self.ln2 = nn.LayerNorm(V.D), nn.LayerNorm(V.D)
        self.ff = nn.Sequential(nn.Linear(V.D, 2 * V.D), nn.GELU(), nn.Dropout(0.1), nn.Linear(2 * V.D, V.D))
        self.pool = V.AttnPool(V.D); self.drop = nn.Dropout(V.DROP)
        self.corr = nn.Linear(V.D, C.NUM); nn.init.zeros_(self.corr.weight); nn.init.zeros_(self.corr.bias)

    def forward(self, x, parts=False):
        g, pv = dino_feat(self.bv, x); z = self.head(g)
        p = self.pp(self.bp.tokens(x)); v = self.pv(pv)
        a, _ = self.att(p, v, v); p = self.ln1(p + a); p = self.ln2(p + self.ff(p))
        dz = self.corr(self.drop(self.pool(p)))
        return (z, dz) if parts else z + dz

    def branches(self): return [self.bv, self.bp]


def build(model):
    """Model constructor; set by experiments.py."""
    raise NotImplementedError(model)


def head_params(model):
    bb = {id(p) for br in model.branches() for p in br.parameters()}
    return [p for p in model.parameters() if id(p) not in bb]


def save_keys(model):
    keys = {n for n, p in model.named_parameters() if id(p) in {id(q) for q in head_params(model)}}
    for name, mod in model.named_children():
        if isinstance(mod, V.Branch):
            for prefixes in mod.tails:
                for n, _ in mod.m.named_parameters():
                    if any(n == q or n.startswith(q + ".") for q in prefixes): keys.add(f"{name}.m.{n}")
    return keys


# ------------------------------------------------------------------ inner split (training images only)
def inner_split(tr_df, seed):
    rng = np.random.default_rng(10_000 + seed); inner = []
    for cls, d in tr_df.groupby("cls"):
        groups = d.similarity_group.unique(); rng.shuffle(groups); target = max(1, round(INNER_FRAC * len(d))); n = 0
        for g in groups:
            if n >= target: break
            ids = d[d.similarity_group == g].image_id.tolist(); inner += ids; n += len(ids)
    inner = set(inner); g_in = set(tr_df[tr_df.image_id.isin(inner)].similarity_group)
    fit = tr_df[~tr_df.similarity_group.isin(g_in)]                         # a similarity group never sits on both sides
    return fit, tr_df[tr_df.image_id.isin(inner)]


# ------------------------------------------------------------------ one run
def train_one(model_name, seed, dev, rgbs):
    out = os.path.join(RUNS, f"{model_name}_s{seed}")
    if os.path.exists(os.path.join(out, "val_metrics.json")): return "exists"
    os.makedirs(out, exist_ok=True)
    off = INIT.get(model_name, 0)
    torch.manual_seed(seed + off); np.random.seed(seed + off)
    tr_df, va_df = dev[dev.fold != seed], dev[dev.fold == seed]
    if QUICK: tr_df, va_df = tr_df.groupby("cls").head(4 * QUICK), va_df.groupby("cls").head(QUICK)
    fit_df, in_df = inner_split(tr_df, seed)
    tr_set = make_set(model_name, fit_df, rgbs, True, seed + off)
    nw = int(os.environ.get("GR_WORKERS", 2 if DEV == "cuda" else 0))
    in_loader = DataLoader(make_set(model_name, in_df, rgbs, False, seed), batch_size=64, shuffle=False, num_workers=nw)
    va_loader = DataLoader(make_set(model_name, va_df, rgbs, False, seed), batch_size=64, shuffle=False, num_workers=nw)
    model = build(model_name).to(DEV); keys = save_keys(model); hp = head_params(model)
    cnt = np.bincount(fit_df.y.to_numpy(), minlength=C.NUM)
    w = torch.tensor(len(fit_df) / (C.NUM * np.maximum(cnt, 1)), dtype=torch.float32, device=DEV)
    dist = torch.abs(torch.arange(C.NUM, device=DEV)[None, :].float() - torch.arange(C.NUM, device=DEV)[:, None].float())
    scaler = torch.amp.GradScaler(enabled=DEV == "cuda")

    def loss_fn(logits, y):
        loss = F.cross_entropy(logits.float(), y, weight=w)
        if model.training and hasattr(model, "penalty"): loss = loss + model.penalty()
        return loss

    @torch.no_grad()
    def predict(loader):
        model.eval(); P, Y = [], []
        for x, y in loader:
            with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
                P.append(F.softmax(model(x.to(DEV)).float(), 1).cpu())
            Y.append(y)
        P = torch.cat(P).numpy(); Y = torch.cat(Y).numpy()
        return P, Y, float(F.nll_loss(torch.log(torch.from_numpy(P).clamp_min(1e-7)), torch.from_numpy(Y)))

    def snapshot(): return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keys}

    hist, best, ep_g, t0 = [], dict(loss=math.inf, state=None, epoch=None, stage=None), 0, time.time()
    print(f"START {model_name} s{seed}: fit {len(fit_df)}, inner {len(in_df)}, outer {len(va_df)}", flush=True)
    for stage in (1, 2):
        for br in model.branches():
            for g in br.tail_groups():
                for p in g: p.requires_grad = (stage == 2)
        if stage == 1:
            opt = torch.optim.AdamW(hp, lr=C.HP["stage1_lr"], weight_decay=C.HP["weight_decay"]); max_ep = C.HP["stage1_max_epochs"]
        else:
            model.load_state_dict(best["state"], strict=False)
            groups = [{"params": hp, "lr": C.HP["stage2_head_lr"]}]
            for br in model.branches():
                tg = br.tail_groups(); n = len(tg)
                for i, g in enumerate(tg): groups.append({"params": g, "lr": C.HP["stage2_backbone_lr"] * C.HP["llrd"] ** (n - 1 - i)})
            opt = torch.optim.AdamW(groups, weight_decay=C.HP["weight_decay"]); max_ep = C.HP["stage2_max_epochs"]
        if QUICK: max_ep = 1
        bad, best_stage = 0, math.inf
        for e in range(max_ep):
            tr_set.epoch = ep_g
            gen = torch.Generator(); gen.manual_seed(1000 * seed + ep_g + off)
            loader = DataLoader(tr_set, batch_size=C.HP["batch"], shuffle=True, num_workers=nw, generator=gen)
            model.train()
            for br in model.branches(): br.bn_eval()
            tl, tc, tn = 0.0, 0, 0
            for x, y in loader:
                x, y = x.to(DEV), y.to(DEV)
                with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
                    logits = model(x)
                loss = loss_fn(logits, y)
                if not torch.isfinite(loss): raise RuntimeError("non-finite loss")
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tl += float(loss) * len(y); tc += int((logits.argmax(1) == y).sum()); tn += len(y)
            P, Y, il = predict(in_loader); m = C.metrics(Y, P); ep_g += 1                  # INNER split only
            hist.append(dict(epoch=ep_g, stage=stage, train_loss=tl / tn, train_acc=tc / tn, inner_loss=il,
                             inner_acc=m["accuracy"], minutes=(time.time() - t0) / 60))
            if il < best["loss"] - 1e-4: best = dict(loss=il, state=snapshot(), epoch=ep_g, stage=stage)
            if il < best_stage - 1e-4: best_stage, bad = il, 0
            else:
                bad += 1
                if bad >= C.HP["patience"]: break
    pd.DataFrame(hist).to_csv(os.path.join(out, "history.csv"), index=False)
    model.load_state_dict(best["state"], strict=False)
    torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in best["state"].items()}, os.path.join(out, "model.pt"))
    P, Y, _ = predict(va_loader)                                                         # outer fold: evaluation only
    np.savez(os.path.join(out, "val_probs.npz"), P=P, y=Y, image_id=va_df.image_id.to_numpy())
    m = C.metrics(Y, P); m.update(best_epoch=best["epoch"], best_stage=best["stage"], epochs_run=ep_g, val_fold=seed,
                                  n_fit=int(len(fit_df)), n_inner=int(len(in_df)), minutes=round((time.time() - t0) / 60, 2))
    if isinstance(model, Residual):                                                      # size of the learned correction
        model.eval(); X = next(iter(va_loader))[0][:64].to(DEV)
        with torch.no_grad(): z, dz = model(X, parts=True)
        m["correction_rms_over_logit_rms"] = float(dz.float().pow(2).mean().sqrt() / z.float().pow(2).mean().sqrt().clamp_min(1e-8))
    C.save_json(m, os.path.join(out, "val_metrics.json"))
    print(f"DONE {model_name} s{seed}: outer val acc {m['accuracy']:.4f} (best epoch {best['epoch']}, {m['minutes']} min, "
          f"total {(time.time() - T0) / 3600:.2f} h)", flush=True)
    del model; torch.cuda.empty_cache() if DEV == "cuda" else None
    return "done"


def out_of_time(margin_h=0.4): return time.time() - T0 > DEADLINE - margin_h * 3600


def run_models(models, dev, rgbs, seeds):
    for rm in models:
        for s in seeds:
            if out_of_time(): print("TIME LIMIT: stopping before", rm, s, flush=True); return False
            try: train_one(rm, s, dev, rgbs)
            except RuntimeError as ex:
                print("FAILED", rm, s, ex); C.save_json({"failed": str(ex)}, os.path.join(RUNS, f"{rm}_s{s}", "failed.json"))
    return True


# ------------------------------------------------------------------ selection (validation only)

In [ ]:
%%writefile {W}/src/hybrid.py
# -*- coding: utf-8 -*-
"""Set GR_OUT to the output folder BEFORE importing this module."""
import os, time, json, hashlib, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import training as T


C, V = T.C, T.V
XKEY = "convnext_xxl_clip"; V.CNNS[XKEY] = os.environ.get("GR_XXL_ARCH", "convnext_xxlarge.clip_laion2b_soup")
V.TAILS[XKEY] = []
XRES, BETA = 256, 0.3
IMNET = (np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32))
CLIP = (np.array([0.48145466, 0.4578275, 0.40821073], np.float32), np.array([0.26862954, 0.26130258, 0.27577711], np.float32))


# ------------------------------------------------------------------ inputs
class ToClip(nn.Module):

    def __init__(self):
        super().__init__()
        a = IMNET[1] / CLIP[1]; b = (IMNET[0] - CLIP[0]) / CLIP[1]
        self.register_buffer("a", torch.tensor(a).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("b", torch.tensor(b).view(1, 3, 1, 1), persistent=False)

    def forward(self, x): return F.interpolate(x * self.a + self.b, size=(XRES, XRES), mode="bilinear", align_corners=False)


# ------------------------------------------------------------------ models
def xxl_branch(tails):
    b = V.Branch(XKEY); n2 = len(b.m.stages[2].blocks)
    b.tails = ([[f"stages.2.blocks.{n2 - 3}"], [f"stages.2.blocks.{n2 - 2}"], [f"stages.2.blocks.{n2 - 1}"],
                ["stages.3.downsample", "stages.3.blocks.0"], ["stages.3.blocks.1"], ["stages.3.blocks.2", "head.norm"]] if tails else [])
    return b


class SingleXXL(nn.Module):
    """XXL-CLIP alone on the zoom view; head on its own pooled output (the 'map' feature of the frozen study)."""

    def __init__(self, tune):
        super().__init__(); self.bc = xxl_branch(tune); self.clip = ToClip(); self.head = T.linear_head(self.bc.dim)

    def forward(self, x): return self.head(self.bc.m.forward_head(self.bc.m.forward_features(self.clip(x)), pre_logits=True))

    def branches(self): return [self.bc]


class XAttnHybrid(T.Residual):
    """the layers are its own."""

    def __init__(self, partner):
        nn.Module.__init__(self)
        self.bv = T.branch("dinov2_small"); self.head = T.linear_head(2 * self.bv.dim)
        if partner == "xxl":
            self.bp = xxl_branch(False); self.prep = ToClip(); n_tok = (XRES // 32) ** 2
        else:
            self.bp = V.Branch("dinov2_small"); self.bp.tails = []; self.prep = nn.Identity(); n_tok = 256
        self.pp, self.pv = V.Proj(self.bp.dim, n_tok), V.Proj(self.bv.dim, 256)
        mha = lambda: nn.MultiheadAttention(V.D, V.HEADS, dropout=0.1, batch_first=True)
        ff = lambda: nn.Sequential(nn.Linear(V.D, 2 * V.D), nn.GELU(), nn.Dropout(0.1), nn.Linear(2 * V.D, V.D))
        self.att_p, self.att_v = mha(), mha()
        self.ln_p1, self.ln_p2, self.ln_v1, self.ln_v2 = (nn.LayerNorm(V.D) for _ in range(4))
        self.ff_p, self.ff_v = ff(), ff()
        self.pool_p, self.pool_v = V.AttnPool(V.D), V.AttnPool(V.D); self.drop = nn.Dropout(V.DROP)
        self.corr = nn.Linear(2 * V.D, C.NUM); nn.init.zeros_(self.corr.weight); nn.init.zeros_(self.corr.bias)
        self._pen = None

    def forward(self, x, parts=False):
        xv, xp = x[:, :3], x[:, 3:]
        g, tv = T.dino_feat(self.bv, xv); z = self.head(g)
        p, v = self.pp(self.bp.tokens(self.prep(xp))), self.pv(tv)
        a, _ = self.att_p(p, v, v); p2 = self.ln_p1(p + a); p2 = self.ln_p2(p2 + self.ff_p(p2))
        b, _ = self.att_v(v, p, p); v2 = self.ln_v1(v + b); v2 = self.ln_v2(v2 + self.ff_v(v2))
        dz = self.corr(self.drop(torch.cat([self.pool_p(p2), self.pool_v(v2)], 1)))
        if self.training: self._pen = BETA * dz.float().pow(2).mean() / z.detach().float().pow(2).mean().clamp_min(1e-6)
        return (z, dz) if parts else z + dz

    def penalty(self): return self._pen if self._pen is not None else torch.zeros((), device=self.corr.weight.device)

    def branches(self): return [self.bv, self.bp]


# ------------------------------------------------------------------ analysis helpers

In [ ]:
%%writefile {W}/src/views.py
# -*- coding: utf-8 -*-
"""Set GR_OUT to the output folder BEFORE importing this module."""
import os
import numpy as np, torch
from PIL import Image
from torch.utils.data import Dataset
import hybrid as HY

T, C = HY.T, HY.C
LO, HI = 0.60, 0.76                                             # kept fraction of the side; evaluation keeps the central 68%


def clean_zoom(rgb, r=None):
    if r is None: return C.base_view(rgb, "robust")                                   # evaluation: unchanged
    x = Image.fromarray(rgb).resize((C.IMG, C.IMG), Image.BILINEAR)
    c = int(C.IMG * r.uniform(LO, HI)); x0, y0 = r.randint(0, C.IMG - c), r.randint(0, C.IMG - c)
    x = np.asarray(x.crop((x0, y0, x0 + c, y0 + c)).resize((C.IMG, C.IMG), Image.BILINEAR))
    return C.std_augment(x, r)                                                         # flip, rotation, brightness, contrast


class CleanZoomSet(T.FullSet):
    def __getitem__(self, i):
        r = self.df.iloc[i]; rgb = self.rgbs[r.image_id]
        x = clean_zoom(rgb, C._rng("clean_aug", r.image_id, self.seed, self.epoch)[1] if self.train else None)
        return torch.from_numpy(C.normalize(x[None])[0]), int(r.y)


class CleanDualSet(Dataset):
    """channels 3-5: clean zoom view."""

    def __init__(self, df, rgbs, train, seed):
        self.a = T.FullSet(df, rgbs, train, seed); self.b = CleanZoomSet(df, rgbs, train, seed)

    @property
    def epoch(self): return self.a.epoch

    @epoch.setter
    def epoch(self, e): self.a.epoch = e; self.b.epoch = e

    def __len__(self): return len(self.a)

    def __getitem__(self, i):
        xa, y = self.a[i]; xb, _ = self.b[i]; return torch.cat([xa, xb], 0), y

In [ ]:
%%writefile {W}/src/anatomy.py
# -*- coding: utf-8 -*-
"""at evaluation those channels are zero and unused. Set GR_OUT (and GR_MASK_ROOT) BEFORE importing this module."""
import os
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset
import views as VW

HY = VW.HY; T, C = HY.T, HY.C
LAM, BITS, P = 0.5, (1, 2, 4, 8, 16, 32), 14
MASKS = {}                                                      # image_id -> uint8 mask (development images only)


def place_mask(m, scale, x0=None, y0=None):
    h, w = m.shape[:2]; s = T.IMG * scale / max(h, w); nw, nh = max(1, round(w * s)), max(1, round(h * s))
    x0, y0 = ((T.IMG - nw) // 2, (T.IMG - nh) // 2) if x0 is None else (x0, y0)
    canvas = np.zeros((T.IMG, T.IMG), np.uint8); canvas[y0:y0 + nh, x0:x0 + nw] = np.asarray(Image.fromarray(m).resize((nw, nh), Image.NEAREST))
    return canvas


def mask_for_aug(m, r):
    import cv2
    h, w = m.shape[:2]; scale = r.uniform(0.85, 1.0); s = T.IMG * scale / max(h, w)
    nw, nh = max(1, round(w * s)), max(1, round(h * s)); x0, y0 = r.randint(0, T.IMG - nw), r.randint(0, T.IMG - nh)
    x = place_mask(m, scale, x0, y0)
    if r.random() < 0.5: r.randint(60, 95)                       # JPEG quality draw (image only)
    if r.random() < 0.3: r.uniform(0.3, 0.8)                     # blur sigma draw (image only)
    if r.random() < 0.5: x = x[:, ::-1].copy()                   # std_augment: flip
    ang = r.uniform(-C.HP["rot_deg"], C.HP["rot_deg"])
    M = cv2.getRotationMatrix2D((T.IMG / 2, T.IMG / 2), ang, 1.0)
    return cv2.warpAffine(x, M, (T.IMG, T.IMG), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT_101)


def targets(m224):
    """uint8 mask (224x224) -> 6 x 224 x 224 float: each 14x14 patch holds the patch's fraction of every layer."""
    layers = np.stack([((m224 & b) > 0).astype(np.float32) for b in BITS])                  # 6 x 224 x 224
    frac = layers.reshape(6, T.IMG // P, P, T.IMG // P, P).mean((2, 4))                       # 6 x 16 x 16
    return np.repeat(np.repeat(frac, P, 1), P, 2)


def mask_of(image_id, rgb):
    m = MASKS[image_id]
    return m if m.shape[:2] == rgb.shape[:2] else np.asarray(Image.fromarray(m).resize((rgb.shape[1], rgb.shape[0]), Image.NEAREST))


class SegFullSet(T.FullSet):

    def __getitem__(self, i):
        r = self.df.iloc[i]; rgb = self.rgbs[r.image_id]
        if self.train:
            key = ("full_aug", r.image_id, self.seed, self.epoch)
            x = T.full_aug(rgb, C._rng(*key)[1]); tg = targets(mask_for_aug(mask_of(r.image_id, rgb), C._rng(*key)[1]))
        else:
            x = T.full_view(rgb); tg = np.zeros((6, T.IMG, T.IMG), np.float32)
        return torch.cat([torch.from_numpy(C.normalize(x[None])[0]), torch.from_numpy(tg)], 0), int(r.y)


class SegDualSet(Dataset):

    def __init__(self, df, rgbs, train, seed):
        self.a = SegFullSet(df, rgbs, train, seed); self.b = VW.CleanZoomSet(df, rgbs, train, seed)

    @property
    def epoch(self): return self.a.epoch

    @epoch.setter
    def epoch(self, e): self.a.epoch = e; self.b.epoch = e

    def __len__(self): return len(self.a)

    def __getitem__(self, i):
        xa, y = self.a[i]; xb, _ = self.b[i]; return torch.cat([xa[:3], xb, xa[3:]], 0), y


# ------------------------------------------------------------------ models
def seg_loss(head, tokens, tgt):
    t = F.avg_pool2d(tgt.float(), P).flatten(2).transpose(1, 2)                               # B x 256 x 6
    return LAM * F.binary_cross_entropy_with_logits(head(tokens).float(), t)


class DinoSeg(T.SingleDino):
    def __init__(self):
        super().__init__(); self.seg = nn.Linear(self.bv.dim, len(BITS)); self._pen = None

    def forward(self, x):
        g, tv = T.dino_feat(self.bv, x[:, :3]); z = self.head(g)
        self._pen = seg_loss(self.seg, tv, x[:, 3:9]) if self.training and x.shape[1] >= 9 else None
        return z

    def penalty(self): return self._pen if self._pen is not None else torch.zeros((), device=self.seg.weight.device)


class SegXAttn(HY.XAttnHybrid):
    def __init__(self, partner):
        super().__init__(partner); self.seg = nn.Linear(self.bv.dim, len(BITS)); self._seg = None

    def forward(self, x, parts=False):
        xv, xp = x[:, :3], x[:, 3:6]
        g, tv = T.dino_feat(self.bv, xv); z = self.head(g)
        p, v = self.pp(self.bp.tokens(self.prep(xp))), self.pv(tv)
        a, _ = self.att_p(p, v, v); p2 = self.ln_p1(p + a); p2 = self.ln_p2(p2 + self.ff_p(p2))
        b, _ = self.att_v(v, p, p); v2 = self.ln_v1(v + b); v2 = self.ln_v2(v2 + self.ff_v(v2))
        dz = self.corr(self.drop(torch.cat([self.pool_p(p2), self.pool_v(v2)], 1)))
        if self.training:
            self._pen = HY.BETA * dz.float().pow(2).mean() / z.detach().float().pow(2).mean().clamp_min(1e-6)
            self._seg = seg_loss(self.seg, tv, x[:, 6:12]) if x.shape[1] >= 12 else None
        return (z, dz) if parts else z + dz

    def penalty(self):
        zero = torch.zeros((), device=self.corr.weight.device)
        return (self._pen if self._pen is not None else zero) + (self._seg if self._seg is not None else zero)

In [ ]:
%%writefile {W}/src/stats_utils.py
# -*- coding: utf-8 -*-
import io, time
import numpy as np, cv2
from PIL import Image
from scipy import stats

REGIONS = {"clinical": 1 | 2 | 4 | 8 | 16,
           "recession": 4 | 8 | 16}                       # where the Cairo type is assessed: margin, papilla, exposed root


# ------------------------------------------------------------------ bootstrap over case groups
def group_bootstrap(stat, groups, n_boot=2000, seed=2026):
    """95% percentile CI of stat(idx) when whole case groups are resampled with replacement."""
    groups = np.asarray(groups); uniq, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == g)[0] for g in range(len(uniq))]
    rng = np.random.default_rng(seed); vals = np.empty(n_boot)
    for b in range(n_boot):
        vals[b] = stat(np.concatenate([members[g] for g in rng.integers(0, len(uniq), len(uniq))]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def accuracy_difference_ci(y, pred_a, pred_b, groups, **kw):
    """CI of accuracy(a) - accuracy(b) on the same images, groups resampled jointly."""
    y, a, b = np.asarray(y), np.asarray(pred_a), np.asarray(pred_b)
    return group_bootstrap(lambda i: (a[i] == y[i]).mean() - (b[i] == y[i]).mean(), groups, **kw)


def region_overlap(cam, mask, bits):
    """cam and mask on the photograph area (same shape, padding excluded). Overlap = fraction of the total map
    intensity inside the region (no threshold); area = fraction of the photograph covered by the region."""
    R = (mask & bits) > 0; cam = np.clip(cam, 0, None)
    if not R.any() or cam.sum() <= 0: return np.nan, float(R.mean())
    return float(cam[R].sum() / cam.sum()), float(R.mean())


def gradcam_regions(cams, masks, correct, classes):
    """Per region: mean overlap, mean area, mean (overlap - area), Wilcoxon signed-rank test of overlap vs area,
    share of images with overlap > area; overall, by class and by correct / incorrect prediction.
    Images whose region is empty are excluded from that region's analysis and counted."""
    from scipy.stats import wilcoxon
    correct, classes = np.asarray(correct, bool), np.asarray(classes)
    out = {}
    for name, bits in REGIONS.items():
        ov, ar = np.array([region_overlap(c, m, bits) for c, m in zip(cams, masks)]).T
        ok = ~np.isnan(ov)

        def summary(sel):
            s = sel & ok; d = ov[s] - ar[s]
            if s.sum() == 0: return {"n": 0}
            r = {"n": int(s.sum()), "overlap": float(ov[s].mean()), "area": float(ar[s].mean()), "difference": float(d.mean()),
                 "share_overlap_above_area": float((d > 0).mean())}
            if s.sum() >= 6 and np.any(d != 0): r["wilcoxon_p"] = float(wilcoxon(ov[s], ar[s]).pvalue)
            return r

        out[name] = {"excluded_empty_region": int((~ok).sum()), "all": summary(np.ones(len(ov), bool)),
                     "correct": summary(correct), "incorrect": summary(~correct),
                     "by_class": {c: summary(classes == c) for c in np.unique(classes)}}
    return out


def midp(a, b):
    """two-sided mid-p McNemar test for a images only the first model gets right and b only the second (Fagerland et al., 2013)."""
    n = a + b
    if n == 0: return 1.0
    k = max(a, b)
    return float(min(1.0, 2 * (stats.binom.sf(k - 1, n, 0.5) - 0.5 * stats.binom.pmf(k, n, 0.5))))


# ------------------------------------------------------------------ treatment pathway
def treatment_changing(y, pred):
    """Errors that change the first-line procedure: every error except RT1 <-> RT2, whose pathways share CAF-based coverage."""
    y, p = np.asarray(y), np.asarray(pred)
    return (p != y) & ~(((y == 1) & (p == 2)) | ((y == 2) & (p == 1)))


def treatment_summary(y, pred):
    y, p = np.asarray(y), np.asarray(pred); t = treatment_changing(y, p)
    return dict(n=int(len(y)), errors=int((p != y).sum()), treatment_changing=int(t.sum()),
                more_invasive=int((t & (p > y)).sum()), less_invasive=int((t & (p < y)).sum()),
                non_adjacent=int((np.abs(p - y) >= 2).sum()), first_line_correct=float(1 - t.mean()))


def treatment_comparison(y, pred_a, pred_b):
    """first-line procedure correct: images only a gets right vs only b, mid-p McNemar; and errors avoided per 100 photographs."""
    ta, tb = treatment_changing(y, pred_a), treatment_changing(y, pred_b)
    a, b = int((~ta & tb).sum()), int((ta & ~tb).sum())
    return dict(only_first=a, only_second=b, p_midp=midp(a, b), avoided_per_100=float(100 * (tb.sum() - ta.sum()) / len(ta)))


# ------------------------------------------------------------------ robustness test (inference only)
DEGRADATIONS = {"jpeg_q30": ("jpeg", 30), "jpeg_q10": ("jpeg", 10),
                "downscale_0.5": ("down", 0.5), "downscale_0.25": ("down", 0.25),
                "blur_s1.5": ("blur", 1.5), "blur_s3": ("blur", 3.0)}


def degrade(rgb, name):
    """the size of the photograph is kept."""
    kind, level = DEGRADATIONS[name]; h, w = rgb.shape[:2]
    if kind == "jpeg":
        buf = io.BytesIO(); Image.fromarray(rgb).save(buf, "JPEG", quality=level); buf.seek(0)
        return np.asarray(Image.open(buf).convert("RGB"))
    if kind == "down":
        small = Image.fromarray(rgb).resize((max(1, round(w * level)), max(1, round(h * level))), Image.BICUBIC)
        return np.asarray(small.resize((w, h), Image.BICUBIC))
    return cv2.GaussianBlur(rgb, (0, 0), level)


def robustness_summary(correct_clean, correct_by_condition):
    """correct_by_condition: {condition: bool array}. Accuracy per condition, mean degraded accuracy, drop from clean,
    and the per-image share of conditions classified correctly (used for the paired comparison of two models)."""
    per = {c: float(np.mean(v)) for c, v in correct_by_condition.items()}
    share = np.mean(np.stack(list(correct_by_condition.values())), 0)
    return dict(clean=float(np.mean(correct_clean)), by_condition=per, mean_degraded=float(np.mean(list(per.values()))),
                drop=float(np.mean(correct_clean) - np.mean(list(per.values())))), share


def robustness_comparison(share_a, share_b):
    """paired Wilcoxon signed-rank test of the per-image share of degraded conditions classified correctly."""
    d = np.asarray(share_a) - np.asarray(share_b)
    return dict(mean_difference=float(d.mean()), images_better_first=int((d > 0).sum()), images_better_second=int((d < 0).sum()),
                wilcoxon_p=float(stats.wilcoxon(share_a, share_b).pvalue) if np.any(d != 0) else 1.0)


# ------------------------------------------------------------------ cost
def model_cost(model, x, device, n_warm=5, n_rep=30):
    """parameters, inference GFLOPs per image (torch FlopCounterMode, one image) and latency in ms per image."""
    import torch
    from torch.utils.flop_counter import FlopCounterMode
    model = model.to(device).eval(); x = x.to(device)
    flags = [p.requires_grad for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(False)        # FlopCounterMode's module tracker fails on expanded trainable parameters
    with torch.no_grad():
        with FlopCounterMode(display=False) as fc: model(x[:1])
        for _ in range(n_warm): model(x[:1])
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_rep): model(x[:1])
        if device == "cuda": torch.cuda.synchronize()
    for p, f in zip(model.parameters(), flags): p.requires_grad_(f)
    return dict(params_m=sum(p.numel() for p in model.parameters()) / 1e6, gflops=fc.get_total_flops() / 1e9,
                latency_ms=1000 * (time.perf_counter() - t0) / n_rep)


def cost_accuracy(rows, reference):
    """rows: {name: dict(gflops, acc)}; For each model: the best reference accuracy reachable at equal or lower cost, and the margin above it."""
    out = {}
    for n, r in rows.items():
        ref = [rows[k]["acc"] for k in reference if k != n and rows[k]["gflops"] <= r["gflops"]]
        best = max(ref) if ref else None
        out[n] = dict(**r, best_reference_at_cost=best, margin=None if best is None else r["acc"] - best)
    return out

In [ ]:
%%writefile {W}/src/experiments.py
# -*- coding: utf-8 -*-
"""Set GR_OUT (output folder) and GR_MASK_ROOT BEFORE importing this module. GR_TEST_LIMIT=n keeps n test photographs."""
import os, glob, json, time, shutil, hashlib, zipfile
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
import anatomy as AN
import stats_utils as X

VW, HY = AN.VW, AN.HY; T, C, V = HY.T, HY.C, HY.V
SEEDS = {k: v for k, v in {"a": 101, "b": 202, "c": 303}.items() if k in os.environ.get("GR_SEEDS", "abc")}
STANDIN = os.environ.get("GR_STANDIN", "0") == "1"
TEST_LIMIT = int(os.environ.get("GR_TEST_LIMIT", "0"))
KEEP_CKPT = os.environ.get("GR_KEEP_CKPT", "0") == "1"
SPLIT_TAG = hashlib.sha256(open(C.SPLIT_CSV, "rb").read()).hexdigest()[:16]   # every run records the split it was trained on

V.CNNS["maxvit_tiny"] = "maxvit_tiny_tf_224.in1k"
V.VITS["dinov2_base"] = "vit_small_patch14_dinov2.lvd142m" if STANDIN else "vit_base_patch14_dinov2.lvd142m"
V.VITS["dinov2_large"] = "vit_small_patch14_dinov2.lvd142m" if STANDIN else "vit_large_patch14_dinov2.lvd142m"
T.DEEP["dinov2_base"] = [["blocks.8"], ["blocks.9"], ["blocks.10"], ["blocks.11", "norm"]]
T.DEEP["dinov2_large"] = T.DEEP["dinov2_base"] if STANDIN else [["blocks.20"], ["blocks.21"], ["blocks.22"], ["blocks.23", "norm"]]
for k in ["maxvit_tiny", "dinov2_base", "dinov2_large"]: V.TAILS.setdefault(k, [])

PARTNERS = {"mnv2": "mobilenetv2", "effb0": "efficientnetb0", "maxvit": "maxvit_tiny"}
PARTS = {"A": ["dino", "h_xxl", "xxl", "mnv2", "effb0", "maxvit", "h_mnv2", "h_effb0", "h_maxvit"],
         "B": ["h_xxl_late", "h_xxl_nomask", "h_mnv2_nomask", "dino_b", "dino_l"],
         "C": ["h_effb0_late", "h_effb0_nomask", "h_effb0_cam"]}   # components of the proposed hybrid
ARMS = [a for p in "ABC" for a in PARTS[p]]
DEGRADE = [a for a in ARMS if not a.endswith("_cam")]
GRADCAM = ["h_xxl", "dino", "h_effb0_cam"]
CTX = {}                                                                        # session data


def arm_of(name): return name.rsplit("_", 1)[0]                                # "h_xxl_a" -> "h_xxl"


base_arm = arm_of


# ------------------------------------------------------------------ models
class PartnerSingle(nn.Module):

    def __init__(self, key):
        super().__init__(); self.bc = V.Branch(key); self.bc.tails = []; self.head = T.linear_head(self.bc.dim)

    def forward(self, x): return self.head(self.bc.m.forward_head(self.bc.m.forward_features(x[:, :3]), pre_logits=True))

    def branches(self): return [self.bc]


class SegXAttnLight(AN.SegXAttn):
    """The proposed hybrid with a light frozen partner on the ImageNet-normalised 224 zoom view (7 x 7 tokens)."""

    def __init__(self, key):
        nn.Module.__init__(self)
        self.bv = T.branch("dinov2_small"); self.head = T.linear_head(2 * self.bv.dim)
        self.bp = V.Branch(key); self.bp.tails = []; self.prep = nn.Identity()
        self.pp, self.pv = V.Proj(self.bp.dim, (C.IMG // 32) ** 2), V.Proj(self.bv.dim, 256)
        mha = lambda: nn.MultiheadAttention(V.D, V.HEADS, dropout=0.1, batch_first=True)
        ff = lambda: nn.Sequential(nn.Linear(V.D, 2 * V.D), nn.GELU(), nn.Dropout(0.1), nn.Linear(2 * V.D, V.D))
        self.att_p, self.att_v = mha(), mha()
        self.ln_p1, self.ln_p2, self.ln_v1, self.ln_v2 = (nn.LayerNorm(V.D) for _ in range(4))
        self.ff_p, self.ff_v = ff(), ff()
        self.pool_p, self.pool_v = V.AttnPool(V.D), V.AttnPool(V.D); self.drop = nn.Dropout(V.DROP)
        self.corr = nn.Linear(2 * V.D, C.NUM); nn.init.zeros_(self.corr.weight); nn.init.zeros_(self.corr.bias)
        self.seg = nn.Linear(self.bv.dim, len(AN.BITS)); self._pen = self._seg = None


class SegLateFusion(AN.SegXAttn):
    """Identical to the proposed hybrid (branches, views, attention pooling, zero-initialised correction, anatomical
    supervision) except that the two token sets are pooled separately, without cross-attention, before concatenation."""

    def __init__(self, partner):
        super().__init__(partner)
        for n in ["att_p", "att_v", "ln_p1", "ln_p2", "ln_v1", "ln_v2", "ff_p", "ff_v"]: delattr(self, n)

    def forward(self, x, parts=False):
        xv, xp = x[:, :3], x[:, 3:6]
        g, tv = T.dino_feat(self.bv, xv); z = self.head(g)
        p, v = self.pp(self.bp.tokens(self.prep(xp))), self.pv(tv)
        dz = self.corr(self.drop(torch.cat([self.pool_p(p), self.pool_v(v)], 1)))
        if self.training:
            self._pen = HY.BETA * dz.float().pow(2).mean() / z.detach().float().pow(2).mean().clamp_min(1e-6)
            self._seg = AN.seg_loss(self.seg, tv, x[:, 6:12]) if x.shape[1] >= 12 else None
        return (z, dz) if parts else z + dz


class SegLateFusionLight(SegXAttnLight):
    """SegLateFusion with a light frozen partner (as SegXAttnLight)."""

    def __init__(self, key):
        super().__init__(key)
        for n in ["att_p", "att_v", "ln_p1", "ln_p2", "ln_v1", "ln_v2", "ff_p", "ff_v"]: delattr(self, n)

    forward = SegLateFusion.forward


class DinoSegX(AN.DinoSeg):

    def __init__(self, key):
        nn.Module.__init__(self)
        self.bv = T.branch(key); self.head = T.linear_head(2 * self.bv.dim)
        self.seg = nn.Linear(self.bv.dim, len(AN.BITS)); self._pen = None


def build(name):
    a = base_arm(name); a = a[:-4] if a.endswith("_cam") else a
    if a == "dino": return AN.DinoSeg()
    if a in ("dino_b", "dino_l"): return DinoSegX({"dino_b": "dinov2_base", "dino_l": "dinov2_large"}[a])
    if a == "xxl": return HY.SingleXXL(False)
    if a in PARTNERS: return PartnerSingle(PARTNERS[a])
    if a == "h_xxl": return AN.SegXAttn("xxl")
    if a == "h_xxl_late": return SegLateFusion("xxl")
    if a == "h_xxl_nomask": return HY.XAttnHybrid("xxl")
    if a.startswith("h_") and a.endswith("_late") and a.split("_")[1] in PARTNERS: return SegLateFusionLight(PARTNERS[a.split("_")[1]])
    if a.startswith("h_") and a.split("_")[1] in PARTNERS: return SegXAttnLight(PARTNERS[a.split("_")[1]])
    raise ValueError(name)


def make_set(name, df, rgbs, train, seed):
    a = base_arm(name)
    if a in ("dino", "dino_b", "dino_l"): return AN.SegFullSet(df, rgbs, train, seed)
    if a == "xxl" or a in PARTNERS: return VW.CleanZoomSet(df, rgbs, train, seed)
    if a.endswith("_nomask"): return VW.CleanDualSet(df, rgbs, train, seed)
    return AN.SegDualSet(df, rgbs, train, seed)                                 # h_xxl, late fusions, light hybrids, h_effb0_cam


T.build = build; T.make_set = make_set


# ------------------------------------------------------------------ one run = training + evaluation of the trained model
def load_net(name, out):
    net = build(name); sd = torch.load(os.path.join(out, "model.pt"), map_location="cpu")
    res = net.load_state_dict({k: v.float() for k, v in sd.items()}, strict=False); assert not res.unexpected_keys, res.unexpected_keys
    return net.to(T.DEV).eval()


@torch.no_grad()
def predict(net, ds, logits=False):
    out = []
    for x, _ in DataLoader(ds, batch_size=32, shuffle=False, num_workers=0):
        with torch.autocast(T.DEV, dtype=torch.float16, enabled=T.DEV == "cuda"):
            z = net(x.to(T.DEV)).float()
        out.append((z if logits else F.softmax(z, 1)).cpu())
    return torch.cat(out).numpy()


def gradcam(net, ds, bs=16):
    """each map scaled to its maximum."""
    for q in net.parameters(): q.requires_grad_(False)
    blk = net.bv.m.blocks[-1]
    for q in blk.parameters(): q.requires_grad_(True)
    acts = {}

    def hook(mod, inp, out): out.retain_grad(); acts["t"] = out

    h = blk.register_forward_hook(hook); kp = net.bv.m.num_prefix_tokens; maps = []
    for x, _ in DataLoader(ds, batch_size=bs, shuffle=False, num_workers=0):
        logits = net(x.to(T.DEV)); per = []
        for c in range(C.NUM):
            acts["t"].grad = None; logits[:, c].sum().backward(retain_graph=c < C.NUM - 1)
            A, G = acts["t"][:, kp:], acts["t"].grad[:, kp:]
            cam = F.relu((A * G.mean(1, keepdim=True)).sum(-1)).detach().cpu().numpy().reshape(-1, 16, 16)
            per.append(cam / (cam.max((1, 2), keepdims=True) + 1e-8))
        maps.append(np.stack(per, 1))
    h.remove(); return np.concatenate(maps).astype(np.float16)                   # (n, 4, 16, 16)


def post_run(name, fold):
    out = os.path.join(T.RUNS, f"{name}_s{fold}"); ck = os.path.join(out, "model.pt")
    if os.path.exists(os.path.join(out, "post.json")) or not os.path.exists(ck): return
    a, test, t0 = arm_of(name), CTX["test"], time.time()
    try:
        net = load_net(name, out)
        sealed = {"clean": predict(net, make_set(name, test, CTX["rgbs_test"], False, fold))}
        if a in DEGRADE:
            for c in X.DEGRADATIONS: sealed[c] = predict(net, make_set(name, test, CTX["degraded"][c], False, fold))
        np.savez(os.path.join(out, "test_sealed.npz"), image_id=test.image_id.to_numpy(), y=test.y.to_numpy(), **sealed)
        if a in GRADCAM:
            np.savez_compressed(os.path.join(out, "gradcam.npz"), image_id=test.image_id.to_numpy(),
                                cams=gradcam(net, make_set(name, test, CTX["rgbs_test"], False, fold)))
        C.save_json(dict(model_sha256=hashlib.sha256(open(ck, "rb").read()).hexdigest(), conditions=list(sealed),
                         minutes=round((time.time() - t0) / 60, 2)), os.path.join(out, "post.json"))
        del net; torch.cuda.empty_cache() if T.DEV == "cuda" else None
        if not KEEP_CKPT: os.remove(ck)
    except RuntimeError as ex:
        print("POST FAILED", name, fold, ex, flush=True); C.save_json({"failed": str(ex)}, os.path.join(out, "post_failed.json"))


_train_one = T.train_one


def train_one(name, fold, dev, rgbs):
    r = _train_one(name, fold, dev, rgbs)
    tag = os.path.join(T.RUNS, f"{name}_s{fold}", "split.txt")
    if os.path.isdir(os.path.dirname(tag)) and not os.path.exists(tag): open(tag, "w").write(SPLIT_TAG)
    post_run(name, fold); return r


T.train_one = train_one


# ------------------------------------------------------------------ session
def import_runs(roots):
    """only runs trained on the current split (split.txt = SPLIT_TAG) are imported, so outputs made with another split are ignored."""
    n, skipped = 0, set()
    for root in roots:
        for z in glob.glob(os.path.join(root, "**", "gingival_part*.zip"), recursive=True):
            with zipfile.ZipFile(z) as f:
                names = set(f.namelist())
                for m in names:
                    run = "/".join(m.split("/")[:2])
                    if not m.startswith("runs/") or os.path.exists(os.path.join(T.OUT, m)): continue
                    if f"{run}/split.txt" not in names or f.read(f"{run}/split.txt").decode().strip() != SPLIT_TAG: skipped.add(run); continue
                    f.extract(m, T.OUT); n += 1
        for v in glob.glob(os.path.join(root, "**", "runs", "*", "val_metrics.json"), recursive=True):
            src = os.path.dirname(v); dst = os.path.join(T.RUNS, os.path.basename(src)); tag = os.path.join(src, "split.txt")
            if os.path.abspath(src) == os.path.abspath(dst) or os.path.exists(dst): continue
            if not os.path.exists(tag) or open(tag).read().strip() != SPLIT_TAG: skipped.add(src); continue
            shutil.copytree(src, dst); n += 1
    if skipped: print(f"ignored {len(skipped)} runs trained on another split", flush=True)
    return n


def names_of(arms): return [f"{a}_{s}" for a in arms for s in SEEDS]


def setup():
    _, dev, test = C.load_split(); test = test.sort_values("image_id").reset_index(drop=True)
    if TEST_LIMIT: test = test.groupby("cls").head(max(1, TEST_LIMIT // C.NUM)).reset_index(drop=True)
    CTX.update(dev=dev, test=test, rgbs=T.load_all_rgbs(dev), rgbs_test=T.load_all_rgbs(test))
    CTX["degraded"] = {c: {i: X.degrade(r, c) for i, r in CTX["rgbs_test"].items()} for c in X.DEGRADATIONS}
    AN.MASKS.update({r.image_id: C.load_mask(r) for r in dev.itertuples()})
    T.INIT.update({n: SEEDS[n[-1]] for n in names_of(ARMS)})


def run_part(part):
    setup(); folds = T.SEEDS[:2] if T.QUICK else T.SEEDS; names = names_of(PARTS[part])
    print(f"part {part}: {len(names)} models x {len(folds)} folds on {T.DEV}; {len(CTX['rgbs'])} development and "
          f"{len(CTX['test'])} test photographs; XXL = {V.CNNS[HY.XKEY]}", flush=True)
    for n in names:
        for k in folds: post_run(n, k)
    done = T.run_models(names, CTX["dev"], CTX["rgbs"], folds)
    missing = [f"{n}_s{k}" for n in names for k in folds if not os.path.exists(os.path.join(T.RUNS, f"{n}_s{k}", "post.json"))]
    print(f"part {part}: {'complete' if done and not missing else 'INCOMPLETE - run this part again with its outputs attached'}"
          f"{'' if not missing else ' (' + str(len(missing)) + ' runs missing)'}", flush=True)
    return done and not missing


# ------------------------------------------------------------------ cost of every arm
def measure_costs(path):
    rows, same = {}, {"h_xxl_nomask": "h_xxl", "h_mnv2_nomask": "h_mnv2", "h_effb0_nomask": "h_effb0"}
    trainable = lambda net: sum(p.numel() for n, p in net.named_parameters() if n in T.save_keys(net)) / 1e6
    for a in DEGRADE:
        net = build(f"{a}_a"); r = dict(rows[same[a]]) if a in same else {}
        if a not in same:
            x = torch.randn(1, 6 if a.startswith("h_") else 3, C.IMG, C.IMG)
            if torch.cuda.is_available(): r["gpu"] = X.model_cost(net, x, "cuda")
            r["cpu"] = X.model_cost(net, x, "cpu", n_warm=1, n_rep=3)
        rows[a] = dict(r, trainable_m=trainable(net)); print("cost", a, rows[a], flush=True)
        del net; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    C.save_json(rows, path); return rows

In [ ]:
%%writefile {W}/src/analysis.py
# -*- coding: utf-8 -*-
"""then the test set, the robustness test, Grad-CAM (Figure 4) and the confusion / ROC figures."""
import os, sys, json, hashlib, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score
from PIL import Image
import common as C
import stats_utils as X

CL, SEEDS, RHO, B = C.CLASSES, ["a", "b", "c"], 0.25, 2000
DINO_BACKBONES, LIGHT = ["dino", "dino_b", "dino_l"], ["h_mnv2", "h_effb0", "h_maxvit"]
PARTNER_OF = {"h_mnv2": "mnv2", "h_effb0": "effb0", "h_maxvit": "maxvit", "h_xxl": "xxl"}
PROPOSED = os.environ.get("GR_PROPOSED", "h_effb0")
TABLE3 = ["xxl", "mnv2", "effb0", "maxvit", "dino", PROPOSED]
HYB = {"h_mnv2": "Hybrid + MobileNetV2", "h_effb0": "Hybrid + EfficientNetB0", "h_maxvit": "Hybrid + MaxViT-T", "h_xxl": "Hybrid + ConvNeXt-XXLarge"}
HYB[PROPOSED] = "Proposed hybrid"
NAME = {"dino": "DINOv2-small", "dino_b": "DINOv2-base", "dino_l": "DINOv2-large",
        "mnv2": "MobileNetV2 (frozen)", "effb0": "EfficientNetB0 (frozen)", "maxvit": "MaxViT-T (frozen)", "xxl": "ConvNeXt-XXLarge CLIP (frozen)",
        **HYB, **{f"{h}_late": f"{n} without cross-attention" for h, n in HYB.items()},
        **{f"{h}_nomask": f"{n} without anatomical supervision" for h, n in HYB.items()}}
P_LATE, P_NOMASK = f"{PROPOSED}_late", f"{PROPOSED}_nomask"
ABLATIONS = list(dict.fromkeys([(PROPOSED, P_LATE), (PROPOSED, P_NOMASK), ("h_xxl", "h_xxl_late"), ("h_xxl", "h_xxl_nomask"), ("h_mnv2", "h_mnv2_nomask")]))
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
acc = lambda P, y: float((P.argmax(1) == y).mean())


# ------------------------------------------------------------------ predictions
class Store:
    """arm -> seed -> fold -> run folder; only seeds with all folds are used."""

    def __init__(self, runs):
        self.R = {}
        for d in sorted(os.listdir(runs)):
            if "_s" not in d or not os.path.exists(os.path.join(runs, d, "val_probs.npz")): continue
            name, k = d.rsplit("_s", 1); self.R.setdefault(name[:-2], {}).setdefault(name[-1], {})[int(k)] = os.path.join(runs, d)
        self.folds = sorted({k for a in self.R.values() for s in a.values() for k in s})

    def seeds(self, arm): return [s for s in SEEDS if all(k in self.R.get(arm, {}).get(s, {}) for k in self.folds)]

    def val(self, arm):
        Ps = []
        for s in self.seeds(arm):
            z = [np.load(os.path.join(self.R[arm][s][k], "val_probs.npz"), allow_pickle=True) for k in self.folds]
            ids = np.concatenate([q["image_id"].astype(str) for q in z]); o = np.argsort(ids)
            y = np.concatenate([q["y"] for q in z]).astype(int)[o]
            f = np.concatenate([np.full(len(q["y"]), k) for k, q in zip(self.folds, z)])[o]; Ps.append(np.concatenate([q["P"] for q in z])[o])
        return (ids[o], y, f, Ps) if Ps else None

    def test(self, arm, cond="clean"):
        Ps = []
        for s in self.seeds(arm):
            paths = [os.path.join(self.R[arm][s][k], "test_sealed.npz") for k in self.folds]
            if not all(os.path.exists(p) for p in paths): return None
            z = [np.load(p, allow_pickle=True) for p in paths]
            if not all(cond in q.files for q in z): return None
            Ps.append(np.mean([q[cond] for q in z], 0)); ids, y = z[0]["image_id"].astype(str), z[0]["y"].astype(int)
        return (ids, y, None, Ps) if Ps else None


def midp(a, b): return X.midp(a, b)


def mcnemar(ca, cb):
    a, b = int((ca & ~cb).sum()), int((~ca & cb).sum()); return dict(first_only=a, second_only=b, p_midp=midp(a, b))


def corrected_t(d):
    d = np.asarray(d, float); n = len(d); v = d.var(ddof=1) if n > 1 else 0.0
    if v == 0: return dict(mean=float(d.mean()), t=0.0, p=1.0, n=n)
    t = d.mean() / np.sqrt((1 / n + RHO) * v); return dict(mean=float(d.mean()), t=float(t), p=float(2 * stats.t.sf(abs(t), n - 1)), n=n)


def perm(d, n=10_000, seed=2026):
    d = np.asarray(d, float); rng = np.random.default_rng(seed)
    null = np.abs((rng.choice([-1.0, 1.0], size=(n, len(d))) * d).mean(1))
    return dict(mean_diff=float(d.mean()), p_perm=float((1 + (null >= abs(d.mean())).sum()) / (n + 1)), n=int(len(d)))


def ece(P, y, bins=10):
    conf, pred = P.max(1), P.argmax(1); e = 0.0
    for lo in np.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)


def aurc(P, y):
    o = np.argsort(-P.max(1), kind="stable"); c = (P.argmax(1) == y)[o]
    return float((1 - np.cumsum(c) / np.arange(1, len(c) + 1)).mean())


def wrong_first_line_kept(P, y, refer=0.2):
    o = np.argsort(-P.max(1), kind="stable")[:int(round((1 - refer) * len(y)))]
    return float(100 * X.treatment_changing(y[o], P.argmax(1)[o]).mean())


def boot_ci(fn, n, seed=2026):
    rng = np.random.default_rng(seed); v = np.array([fn(rng.integers(0, n, n)) for _ in range(B)])
    return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))], v


# ------------------------------------------------------------------ summaries and comparisons
def confidence(E, y):
    p, conf = E.argmax(1), E.max(1); ok = p == y; o = np.argsort(-conf, kind="stable"); n = len(y)
    r = dict(log_loss=float(-np.log(np.clip(E[np.arange(n), y], 1e-7, 1)).mean()), aurc=aurc(E, y), ece=ece(E, y),
             errors_conf_ge_90=int((~ok & (conf >= 0.9)).sum()), wrong_first_line_per_100_kept=wrong_first_line_kept(E, y))
    for c in (0.7, 0.8, 0.9): r[f"acc_at_{int(100 * c)}"] = float(ok[o][:int(c * n)].mean())
    return r


def summarise(y, f, Ps):
    E = np.mean(Ps, 0); m = C.metrics(y, E); pr = [P.argmax(1) for P in Ps]; sa = [float((q == y).mean()) for q in pr]
    s = dict(n=int(len(y)), n_seeds=len(Ps), ensemble={k: m[k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "kappa_quadratic", "macro_auc", "log_loss"]},
             per_class=m["per_class"], confusion_matrix=m["confusion_matrix"], seed_accuracy=sa, seed_mean=float(np.mean(sa)),
             seed_sd=float(np.std(sa, ddof=1)) if len(sa) > 1 else 0.0, worst_seed=float(min(sa)),
             unstable_images=int((np.stack(pr) != pr[0]).any(0).sum()), confidence=confidence(E, y), treatment=X.treatment_summary(y, E.argmax(1)))
    if f is not None:
        fa = [float((q == y)[f == k].mean()) for q in pr for k in np.unique(f)]
        s.update(fold_accuracy_mean=float(np.mean(fa)), fold_accuracy_sd=float(np.std(fa, ddof=1)) if len(fa) > 1 else 0.0)
    return s


def compare(y, f, PA, PB, groups=None):
    EA, EB = np.mean(PA, 0), np.mean(PB, 0); ca, cb = EA.argmax(1) == y, EB.argmax(1) == y; n = len(y); k = min(len(PA), len(PB))
    d = [acc(PA[i], y) - acc(PB[i], y) for i in range(k)]
    out = dict(acc=[float(ca.mean()), float(cb.mean())], mcnemar=mcnemar(ca, cb), seed_diffs=d, wins=int(sum(x > 0 for x in d)))
    if f is not None:
        out["corrected_t_test"] = corrected_t([((PA[i].argmax(1) == y)[f == j].mean() - (PB[i].argmax(1) == y)[f == j].mean())
                                               for i in range(k) for j in np.unique(f)])
    la, lb = (-np.log(np.clip(E[np.arange(n), y], 1e-7, 1)) for E in (EA, EB))
    out.update(log_loss=[float(la.mean()), float(lb.mean())], log_loss_perm=perm(lb - la), aurc=[aurc(EA, y), aurc(EB, y)])
    out["aurc_diff_ci95"] = boot_ci(lambda i: aurc(EB[i], y[i]) - aurc(EA[i], y[i]), n)[0]                 # > 0: first better
    out["treatment"] = X.treatment_comparison(y, EA.argmax(1), EB.argmax(1))
    out["wrong_first_line_per_100_kept"] = [wrong_first_line_kept(EA, y), wrong_first_line_kept(EB, y)]
    out["wrong_first_line_diff_ci95"] = boot_ci(lambda i: wrong_first_line_kept(EB[i], y[i]) - wrong_first_line_kept(EA[i], y[i]), n)[0]
    if groups is not None: out["accuracy_difference_case_group_ci95"] = X.accuracy_difference_ci(y, EA.argmax(1), EB.argmax(1), groups)
    return out


def analyse(D, y, f, groups):
    """Model summaries and paired comparisons for one set (cross-validation or test)."""
    R = dict(models={a: summarise(y, f, P) for a, P in D.items()}, comparisons={})
    cmp = lambda a, b: compare(y, f, D[a], D[b], groups) if a in D and b in D else None
    for a, b in dict.fromkeys([(PROPOSED, "dino"), ("h_xxl", "dino")] + ABLATIONS + [(h, p) for h, p in PARTNER_OF.items()] +
                              [(h, "dino") for h in LIGHT + [P_LATE, "h_xxl_late", "dino_b", "dino_l"]]):
        c = cmp(a, b)
        if c: R["comparisons"][f"{a}_vs_{b}"] = c
    return R


def cost_accuracy(costs, cv):
    rows = {}
    for a, c in costs.items():
        if a not in cv["models"]: continue
        g = (c.get("gpu") or c["cpu"])["gflops"]; m = cv["models"][a]
        rows[f"{a} x1"] = dict(gflops=g, acc=m["seed_mean"]); rows[f"{a} x3"] = dict(gflops=3 * g, acc=m["ensemble"]["accuracy"])
    ref = [k for k in rows if k.split()[0] in DINO_BACKBONES]
    return X.cost_accuracy(rows, ref)


# ------------------------------------------------------------------ Grad-CAM and figures
def photo_box(h, w):
    s = C.IMG / max(h, w); nw, nh = max(1, round(w * s)), max(1, round(h * s)); return (C.IMG - nw) // 2, (C.IMG - nh) // 2, nw, nh


def gradcam_maps(S, arm, pred):
    maps, ids = [], None
    for s in S.seeds(arm):
        for k in S.folds:
            p = os.path.join(S.R[arm][s][k], "gradcam.npz")
            if not os.path.exists(p): return None, None
            z = np.load(p, allow_pickle=True); maps.append(z["cams"].astype(np.float32)); ids = z["image_id"].astype(str)
    M = np.mean(maps, 0); return M[np.arange(len(pred)), pred], ids


def gradcam_on_photos(cams, ids, split):
    out = []
    for cam, i in zip(cams, ids):
        r = split.loc[i]; rgb = C.load_rgb(r.path); m = C.load_mask(r); h, w = rgb.shape[:2]; x0, y0, nw, nh = photo_box(h, w)
        c = cv2.resize(cam, (C.IMG, C.IMG), interpolation=cv2.INTER_CUBIC).clip(0, None)[y0:y0 + nh, x0:x0 + nw]; c = c / (c.max() + 1e-8)
        out.append(dict(image_id=i, photo=np.asarray(Image.fromarray(rgb).resize((nw, nh), Image.BICUBIC)), cam=c,
                        mask=np.asarray(Image.fromarray(m).resize((nw, nh), Image.NEAREST))))
    return out


def figure4(items, rows, path):
    WR, CW = [2.2, 2.45, 2.45, 2.45, 1.75], 2.45
    HR = [max(CW * it["photo"].shape[0] / it["photo"].shape[1], 1.2) for it in items]; H = sum(HR) * 1.07 + 0.55
    fig = plt.figure(figsize=(sum(WR) + 0.3, H))
    gs = fig.add_gridspec(len(items), 5, width_ratios=WR, height_ratios=HR, wspace=0.06, hspace=0.07, left=0.01, right=0.99, top=1 - 0.5 / H, bottom=0.005)
    for r, (it, d) in enumerate(zip(items, rows)):
        ax = [fig.add_subplot(gs[r, j]) for j in range(5)]; ok = d["correct"]
        heat = (plt.cm.jet(it["cam"])[..., :3] * 255).astype(np.uint8); overlay = (0.55 * it["photo"] + 0.45 * heat).astype(np.uint8)
        at = lambda a, x, dy, t, **k: a.annotate(t, (x, 0.5), xycoords="axes fraction", xytext=(0, dy), textcoords="offset points", va="center", **k)
        at(ax[0], 0.06, 13, f"Actual: {d['actual']}\nPredicted: {d['predicted']}\nConfidence: {d['confidence']:.2f}", fontsize=10.5, color="#222222", linespacing=1.35)
        at(ax[0], 0.06, -19, "✓ correct" if ok else "✗ incorrect", fontsize=10.5, fontweight="bold", color="#1b7a3a" if ok else "#b2182b")
        at(ax[0], 0.06, -34, d["file"], fontsize=8.5, color="#777777")
        for a, im in zip(ax[1:4], [it["photo"], it["cam"], overlay]):
            a.imshow(im, cmap="jet" if im is it["cam"] else None, vmin=0, vmax=1, aspect="equal", interpolation="bilinear"); a.set_xticks([]); a.set_yticks([])
            for s in a.spines.values(): s.set_linewidth(0.5); s.set_color("#999999")
        at(ax[4], 0.5, 11, f"{100 * d['overlap_clinical']:.1f}%", ha="center", fontsize=13, fontweight="bold", color="#222222")
        at(ax[4], 0.5, -11, f"clinical region\n= {100 * d['area_clinical']:.1f}% of image", ha="center", fontsize=8.5, color="#666666", linespacing=1.25)
        for a in (ax[0], ax[4]): a.axis("off")
        if r: fig.add_artist(plt.Line2D([0.015, 0.985], [ax[0].get_position().y1 + 0.035 * HR[r] / H] * 2, color="#dddddd", lw=0.6))
        else:
            for a, t in zip(ax, ["Class (actual / predicted)", "Original", "Grad-CAM", "Overlay", "Overlap with\nclinical region"]):
                a.set_title(t, fontsize=11.5, fontweight="bold", color="#222222", pad=7)
    plt.savefig(path, dpi=300, facecolor="white"); plt.close()


def confusion_roc_figures(sets, out):
    from sklearn.metrics import roc_curve, auc as sk_auc
    fig, ax = plt.subplots(1, len(sets), figsize=(5.2 * len(sets), 4.6))
    for a, (title, y, E) in zip(np.atleast_1d(ax), sets):
        cm = np.zeros((4, 4), int); np.add.at(cm, (y, E.argmax(1)), 1)
        a.imshow(cm, cmap="Blues"); a.set_xticks(range(4)); a.set_yticks(range(4)); a.set_xticklabels(CL); a.set_yticklabels(CL)
        for i in range(4):
            for j in range(4): a.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=11)
        a.set_xlabel("Predicted"); a.set_ylabel("Actual"); a.set_title(title, fontsize=11)
    plt.tight_layout(); plt.savefig(os.path.join(out, "Fig2_confusion_matrices.png"), dpi=300); plt.close()
    fig, ax = plt.subplots(1, len(sets), figsize=(5.2 * len(sets), 4.6))
    for a, (title, y, E) in zip(np.atleast_1d(ax), sets):
        for c in range(4):
            if 0 < (y == c).sum() < len(y):
                fpr, tpr, _ = roc_curve(y == c, E[:, c]); a.plot(fpr, tpr, lw=1.6, label=f"{CL[c]} (AUC {sk_auc(fpr, tpr):.3f})")
        a.plot([0, 1], [0, 1], "--", color="#999999", lw=0.8); a.set_xlabel("False positive rate"); a.set_ylabel("True positive rate")
        a.set_title(title, fontsize=11); a.legend(fontsize=8.5, loc="lower right")
    plt.tight_layout(); plt.savefig(os.path.join(out, "Fig3_roc_curves.png"), dpi=300); plt.close()


# ------------------------------------------------------------------ tables
def tables(cv, te, rob, costs, out):
    M, Mt = cv["models"], te["models"] if te else {}
    t3 = []
    for a in [x for x in TABLE3 if x in M]:
        m, t = M[a], Mt.get(a)
        t3.append({"Model": NAME[a], "Seeds": m["n_seeds"], "CV accuracy": f"{100 * m['fold_accuracy_mean']:.1f}% ± {100 * m['fold_accuracy_sd']:.1f}",
                   "CV ensemble": f"{100 * m['ensemble']['accuracy']:.1f}%", "Balanced accuracy (CV)": f"{100 * m['ensemble']['balanced_accuracy']:.1f}%",
                   "Test accuracy (95% CI)": f"{100 * t['ensemble']['accuracy']:.1f}% ({100 * t['accuracy_ci95'][0]:.1f}–{100 * t['accuracy_ci95'][1]:.1f})" if t else "",
                   "Macro-F1 (test)": f"{t['ensemble']['macro_f1']:.3f}" if t else "", "Quadratic κ (test)": f"{t['ensemble']['kappa_quadratic']:.3f}" if t else "",
                   "Macro-AUC (test)": f"{t['ensemble']['macro_auc']:.3f}" if t else ""})
    pd.DataFrame(t3).to_csv(os.path.join(out, "singles_and_proposed.csv"), index=False)
    if PROPOSED in M:
        pc, ci = M[PROPOSED]["per_class"], M[PROPOSED]["per_class_ci95"]
        pd.DataFrame([{"Class": c, "n": pc[c]["n"], "Sensitivity": f"{100 * pc[c]['sensitivity']:.1f}% ({100 * ci[c]['sensitivity'][0]:.1f}–{100 * ci[c]['sensitivity'][1]:.1f})",
                       "Specificity": f"{100 * pc[c]['specificity']:.1f}%", "PPV": f"{100 * pc[c]['ppv']:.1f}%", "NPV": f"{100 * pc[c]['npv']:.1f}%",
                       "F1": f"{pc[c]['f1']:.3f}", "AUC": f"{pc[c]['auc']:.3f} ({ci[c]['auc'][0]:.3f}–{ci[c]['auc'][1]:.3f})"} for c in CL]
                     ).to_csv(os.path.join(out, "per_class_proposed_cv.csv"), index=False)
    t5 = []
    for h, p in list(PARTNER_OF.items()) + ABLATIONS:
        c = cv["comparisons"].get(f"{h}_vs_{p}")
        if c: t5.append({"Hybrid": NAME[h], "Compared with": NAME[p], "CV ensemble (hybrid)": f"{100 * c['acc'][0]:.1f}%",
                         "CV ensemble (compared)": f"{100 * c['acc'][1]:.1f}%", "Difference": f"{100 * (c['acc'][0] - c['acc'][1]):+.1f}",
                         "Only hybrid / only compared correct": f"{c['mcnemar']['first_only']} / {c['mcnemar']['second_only']}",
                         "mid-p": f"{c['mcnemar']['p_midp']:.3f}", "Log loss": f"{c['log_loss'][0]:.3f} / {c['log_loss'][1]:.3f}"})
    pd.DataFrame(t5).to_csv(os.path.join(out, "partners_and_components.csv"), index=False)
    rows = []
    for a in M:
        for tag, S in [("CV", M), ("test", Mt)]:
            if a in S:
                m = S[a]; c, t = m["confidence"], m["treatment"]
                rows.append({"Model": NAME.get(a, a), "Set": tag, "Accuracy": round(m["ensemble"]["accuracy"], 4), "Log loss": round(c["log_loss"], 4),
                             "AURC": round(c["aurc"], 4), "ECE": round(c["ece"], 4), "Accuracy at 80% coverage": round(c["acc_at_80"], 4),
                             "Errors with confidence >= 0.9": c["errors_conf_ge_90"], "Treatment-changing errors": t["treatment_changing"],
                             "More / less invasive": f"{t['more_invasive']} / {t['less_invasive']}", "Non-adjacent": t["non_adjacent"],
                             "First-line correct": round(t["first_line_correct"], 4), "Wrong first-line per 100 kept (20% referred)": round(c["wrong_first_line_per_100_kept"], 2),
                             "Seed SD": round(m["seed_sd"], 4), "Worst seed": round(m["worst_seed"], 4), "Photos whose class differs between seeds": m["unstable_images"]})
    pd.DataFrame(rows).to_csv(os.path.join(out, "table_confidence_treatment_stability.csv"), index=False)
    if costs:
        pd.DataFrame([{"Model": NAME.get(a, a), "Parameters (M)": round(c["cpu"]["params_m"], 1), "Trainable (M)": round(c.get("trainable_m", float("nan")), 2),
                       "GFLOPs": round((c.get("gpu") or c["cpu"])["gflops"], 1), "Latency GPU (ms)": round(c["gpu"]["latency_ms"], 1) if c.get("gpu") else "",
                       "Latency CPU (ms)": round(c["cpu"]["latency_ms"], 0), "CV accuracy (single)": round(M[a]["seed_mean"], 4) if a in M else "",
                       "CV accuracy (3 seeds)": round(M[a]["ensemble"]["accuracy"], 4) if a in M else ""} for a, c in costs.items()]
                     ).to_csv(os.path.join(out, "table_cost.csv"), index=False)
    if rob: pd.DataFrame([{"Model": NAME.get(a, a), **{k: round(v, 4) for k, v in r["by_condition"].items()}, "clean": round(r["clean"], 4),
                           "mean degraded": round(r["mean_degraded"], 4), "drop": round(r["drop"], 4)} for a, r in rob["models"].items()]
                         ).to_csv(os.path.join(out, "table_robustness.csv"), index=False)


# ------------------------------------------------------------------ main
def per_class_ci(y, E):
    fns = {}
    for i, c in enumerate(CL):
        fns[f"{c}|sensitivity"] = lambda yy, P, i=i: float((P.argmax(1)[yy == i] == i).mean()) if (yy == i).any() else np.nan
        fns[f"{c}|auc"] = lambda yy, P, i=i: float(roc_auc_score(yy == i, P[:, i]))
    r = C.bootstrap_ci(y, E, fns); out = {c: {} for c in CL}
    for k, v in r.items(): c, m = k.split("|"); out[c][m] = v
    return out


def main(runs, out, costs_path=None):
    os.makedirs(out, exist_ok=True); S = Store(runs); split = C.load_split()[0].set_index("image_id")
    GROUP = "case_group"                   # case groups of the split in use
    arms = [a for a in NAME if S.seeds(a)]
    V = {a: S.val(a) for a in arms}; ids, y, f = V[arms[0]][0], V[arms[0]][1], V[arms[0]][2]
    assert all((v[0] == ids).all() for v in V.values()), "out-of-fold images differ between arms"
    groups = split.loc[ids, GROUP].to_numpy()
    cv = analyse({a: V[a][3] for a in arms}, y, f, groups)
    E = np.mean(V[PROPOSED][3], 0) if PROPOSED in V else None
    if E is not None: cv["models"][PROPOSED]["per_class_ci95"] = per_class_ci(y, E)
    cv["proposed"] = PROPOSED
    costs = C.load_json(costs_path) if costs_path and os.path.exists(costs_path) else None
    if costs: cv["cost_accuracy"] = cost_accuracy(costs, cv)
    cv.update(n_images=int(len(y)), folds=S.folds, seeds={a: S.seeds(a) for a in arms}, written=time.strftime("%Y-%m-%d %H:%M:%S"))
    C.save_json(cv, os.path.join(out, "cv_analysis.json"))
    C.save_json(dict(written=time.strftime("%Y-%m-%d %H:%M:%S"), cv_analysis_sha256=sha(os.path.join(out, "cv_analysis.json")),
                     val_probs_sha256={os.path.basename(S.R[a][s][k]): sha(os.path.join(S.R[a][s][k], "val_probs.npz")) for a in arms for s in S.seeds(a) for k in S.folds}),
                os.path.join(out, "cv_analysis_hash.json"))
    print("cross-validation analysis written and locked", flush=True)
    Tt = {a: S.test(a) for a in arms}; Tt = {a: v for a, v in Tt.items() if v}; te = rob = gc = None
    if Tt:
        tid, ty = next(iter(Tt.values()))[:2]; assert all((v[0] == tid).all() for v in Tt.values())
        te = analyse({a: v[3] for a, v in Tt.items()}, ty, None, split.loc[tid, GROUP].to_numpy())
        for a, v in Tt.items():
            Et = np.mean(v[3], 0); te["models"][a]["accuracy_ci95"] = C.bootstrap_ci(ty, Et, {"acc": lambda yy, P: float((P.argmax(1) == yy).mean())})["acc"]
        rob = dict(models={}, comparisons={}); share = {}
        for a in Tt:
            deg = {c: S.test(a, c) for c in X.DEGRADATIONS}
            if not all(deg.values()): continue
            ok_clean = np.mean(Tt[a][3], 0).argmax(1) == ty
            s, share[a] = X.robustness_summary(ok_clean, {c: np.mean(v[3], 0).argmax(1) == ty for c, v in deg.items()}); rob["models"][a] = s
        for a in dict.fromkeys([PROPOSED, "h_xxl"]):
            if a in share and "dino" in share: rob["comparisons"][f"{a}_vs_dino"] = X.robustness_comparison(share[a], share["dino"])
        gc = {}
        for a in [PROPOSED, "dino"]:
            if a not in Tt: continue
            src = f"{a}_cam" if S.seeds(f"{a}_cam") else a                  # replicate trained for Grad-CAM (same seeds)
            Ea = np.mean(Tt[a][3], 0); pred = Ea.argmax(1); cams, cid = gradcam_maps(S, src, pred)
            if cams is None: continue
            if src != a and (rep := S.test(src)):
                gc["cam_source"] = dict(arm=src, seeds=S.seeds(src), test_prediction_agreement=float((np.mean(rep[3], 0).argmax(1) == pred).mean()),
                                        test_accuracy=acc(np.mean(rep[3], 0), ty))
            assert (cid == tid).all(); items = gradcam_on_photos(cams, cid, split)
            gc[a] = dict(regions=X.gradcam_regions([it["cam"] for it in items], [it["mask"] for it in items], pred == ty, [CL[i] for i in ty]),
                         per_image={reg: [X.region_overlap(it["cam"], it["mask"], b) for it in items] for reg, b in X.REGIONS.items()})
            if a == PROPOSED:
                rng = np.random.default_rng(2026); picks, rows = [], []
                for i, c in enumerate(CL):
                    for ok in (True, False):
                        cand = np.where((ty == i) & ((pred == ty) == ok))[0]
                        if len(cand): picks.append(int(rng.choice(cand)))
                for k in picks:
                    ov, ar = X.region_overlap(items[k]["cam"], items[k]["mask"], X.REGIONS["clinical"])
                    rows.append(dict(image_id=tid[k], actual=CL[ty[k]], predicted=CL[pred[k]], correct=bool(pred[k] == ty[k]), confidence=float(Ea[k].max()),
                                     file=os.path.basename(split.loc[tid[k], "path"]), overlap_clinical=ov, area_clinical=ar))
                if picks: figure4([items[k] for k in picks], rows, os.path.join(out, "Fig4_gradcam.png"))
                gc["figure4"] = rows
        if PROPOSED in gc and "dino" in gc:
            dh = np.array([o - a for o, a in gc[PROPOSED]["per_image"]["recession"]]); dd = np.array([o - a for o, a in gc["dino"]["per_image"]["recession"]])
            ok = ~np.isnan(dh) & ~np.isnan(dd)
            gc[f"{PROPOSED}_vs_dino_recession"] = dict(n=int(ok.sum()), mean_difference=float((dh[ok] - dd[ok]).mean()),
                                                 wilcoxon_p=float(stats.wilcoxon(dh[ok], dd[ok]).pvalue) if np.any(dh[ok] != dd[ok]) else 1.0)
        for a in gc:
            if isinstance(gc[a], dict) and "per_image" in gc[a]: gc[a]["per_image"] = {k: [[float(o), float(r)] for o, r in v] for k, v in gc[a]["per_image"].items()}
        if PROPOSED in Tt and E is not None:
            confusion_roc_figures([("(a) Cross-validation", y, E), ("(b) Test set", ty, np.mean(Tt[PROPOSED][3], 0))], out)
    tables(cv, te, rob, costs, out)
    res = dict(cross_validation=cv, test=te, robustness=rob, gradcam=gc, lock=C.load_json(os.path.join(out, "cv_analysis_hash.json")))
    C.save_json(res, os.path.join(out, "results.json")); print("results written to", out, flush=True)
    return res


if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2], sys.argv[3] if len(sys.argv) > 3 else None)

In [ ]:
%%writefile {W}/src/treatment_mapping.py
# -*- coding: utf-8 -*-
"""Rule-based management pathway: deterministic lookup from the predicted Cairo class.
Wording copied from Table 3.7 of the thesis ("Simplified rule-based management pathway according to the Cairo
recession classification (decision aid; not a clinical guideline)")."""

TABLE_3_7 = {
    "HEALTHY": dict(definition="No gingival recession.",
                    goal="Maintain periodontal health.",
                    approaches="Preventive monitoring; routine maintenance and oral hygiene.",
                    prognosis="—"),
    "RT1": dict(definition="Recession with no interproximal attachment loss; interproximal CEJ not clinically detectable.",
                goal="Complete root coverage (CRC).",
                approaches="CAF as a well-established first-line approach; CTG may be added according to phenotype and "
                           "objectives (CAF ± CTG); tunnel or MCAT ± CTG; VISTA ± CTG.",
                prognosis="Favourable: CRC generally predictable."),
    "RT2": dict(definition="Interproximal attachment loss ≤ buccal attachment loss.",
                goal="Maximum achievable root coverage and improved periodontal phenotype.",
                approaches="CAF + CTG commonly considered when additional tissue augmentation is indicated; MCAT + CTG; "
                           "tunnel techniques + CTG; VISTA + CTG.",
                prognosis="Intermediate: CRC possible in selected cases, less predictable than RT1."),
    "RT3": dict(definition="Interproximal attachment loss > buccal attachment loss.",
                goal="Tissue stabilisation, hypersensitivity reduction, plaque control, aesthetics.",
                approaches="Individualised management: soft-tissue augmentation; CTG-based, tunnel or regenerative "
                           "procedures when indicated; specialist evaluation.",
                prognosis="Less predictable: CRC generally not predictable; outcomes depend on individual clinical factors."),
}
FIRST_LINE = {"HEALTHY": "Preventive monitoring", "RT1": "CAF-based root coverage", "RT2": "CAF-based root coverage",
              "RT3": "Individualised management"}


def suggest(predicted_class):
    return TABLE_3_7[predicted_class.upper()]

## Pretrained weights

In [ ]:
import timm, time
NEED = {"A": ["vit_small_patch14_dinov2.lvd142m", "convnext_xxlarge.clip_laion2b_soup", "mobilenetv2_100.ra_in1k", "efficientnet_b0.ra_in1k", "maxvit_tiny_tf_224.in1k"],
        "B": ["vit_small_patch14_dinov2.lvd142m", "convnext_xxlarge.clip_laion2b_soup", "mobilenetv2_100.ra_in1k", "vit_base_patch14_dinov2.lvd142m", "vit_large_patch14_dinov2.lvd142m"],
        "C": ["vit_small_patch14_dinov2.lvd142m", "convnext_xxlarge.clip_laion2b_soup", "mobilenetv2_100.ra_in1k", "efficientnet_b0.ra_in1k", "maxvit_tiny_tf_224.in1k",
              "vit_base_patch14_dinov2.lvd142m", "vit_large_patch14_dinov2.lvd142m"]}[PART]
for n in NEED:
    for k in range(6):
        try: timm.create_model(n, pretrained=True, num_classes=0, **({"img_size": 224} if "dinov2" in n else {})); print("ok", n, flush=True); break
        except Exception as e: print("retry", n, str(e)[:200], flush=True); time.sleep(20)

## Training and evaluation of this part
Finished runs from attached outputs are imported first and skipped.

In [ ]:
import experiments as G
print("imported runs:", G.import_runs([INP]))
complete = G.run_part(PART)

## Part C: cost of every model and the full analysis
The cross-validation analysis is written before the test predictions are read.

In [ ]:
if PART == "C" and complete:
    os.environ.setdefault("GR_PROPOSED", "h_effb0")
    import analysis as A
    G.measure_costs(f"{W}/output/costs.json")
    R = A.main(G.T.RUNS, f"{W}/output/results", f"{W}/output/costs.json")
    m = R["cross_validation"]["models"][R["cross_validation"]["proposed"]]["ensemble"]
    print("proposed model, cross-validation:", {k: round(v, 4) for k, v in m.items()})

## Output

In [ ]:
out = f"{W}/gingival_part{PART}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for base in ["runs", "results"]:
        for p in glob.glob(f"{W}/output/{base}/**/*", recursive=True):
            if os.path.isfile(p) and not p.endswith("model.pt"): z.write(p, os.path.relpath(p, f"{W}/output"))
    if os.path.exists(f"{W}/output/costs.json"): z.write(f"{W}/output/costs.json", "costs.json")
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")